<div style="background:linear-gradient(120deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            border-radius:14px;padding:38px 42px 30px 42px;color:#fff;font-family:Calibri,Segoe UI,sans-serif;">
  <div style="font-size:11px;letter-spacing:3px;font-weight:700;color:#F5C242;text-transform:uppercase;">
    Banque africaine de développement &nbsp;·&nbsp; UA STATAFRIC &nbsp;·&nbsp; STG17 &nbsp;·&nbsp; Jour 4
  </div>
  <div style="font-size:40px;font-weight:700;line-height:1.1;margin-top:14px;">
    L'Afrique après la tombée de la nuit
  </div>
  <div style="font-size:19px;font-style:italic;color:#E6F6EE;margin-top:6px;">
    Lire le développement dans les lumières nocturnes
  </div>
  <div style="height:5px;background:#F5C242;border-radius:3px;margin-top:22px;width:190px;"></div>
  <div style="font-size:13.5px;color:#E6F6EE;margin-top:18px;max-width:960px;line-height:1.55;">
    Une chaîne de traitement complète et reproductible : donnez-lui un code pays ISO3, elle acquiert les lumières
    nocturnes VIIRS, exécute sept scénarios d'analyse sur trois niveaux administratifs, produit un tableau de bord
    interactif et le publie comme site public GitHub Pages.
  </div>
</div>

## 0 · Ce que vous allez construire aujourd'hui

À la fin de ce notebook, vous disposez de quatre livrables, tous rattachés au Plan d'action STG17
(activités 2.1.1, 4.2.1, 4.3 et 3.1.1) :

| # | Livrable | Fichier | Alimente |
|---|---|---|---|
| 1 | Un notebook documenté et reproductible | `ce fichier` | 2.1.1 Lignes directrices méthodologiques |
| 2 | Une table d'indicateurs par niveau administratif | `outputs/data/*.csv` | 4.2.1 Manuel de référence |
| 3 | Un tableau de bord interactif | `outputs/site/index.html` | 3.1.1 Projets pilotés par les champions |
| 4 | Une **déclaration de limites** explicite | `outputs/LIMITATIONS.md` | 4.3 Gouvernance des données |

La déclaration de limites fait partie intégrante du livrable : ce n'est pas un avertissement de bas de page que
l'on ajoute à la fin, c'est la pièce qui autorise ou interdit la diffusion du reste.

---

### Les sept scénarios

| # | Scénario | Question posée | Niveaux |
|---|---|---|---|
| **A** | Activité économique | Où l'activité mesurable se concentre-t-elle ? | ADM0/1/2 |
| **B** | Électrification | Combien de personnes vivent dans des cellules habitées non éclairées ? | ADM1/2 |
| **C** | Urbanisation | Où l'empreinte éclairée s'est-elle étendue ? | ADM1/2 |
| **D** | Inégalités spatiales | La lumière se concentre-t-elle ou se diffuse-t-elle ? | ADM1 |
| **E** | Détection de changement | Qui a gagné, qui a perdu, et est-ce significatif ? | ADM2 |
| **F** | Suivi des chocs | Y a-t-il une anomalie mensuelle par rapport à sa propre référence ? | ADM1 |
| **G** | Validation | L'indicateur survit-il au contact des statistiques officielles ? | ADM1 |

---

> ⚠️ **À lire avant toute chose**
>
> Les lumières nocturnes sont un **indicateur indirect**. Elles mesurent une radiance quittant le sol vers
> 01h30 heure locale, rien d'autre. Chaque chiffre produit par ce notebook est une *mesure de lumière* dont vous
> *défendez* ensuite la pertinence pour l'électrification, l'activité économique ou l'urbanisation. Cet argument
> doit être formulé explicitement, testé au scénario G, et publié avec le résultat. Un tableau de bord qui masque
> cette étape est pire que pas de tableau de bord du tout.

## 1 · Comment exécuter ce notebook

Quatre modes d'acquisition. Vous en choisissez un dans la cellule de configuration ; tout ce qui suit est
rigoureusement identique. **Le mode par défaut est `gee`, c'est-à-dire des données VIIRS réelles.**

| Mode | `PROVIDER` | Prérequis | À utiliser quand |
|---|---|---|---|
| 🟣 **Earth Engine** *(défaut)* | `"gee"` | un compte Google Earth Engine + un projet Cloud | Le cas normal. Données VIIRS réelles, à n'importe quelle échelle, sans rien télécharger à la main. |
| 🟠 **Black Marble** | `"blackmarble"` | un jeton NASA Earthdata | Vous voulez directement les produits NASA VNP46A3/A4, corrigés BRDF. |
| 🔵 **Local** | `"local"` | vos propres GeoTIFF | Vous avez téléchargé vos rasters au jour 4 · partie 1. |
| ⚪ **Démo** | `"demo"` | aucun | **Repli uniquement** : pas de réseau, pas d'identifiants. Fabrique un pays **synthétique**. |

### Identifiants : le notebook vous les demande

Vous n'avez rien à coller dans le code. Au premier accès aux données, le notebook demande ce qui manque :
l'identifiant de votre projet Earth Engine puis, si nécessaire, il ouvre la page d'authentification Google ; ou
le jeton Earthdata en saisie masquée pour Black Marble. La bibliothèque correspondante (`earthengine-api`,
`h5py`) est installée automatiquement si elle est absente.

Pour obtenir un compte Earth Engine : `earthengine.google.com` → *Get started*, puis créez ou rattachez un
projet Google Cloud. L'usage non commercial et la recherche sont gratuits.

### Le mode démo, et pourquoi il existe

Le mode ⚪ **démo** fabrique un pays synthétique avec une physique NTL réaliste : villes suivant une loi de Zipf,
halo lumineux, torchères de gaz, corridors routiers, plancher de bruit, un district en déclin. Il fonctionne
**sans internet et sans identifiants**, et c'est le filet de sécurité de l'atelier si la salle perd sa
connexion ou si les comptes ne sont pas encore validés. Ses sorties sont filigranées `DONNÉES SIMULÉES` et ne
doivent **jamais** être publiées comme statistiques.

> 💡 **Astuce d'atelier** — Si vous découvrez le notebook, une première exécution en `demo` (≈ 2 minutes) vous
> montre le produit fini. Basculez ensuite sur `gee` pour votre vrai pays : vous déboguerez dix fois plus vite,
> parce que vous saurez déjà à quoi la sortie est censée ressembler.

### Où exécuter ce notebook

Le notebook est autonome : la cellule 1.1 détecte l'environnement et installe uniquement les paquets manquants.
Aucune adaptation de code n'est nécessaire de l'un à l'autre, et les chemins de sortie sont résolus
automatiquement (`/kaggle/working` sur Kaggle, le répertoire courant ailleurs).

| Environnement | Comment l'ouvrir | À savoir |
|---|---|---|
| **Google Colab** | `Fichier ▸ Importer un notebook` → déposer le `.ipynb`. Une fois votre dépôt publié (§9), il s'ouvre aussi par `Fichier ▸ Ouvrir ▸ GitHub`. | Seuls numpy/pandas/matplotlib sont préinstallés : la cellule 1.1 ajoute `geopandas`, `rasterio` et `plotly` (≈ 60 s). Aucun GPU nécessaire. |
| **Kaggle** | `Create ▸ Notebook ▸ File ▸ Import Notebook` → téléverser le `.ipynb`. | La pile géospatiale est déjà présente. Le mode `demo` tourne entièrement **hors ligne** ; les autres fournisseurs exigent `Settings ▸ Internet: On`. |
| **JupyterLab / VS Code (local)** | `pip install -r requirements.txt`, puis ouvrir le fichier. | Le plus rapide, et le seul endroit confortable pour `PROVIDER = "local"` avec vos propres GeoTIFF. |
| **Binder / JupyterHub institutionnel** | Ouvrir normalement. | La cellule 1.1 n'installe que ce qui manque. |

> ⚠️ **Colab** — si la cellule 1.2 (imports) échoue juste après une installation, faites
> `Exécution ▸ Redémarrer la session` puis reprenez à la cellule 1.1. C'est le comportement normal de Colab
> lorsqu'une bibliothèque compilée (`rasterio`) remplace une version déjà chargée en mémoire.
>
> ⚠️ **Kaggle** — sans `Internet: On`, seul le mode `demo` fonctionne, mais il fonctionne **entièrement** :
> les sept scénarios, le tableau de bord et la fiche pays sont produits sans aucun accès réseau.

In [ ]:
# --------------------------------------------------------------------------------------
# 1.1 · Amorçage de l'environnement
#
#   Détecte Colab / Kaggle / Binder / Jupyter local, puis installe UNIQUEMENT ce qui manque.
#
#   Idempotent : ré-exécutable sans risque.
# --------------------------------------------------------------------------------------
import sys, os, subprocess, importlib, importlib.util, platform
from pathlib import Path

REQUIREMENTS = '''numpy>=1.24
pandas>=2.0
geopandas>=0.14
rasterio>=1.3
shapely>=2.0
plotly>=5.18
matplotlib>=3.7
scipy>=1.10
requests>=2.31
# optionnel
# earthengine-api>=0.1.380   # PROVIDER = "gee"
# h5py>=3.9                  # PROVIDER = "blackmarble"
'''

# nom de module -> dépendance pip
_NEEDED = [
    ("numpy",      "numpy>=1.24"),
    ("pandas",     "pandas>=2.0"),
    ("scipy",      "scipy>=1.10"),
    ("matplotlib", "matplotlib>=3.7"),
    ("requests",   "requests>=2.31"),
    ("shapely",    "shapely>=2.0"),
    ("geopandas",  "geopandas>=0.14"),
    ("rasterio",   "rasterio>=1.3"),
    ("plotly",     "plotly>=5.18"),
]


def detect_env():
    '''Renvoie l'un de : colab | kaggle | binder | jupyter | script.'''
    if importlib.util.find_spec("google.colab") is not None:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if os.environ.get("BINDER_SERVICE_HOST") or os.environ.get("JUPYTERHUB_USER"):
        return "binder"
    try:
        shell = get_ipython().__class__.__name__          # noqa: F821
        return "jupyter" if shell in ("ZMQInteractiveShell", "TerminalInteractiveShell") else "script"
    except NameError:
        return "script"


def _pip_install(specs):
    '''Installation pip qui survit aussi aux environnements « externally managed » (PEP-668).'''
    if not specs:
        return True
    base = [sys.executable, "-m", "pip", "install", "-q"]
    for extra in ([], ["--break-system-packages"], ["--user"]):
        try:
            subprocess.check_call(base + extra + list(specs))
            return True
        except subprocess.CalledProcessError:
            continue
    return False


ENV = detect_env()

_missing = [spec for mod, spec in _NEEDED if importlib.util.find_spec(mod) is None]
if _missing:
    print(f"Installation de {len(_missing)} paquet(s) manquant(s) :")
    for _m in _missing:
        print("   ·", _m)
    _pip_install(_missing)
    importlib.invalidate_caches()
    _still = [spec for mod, spec in _NEEDED if importlib.util.find_spec(mod) is None]
    if _still:
        print("\n⚠️  Toujours absent :", ", ".join(_still))
        print("    Colab/Kaggle → Exécution ▸ Redémarrer la session, puis ré-exécuter cette cellule.")
    else:
        print("✅  Toutes les dépendances sont présentes.")
else:
    print("✅  Toutes les dépendances sont déjà présentes.")

# ---- rapport d'environnement -----------------------------------------------------------
_LABEL = {"colab": "Google Colab", "kaggle": "Kaggle", "binder": "Binder / JupyterHub",
          "jupyter": "Jupyter local", "script": "Python simple"}
print(f"\nEnvironnement : {_LABEL.get(ENV, ENV)}")
print(f"Python {platform.python_version()} · {platform.system()} {platform.machine()}")

if ENV == "kaggle":
    print("Kaggle → les sorties iront sous /kaggle/working (géré au §2.1).")
    print("Kaggle → sans « Settings ▸ Internet: On », seul PROVIDER='demo' fonctionnera.")
elif ENV == "colab":
    print("Colab → si la cellule suivante échoue à l'import : redémarrez la session, puis reprenez en 1.1.")

In [ ]:
# --------------------------------------------------------------------------------------
# 1.2 · Imports, contrôle d'environnement et charte graphique
# --------------------------------------------------------------------------------------
import os, sys, io, json, math, gzip, time, base64, hashlib, warnings, platform, textwrap, getpass
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

import rasterio
from rasterio.transform import from_origin, xy
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject, Resampling, calculate_default_transform

import geopandas as gpd
from shapely.geometry import box, Polygon, mapping, shape

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm

import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# ---- palette institutionnelle inspirée de la BAD (cf. guide de style) -----------------
PAL = {
    "green":      "#00A86A",   # primary
    "deep":       "#00704A",   # primary dark
    "forest":     "#00553A",   # darkest
    "gold":       "#F5C242",   # accent
    "ochre":      "#D49A00",   # accent on white
    "teal":       "#0E7C86",
    "terracotta": "#C4621D",
    "brick":      "#B83B2E",   # negative data only
    "ink":        "#231F20",
    "slate":      "#5E6964",
    "mist":       "#F4F7F5",
    "mintmist":   "#E8F5EF",
    "sage":       "#D5DED9",
    "grey":       "#A9B5B0",
}
RAMP = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]
SECTION_COLORS = [PAL["green"], PAL["deep"], PAL["teal"], PAL["ochre"],
                  PAL["terracotta"], PAL["brick"], PAL["forest"]]

# Palette appliquée à tous les rasters de lumières nocturnes du notebook
NTL_CMAP = LinearSegmentedColormap.from_list(
    "ntl", ["#050A08", "#0B2B22", "#00553A", "#00A86A", "#F5C242", "#FFF6DA"], N=512)

matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Calibri", "DejaVu Sans"],
    "axes.edgecolor": PAL["sage"], "axes.labelcolor": PAL["slate"],
    "xtick.color": PAL["slate"], "ytick.color": PAL["slate"],
    "axes.titlecolor": PAL["ink"], "axes.titleweight": "bold",
    "figure.facecolor": "white", "axes.grid": False, "figure.dpi": 110,
})

PLOTLY_LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Calibri, Segoe UI, sans-serif", size=13, color=PAL["ink"]),
    margin=dict(l=55, r=25, t=45, b=55),
    colorway=[PAL["green"], PAL["deep"], PAL["teal"], PAL["ochre"],
              PAL["terracotta"], PAL["brick"], PAL["grey"]],
    hoverlabel=dict(font_size=12, font_family="Calibri, sans-serif"),
    xaxis=dict(gridcolor="#E1E7E4", zeroline=False),
    yaxis=dict(gridcolor="#E1E7E4", zeroline=False),
    legend=dict(orientation="h", y=-0.18, x=0),
)

print(f"Python      {platform.python_version()}  ·  {platform.system()}")
for m in (np, pd, gpd, rasterio):
    print(f"{m.__name__:<12}{m.__version__}")
print("\n✅  Environnement prêt")

## 2 · Configuration — la seule cellule que vous modifiez normalement

Tout ce que fait le notebook est piloté par le dictionnaire ci-dessous. La règle de conception est délibérée :
*une seule* surface de configuration, aucune constante cachée plus bas dans le code. C'est ce qui rend
l'exécution reproductible, et c'est ce qui permet à un collègue de relancer votre analyse sur un autre pays en
changeant deux lignes.

### Les paramètres qui changent réellement vos résultats

| Paramètre | Ce qu'il fait | Pourquoi il n'est pas neutre |
|---|---|---|
| `NOISE_FLOOR` | radiance en dessous de ce seuil → traitée comme non éclairée | Trop bas : vous comptez du bruit de capteur comme de l'électrification. Trop haut : vous effacez les petits villages. **Documentez votre choix.** |
| `URBAN_CORE` | radiance au-dessus de ce seuil → « cœur urbain » | Définit ce que vous appellerez une ville. Ne comparez entre pays qu'à seuil identique. |
| `TOPCODE_PCT` | écrêtage de la radiance à ce percentile | Contrôle le poids qu'une seule torchère ou la capitale peut prendre dans le total national. |
| `YEARS` | année de référence et année finale | Deux années VIIRS sont comparables ; une année DMSP et une année VIIRS ne le sont **pas**. |

> 🎓 **Exercice 0** — Exécutez le notebook deux fois, avec `NOISE_FLOOR = 0.15` puis `NOISE_FLOOR = 0.50`.
> Notez de combien varie la « population non éclairée » nationale. Cet écart est la véritable marge
> d'incertitude de votre indicateur d'électrification.

In [ ]:
# ======================================================================================
# ⚙️  CONFIGURATION  —  c'est ici que vous éditez
# ======================================================================================
CONFIG = {
    # ---------- Pays --------------------------------------------------------------
    "ISO3":            "TUN",          # ISO 3166-1 alpha-3.  RWA, CIV, SEN, MOZ, CMR, SOM, KEN, GHA...
    "COUNTRY_NAME":    None,           # None -> resolved automatically / résolu automatiquement

    # ---------- Fournisseur de données --------------------------------------------
    # DONNÉES RÉELLES par défaut. "demo" ne fabrique que des données simulées et ne
    # doit servir que de repli (pas de réseau, pas d'identifiants).
    "PROVIDER":        "gee",          # "gee" | "blackmarble" | "local" | "demo"
    "GEE_PROJECT":     None,           # identifiant de projet Earth Engine ; None -> demandé
    "EARTHDATA_TOKEN": None,           # jeton NASA Earthdata (blackmarble) ; None -> demandé
    "LOCAL_RASTERS":   {},             # {"2015": "path/ntl_2015.tif", "2024": "path/ntl_2024.tif"}
    "LOCAL_POP":       None,           # GeoTIFF de population, ou {année: chemin}, ou None

    # ---------- Période -----------------------------------------------------------
    "YEARS":           [2015, 2024],   # [baseline, endline] — VIIRS era only (>= 2013)
    "MONTHLY_YEARS":   [2021, 2022, 2023, 2024],   # Scenario F — 4+ years recommended

    # ---------- Niveaux administratifs --------------------------------------------
    "ADMIN_LEVELS":    ["ADM0", "ADM1", "ADM2"],
    "BOUNDARY_SOURCE": "geoBoundaries",      # open, CC BY 4.0

    # ---------- Seuils analytiques ------------------------------------------------
    "NOISE_FLOOR":     0.25,   # nW·cm⁻²·sr⁻¹  — en dessous, un pixel est « non éclairé »
    "URBAN_CORE":      10.0,   # nW·cm⁻²·sr⁻¹  — au-dessus, « cœur urbain »
    "TOPCODE_PCT":     99.9,   # percentile d'écrêtage de la radiance
    "POP_LIT_MIN":     1.0,    # personnes/pixel pour qu'une cellule compte comme « habitée »

    # ---------- Robustesse (corrections méthodologiques) --------------------------
    "USE_TOPCODED":    True,   # scénarios A, D, E, G sur les valeurs écrêtées (recommandé)
    "MASK_FLARES":     True,   # neutraliser les torchères détectées avant tout calcul
    "FLARE_BUFFER_PX": 2,      # halo retiré autour de chaque torchère (pixels)
    "NOISE_SENSITIVITY": [0.15, 0.25, 0.50],  # planchers testés pour l'incertitude (B)
    "CHANGE_MIN_BASE_PCT": 10, # scénario E : sous ce percentile de SoL initiale, pas de signalement
    "MONTHLY_MIN_COVERAGE": 0.80,  # scénario F : part minimale de pixels sans nuage par mois
    "ALLOW_MIXED_PRODUCTS": False, # interdit de comparer deux produits NTL différents
    "TREND_ALL_YEARS": True,   # scénario E : tendance sur toutes les années (plus robuste)
    "SPATIAL_CHECK":   True,   # test d'autocorrélation spatiale (I de Moran) sur ADM2

    # ---------- Traitement du raster ----------------------------------------------
    "TARGET_SCALE_M":  500,    # VIIRS native ≈ 463 m at the equator
    "DEMO_MAX_PIX":    720,    # demo raster size cap (speed)

    # ---------- Validation (scénario G) -------------------------------------------
    # Statistiques officielles infranationales que vous apportez. CSV avec une colonne
    # de noms (ou de codes shapeID) au niveau OFFICIAL_STATS_LEVEL, plus des colonnes
    # numériques. None -> la validation ne peut pas dépasser « expérimental ».
    "OFFICIAL_STATS_CSV":   None,
    "OFFICIAL_JOIN_COL":    "adm1_name",
    "OFFICIAL_STATS_LEVEL": "ADM1",   # "ADM1" ou "ADM2" — niveau de la statistique fournie
    "VALIDATION_MIN_N":     20,       # nombre minimal d'unités pour « diffusable »
    # Nature de chaque colonne : "rate" (taux, part, %) ou "total" (effectif, montant).
    # Colonne absente -> déduite : valeurs toutes entre 0 et 100 => "rate".
    "OFFICIAL_STATS_TYPES": {},       # ex. {"taux_electrification": "rate", "pib_regional": "total"}
    # Pour un total, corrélation minimale une fois l'effet de taille (population) retiré
    "VALIDATION_MIN_PARTIAL": 0.30,

    # ---------- Langues du tableau de bord ----------------------------------------
    # ["fr", "en"] -> page bilingue avec bouton ; ["fr"] -> français seul, sans bouton.
    "DASHBOARD_LANGS": ["fr", "en"],

    # ---------- Sorties et publication --------------------------------------------
    "OUT_DIR":         "outputs",
    "REPO_NAME":       None,   # None -> ntl-<iso3>-stg17
    "REPO_DESCRIPTION": "Night-Time Lights subnational indicators — STG17 / AfDB / AU STATAFRIC",
    "SITE_TITLE":      None,   # None -> auto
    "AUTHOR":          "National Statistical Office",
    "AUTHOR_EMAIL":    "",
    "LICENSE_DATA":    "CC BY 4.0",
    "LICENSE_CODE":    "MIT",

    # ---------- Contrôle d'exécution ----------------------------------------------
    "RANDOM_SEED":     17,
    "VERBOSE":         True,
}

# ---- valeurs dérivées -----------------------------------------------------------------
CONFIG["ISO3"] = CONFIG["ISO3"].upper().strip()
CONFIG["REPO_NAME"] = CONFIG["REPO_NAME"] or f"ntl-{CONFIG['ISO3'].lower()}-stg17"
# Racine des sorties, résolue selon l'environnement
_ENV = globals().get("ENV", "script")
_BASE = Path("/kaggle/working") if (_ENV == "kaggle" and Path("/kaggle/working").is_dir()) else Path.cwd()
OUT = Path(CONFIG["OUT_DIR"])
if not OUT.is_absolute():
    OUT = _BASE / OUT
for sub in ("raster", "data", "site", "figures", "cache"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

LANGS = [l for l in CONFIG["DASHBOARD_LANGS"] if l in ("en", "fr")] or ["fr"]
LANG0 = 0 if LANGS[0] == "en" else 1          # index dans les couples (EN, FR)

np.random.seed(CONFIG["RANDOM_SEED"])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
IS_DEMO = CONFIG["PROVIDER"] == "demo"

def log(msg, level="info"):
    if not CONFIG["VERBOSE"]:
        return
    icon = {"info": "·", "ok": "✅", "warn": "⚠️ ", "step": "▶"}.get(level, "·")
    print(f"{icon} {msg}")

def ask(question, secret=False, default=""):
    '''Question posée à l'utilisateur. Renvoie `default` si le noyau n'a pas d'entrée
       interactive (exécution automatisée, nbconvert, CI…).'''
    try:
        import getpass as _gp
        return (_gp.getpass(question) if secret else input(question)).strip()
    except Exception:                                   # noqa: BLE001
        return default


def ask_yes_no(question, default=False):
    '''Question fermée. « o », « oui », « y », « yes » valent oui ; tout le reste non.'''
    r = ask(question, default="").lower()
    if not r:
        return default
    return r in ("o", "oui", "y", "yes")


log(f"Exécution {RUN_ID} — pays {CONFIG['ISO3']} — fournisseur {CONFIG['PROVIDER']}", "step")
if IS_DEMO:
    print("\n" + "═" * 78)
    print("  ⚠️   MODE DÉMO — DONNÉES SIMULÉES — NE PAS DIFFUSER")
    print("      Pour des données réelles : CONFIG['PROVIDER'] = 'gee'")
    print("      (ou 'blackmarble', ou 'local' si vous avez déjà vos GeoTIFF).")
    print("═" * 78)
else:
    print("\n" + "═" * 78)
    print(f"  🛰️   DONNÉES RÉELLES — fournisseur « {CONFIG['PROVIDER']} »")
    print("═" * 78)

## 3 · Dix minutes de physique incontournables

Deux ères de capteurs, et elles ne forment **pas** une série continue.

| Ère | Capteur | Période | Résolution | Radiométrie | Exploitable comment |
|---|---|---|---|---|---|
| 1 | **DMSP-OLS** | 1992–2013 | ~2,7 km (rééchantillonné à 1 km) | **nombres numériques 0–63** sur 6 bits, sans calibration embarquée | Historique long, mais sature sur tout centre-ville et exige une inter-calibration entre satellites |
| 2 | **VIIRS / DNB** (Suomi-NPP, NOAA-20/21) | 2012 → | ~463 m (~500 m) | **radiance calibrée**, nW·cm⁻²·sr⁻¹, ~7 ordres de grandeur | La seule ère adaptée à des statistiques infranationales quantitatives |

À l'intérieur de l'ère VIIRS, deux familles de produits comptent :

- **NASA Black Marble** — `VNP46A2` (journalier, corrigé BRDF), `VNP46A3` (mensuel), `VNP46A4` (annuel). Corrigés
  de l'atmosphère et de la BRDF lunaire : **c'est ce qu'il vous faut pour des séries temporelles**, puisque
  l'effet du clair de lune est retiré.
- **EOG / Colorado School of Mines VNL V2** — composites annuels, corrigés de la lumière parasite et des valeurs
  aberrantes. Très utilisés dans la littérature économique, et commodes puisqu'un fichier équivaut à une année.

**L'unité** — `nW·cm⁻²·sr⁻¹` : nanowatts par centimètre carré par stéradian. C'est une *radiance*, soit une
énergie par unité de surface et par unité d'angle solide. L'agrégat que tout le monde utilise, la **somme des
lumières (SoL)**, est la somme de cette radiance sur tous les pixels d'une zone. La SoL n'est donc *pas* une
énergie physique totale : c'est un indice. Traitez-la comme tel.

> 🚫 **À ne jamais faire** — Ne jamais coller une année DMSP à une année VIIRS pour fabriquer une « série
> 1992–2024 ». Ce sont deux grandeurs physiques différentes. Si vous avez besoin de l'historique long, vous devez
> conduire une inter-calibration et la publier comme une série distincte, clairement étiquetée comme telle.

In [ ]:
# --------------------------------------------------------------------------------------
# 3.1 · Frise des capteurs — une figure à réutiliser dans vos présentations
# --------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11.5, 3.5))
bars = [
    ("DMSP-OLS  ·  DN 0–63, ~2.7 km",              1992, 2013, PAL["grey"],   "#FFFFFF"),
    ("VIIRS/DNB  ·  radiance, ~463 m",             2012, 2026, PAL["green"],  "#FFFFFF"),
    ("Black Marble VNP46A2/A3/A4  ·  corrigé BRDF",  2012, 2026, PAL["deep"],   "#FFFFFF"),
    ("EOG VNL V2  ·  composites annuels",           2012, 2025, PAL["teal"],   "#FFFFFF"),
]
for i, (label, y0, y1, c, tc) in enumerate(bars):
    ax.barh(i, y1 - y0, left=y0, height=0.55, color=c, edgecolor="none", zorder=3)
    ax.text(y0 + 0.4, i, label, va="center", ha="left", fontsize=9.5, color=tc,
            fontweight="bold", zorder=4)

for yr in CONFIG["YEARS"]:
    ax.axvline(yr, color=PAL["ochre"], lw=1.8, ls="--", zorder=5)
    ax.text(yr, len(bars) - 0.25, f" {yr}", color=PAL["ochre"], fontsize=10,
            fontweight="bold", va="bottom")

ax.axvspan(1992, 2013, color=PAL["mist"], zorder=0)
ax.text(2002.5, -0.95, "radiométrie non comparable",
        ha="center", fontsize=8.5, color=PAL["slate"], style="italic")

ax.set_yticks([]); ax.set_xlim(1991, 2027); ax.set_ylim(-1.3, len(bars) - 0.1)
ax.set_xlabel("Année")
ax.set_title("Capteurs et produits de lumières nocturnes",
             fontsize=12.5, loc="left", pad=12)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / "figures" / "01_sensor_timeline.png", dpi=150,
                                bbox_inches="tight"); plt.show()

### 3.2 · Les six artefacts à traquer

Le laboratoire du jour 4 vous demande de retrouver ces artefacts dans *votre propre* pays. Voici ce qu'est chacun
d'eux, et comment il corrompt un indicateur naïf si vous ne le traitez pas.

| # | Artefact | Cause physique | Ce qu'il fait à votre chiffre | Traitement dans ce notebook |
|---|---|---|---|---|
| 1 | **Halo lumineux (blooming)** | Diffusion atmosphérique + fonction d'étalement du capteur | Gonfle la *surface éclairée* des villes d'un facteur 2 à 5 et fait déborder la lumière sur les districts voisins | Les surfaces éclairées sont toujours publiées *avec* le seuil utilisé ; les résultats ADM2 sont signalés quand la zone est petite devant le rayon de halo |
| 2 | **Saturation** | VIIRS sature rarement ; DMSP sature au-delà de DN 63 | Écrête le haut de la distribution → sous-estime les cœurs les plus denses | Nous détectons et publions la part de pixels proches du maximum du produit |
| 3 | **Torchères de gaz** | Le torchage pétrolier brûle intensément, en continu, sans population | Une seule torchère peut peser autant qu'une ville moyenne dans la SoL → une « croissance » qui n'est qu'un champ pétrolier | Détection de torchères candidates : très brillantes, spatialement isolées, population faible |
| 4 | **Saisonnalité** | Neige et albédo, végétation, clair de lune, fêtes (Ramadan, Noël, Diwali) | Des variations mensuelles de 10 à 30 % sans aucun contenu économique | La série mensuelle est toujours comparée à une **référence du même mois calendaire**, jamais au mois précédent |
| 5 | **Rupture de capteur** | Changement de satellite, changement de version d'algorithme | Une marche d'escalier dans la série, qui ressemble à un effet de politique publique | Fournisseur et version de produit consignés dans le manifeste d'exécution ; tout changement de version est signalé |
| 6 | **Bruit rural de faible intensité** | Plancher de bruit du détecteur, lumière parasite, luminescence atmosphérique | Fait apparaître le désert vide comme éclairé à 3 % | Seuil `NOISE_FLOOR`, et un test de sensibilité que vous êtes invité à exécuter |

> 🎓 **Exercice 1** — Une fois la section 5 exécutée, nommez lesquels de ces six artefacts sont présents dans
> **votre** pays, avec la sortie de cellule qui le prouve. Cette liste est une diapositive obligatoire de votre
> présentation du vendredi.

## 4 · Acquisition — limites, lumières, population

Trois ingrédients, chacun issu d'une source ouverte et citable.

1. **Limites administratives** — [geoBoundaries](https://www.geoboundaries.org) (`gbOpen`, CC BY 4.0). Ouvertes,
   versionnées, lisibles par machine, de l'ADM0 à l'ADM2 pour presque tous les pays africains. *Si votre institut
   national dispose d'un fichier officiel, utilisez-le à la place* — une seule fonction est à remplacer, tout le
   reste est inchangé.
2. **Lumières nocturnes** — l'un des quatre fournisseurs, derrière une interface unique.
3. **Population** — WorldPop (100 m ou 1 km, CC BY 4.0), qui sert à pondérer, à normaliser, et surtout à repérer
   les cellules *habitées mais non éclairées*.

> ⚖️ **Point licences** — geoBoundaries `gbOpen` est en CC BY 4.0 (attribution). WorldPop est en CC BY 4.0. Les
> produits VIIRS et Black Marble sont des œuvres du gouvernement américain, libres de restriction, avec
> attribution souhaitée. **Les trois sont publiables.** Comparez avec les données Ookla du jour 3
> (CC BY-NC-SA 4.0), dont les clauses *non commerciale* et *partage à l'identique* contraignent ce qu'un institut
> national peut diffuser. Connaissez la différence avant de publier, pas après.

In [ ]:
# --------------------------------------------------------------------------------------
# 4.0 · Utilitaires HTTP partagés, avec cache sur disque
# --------------------------------------------------------------------------------------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "STG17-NTL-Toolkit/1.0 (AfDB AU-STATAFRIC workshop)"})
CACHE = OUT / "cache"

def _cache_path(url, suffix=""):
    return CACHE / (hashlib.md5(url.encode()).hexdigest()[:16] + suffix)

def http_get(url, *, timeout=90, headers=None, stream=False, retries=3):
    '''GET avec temporisation exponentielle. Lève une exception en cas d'échec final.'''
    last = None
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=timeout, headers=headers, stream=stream)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}"
        except Exception as e:                      # noqa: BLE001
            last = f"{type(e).__name__}: {e}"
        time.sleep(1.6 ** attempt)
    raise RuntimeError(f"GET failed after {retries} attempts — {url}\\n  last error: {last}")

def download(url, dest, *, headers=None, force=False):
    '''Télécharge un fichier sur disque, avec cache. Renvoie un Path.'''
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0 and not force:
        log(f"en cache → {dest.name}")
        return dest
    log(f"téléchargement {url[:96]}…")
    r = http_get(url, headers=headers, stream=True, timeout=600)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with open(tmp, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    tmp.rename(dest)
    log(f"enregistré → {dest.name}  ({dest.stat().st_size/1e6:.1f} Mo)", "ok")
    return dest

log("Utilitaires HTTP prêts", "ok")

### 4.1 · Limites administratives

`geoBoundaries` expose une petite API REST. Nous demandons la géométrie *simplifiée* lorsqu'elle existe : un
tableau de bord n'a pas besoin de 2 Mo de détail côtier, et un GeoJSON allégé fait la différence entre un site qui
charge en 1 seconde et un site qui charge en 12 secondes sur une connexion 3G. Ce détail compte, puisque vos
utilisateurs se trouvent précisément dans le pays que vous cartographiez.

> 🔁 **Substituer vos limites officielles** — remplacez le corps de `fetch_admin()` par
> `gpd.read_file("chemin/vers/adm1_officiel.shp")` et renommez les colonnes en `shapeName` et `shapeID`.
> Rien d'autre ne change dans le notebook.

In [ ]:
# --------------------------------------------------------------------------------------
# 4.1 · Limites administratives
# --------------------------------------------------------------------------------------
GB_API  = "https://www.geoboundaries.org/api/current/gbOpen/{iso}/{lvl}/"
GB_RAW  = ("https://raw.githubusercontent.com/wmgeolab/geoBoundaries/main/releaseData/"
           "gbOpen/{iso}/{lvl}/geoBoundaries-{iso}-{lvl}_simplified.geojson")

# Emprises utilisées uniquement par le mode DÉMO (lon_min, lat_min, lon_max, lat_max)
DEMO_BBOX = {
    "TUN": (7.5, 30.2, 11.6, 37.5),  "RWA": (28.8, -2.9, 30.9, -1.0),
    "CIV": (-8.6, 4.3, -2.5, 10.7),  "SEN": (-17.6, 12.3, -11.3, 16.7),
    "MOZ": (30.2, -26.9, 40.9, -10.4), "CMR": (8.4, 1.6, 16.2, 13.1),
    "SOM": (40.9, -1.7, 51.5, 12.0), "KEN": (33.9, -4.7, 41.9, 5.5),
    "GHA": (-3.3, 4.7, 1.2, 11.2),   "MAR": (-13.2, 27.6, -1.0, 35.9),
    "DZA": (-8.7, 18.9, 12.0, 37.1), "EGY": (24.7, 22.0, 36.9, 31.7),
    "NGA": (2.7, 4.3, 14.7, 13.9),   "ETH": (33.0, 3.4, 48.0, 14.9),
    "ZAF": (16.5, -34.8, 32.9, -22.1), "UGA": (29.6, -1.5, 35.0, 4.2),
    "TZA": (29.3, -11.7, 40.4, -0.99), "BFA": (-5.5, 9.4, 2.4, 15.1),
    "MLI": (-12.2, 10.1, 4.3, 25.0), "NER": (0.2, 11.7, 16.0, 23.5),
    "TCD": (13.5, 7.4, 24.0, 23.4),  "COD": (12.2, -13.5, 31.3, 5.4),
    "GAB": (8.7, -3.98, 14.5, 2.3),  "ZMB": (21.9, -18.1, 33.7, -8.2),
    "MWI": (32.7, -17.1, 35.9, -9.4), "BEN": (0.77, 6.2, 3.85, 12.4),
    "TGO": (-0.15, 6.1, 1.8, 11.1),  "GIN": (-15.1, 7.2, -7.6, 12.7),
    "MDG": (43.2, -25.6, 50.5, -11.9), "BDI": (28.9, -4.5, 30.9, -2.3),
}

def _demo_admin(iso3, level, seed=17):
    '''Grille administrative synthétique : le notebook tourne sans aucun réseau.'''
    bbox = DEMO_BBOX.get(iso3, (9.0, 30.0, 15.0, 36.0))
    lon0, lat0, lon1, lat1 = bbox
    n = {"ADM0": 1, "ADM1": 4, "ADM2": 8}[level]          # n x n grid (ADM0 = whole box)
    rng = np.random.default_rng(seed + len(level))
    xs = np.linspace(lon0, lon1, n + 1)
    ys = np.linspace(lat0, lat1, n + 1)
    # on perturbe les lignes internes pour que les polygones ne soient pas trop rectangulaires
    if n > 1:
        xs[1:-1] += rng.normal(0, (lon1 - lon0) / (9 * n), n - 1)
        ys[1:-1] += rng.normal(0, (lat1 - lat0) / (9 * n), n - 1)
    rows = []
    for j in range(n):
        for i in range(n):
            rows.append({
                "shapeName": f"{level[-1]}-{chr(65+j)}{i+1}" if n > 1 else f"{iso3}",
                "shapeID":   f"{iso3}-{level}-{j}{i}",
                "geometry":  box(xs[i], ys[j], xs[i+1], ys[j+1]),
            })
    return gpd.GeoDataFrame(rows, crs="EPSG:4326")

def fetch_admin(iso3, level, demo=False):
    '''Renvoie un GeoDataFrame avec les colonnes shapeName, shapeID, geometry (EPSG:4326).'''
    if demo:
        gdf = _demo_admin(iso3, level)
        log(f"{level} : {len(gdf)} unités synthétiques", "warn")
    else:
        gdf = None
        try:                                   # 1) REST API → download URL
            meta = http_get(GB_API.format(iso=iso3, lvl=level), timeout=45).json()
            if isinstance(meta, list):
                meta = meta[0]
            url = meta.get("simplifiedGeometryGeoJSON") or meta.get("gjDownloadURL")
            gdf = gpd.read_file(url)
        except Exception as e:                 # noqa: BLE001
            log(f"API geoBoundaries en échec pour {level} ({e}) ; essai du miroir brut", "warn")
            try:                               # 2) raw GitHub mirror
                gdf = gpd.read_file(GB_RAW.format(iso=iso3, lvl=level))
            except Exception as e2:            # noqa: BLE001
                raise RuntimeError(
                    f"Could not obtain {level} boundaries for {iso3}. "
                    f"Set CONFIG['PROVIDER']='demo' or supply your official file. ({e2})")
        log(f"{level} : {len(gdf)} unités issues de geoBoundaries", "ok")

    keep = [c for c in ("shapeName", "shapeID", "shapeGroup", "geometry") if c in gdf.columns]
    gdf = gdf[keep].copy()
    if "shapeName" not in gdf:
        gdf["shapeName"] = [f"{level}-{i}" for i in range(len(gdf))]
    if "shapeID" not in gdf:
        gdf["shapeID"] = [f"{iso3}-{level}-{i}" for i in range(len(gdf))]
    gdf["shapeName"] = gdf["shapeName"].astype(str).str.strip()
    gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    gdf["geometry"] = gdf.geometry.buffer(0)                     # repair invalid rings
    return gdf.reset_index(drop=True)

ADMIN = {lvl: fetch_admin(CONFIG["ISO3"], lvl, demo=IS_DEMO) for lvl in CONFIG["ADMIN_LEVELS"]}
COUNTRY_BBOX = tuple(ADMIN["ADM0"].total_bounds) if "ADM0" in ADMIN else tuple(
    list(ADMIN.values())[0].total_bounds)
CONFIG["COUNTRY_NAME"] = CONFIG["COUNTRY_NAME"] or (
    str(ADMIN["ADM0"]["shapeName"].iloc[0]) if "ADM0" in ADMIN else CONFIG["ISO3"])

print(f"\nPays  : {CONFIG['COUNTRY_NAME']}  ({CONFIG['ISO3']})")
print(f"Emprise : {np.round(COUNTRY_BBOX, 3)}")
for lvl, g in ADMIN.items():
    print(f"  {lvl}: {len(g):>4} unités  ·  ex. {', '.join(g['shapeName'].head(3))}")

### 4.2 · Lumières nocturnes — une interface, quatre fournisseurs

L'acquisition est l'étape la plus susceptible de casser pendant un atelier : identifiants expirés, quotas
atteints, Wi-Fi coupé. Elle est donc isolée derrière une fonction unique, `fetch_ntl(year) → chemin GeoTIFF`,
dotée de quatre implémentations interchangeables. Si l'une échoue, vous changez une chaîne de caractères dans
`CONFIG` et vous continuez. Ce n'est pas de la programmation défensive gratuite : c'est ce qui rend une chaîne de
traitement **exploitable par un institut national de statistique** et pas seulement par son auteur.

#### Ce que le générateur de démo simule réellement

Ce n'est pas du bruit aléatoire. Il reproduit délibérément la physique que vous devez apprendre à reconnaître :

- une **hiérarchie urbaine rang-taille** (loi de Zipf : la deuxième ville vaut environ la moitié de la première),
- le **halo lumineux** — chaque cœur urbain est enveloppé d'une auréole large et faible,
- des **corridors de transport** entre les plus grandes villes,
- **deux torchères de gaz** — extrêmement brillantes, minuscules, sans population alentour,
- un **plancher de bruit log-normal** sur les terres vides,
- une **croissance différenciée** entre l'année de référence et l'année finale : les cœurs croissent lentement,
  les périphéries rapidement (étalement urbain),
- un **champ de population** corrélé à la lumière mais volontairement *non identique* — c'est exactement ce qui
  engendre les cellules habitées mais non éclairées que le scénario B est fait pour trouver.

In [ ]:
# --------------------------------------------------------------------------------------
# 4.2 · Fournisseurs NTL — démo / local / Earth Engine / Black Marble
# --------------------------------------------------------------------------------------
def _grid_for_bbox(bbox, scale_m, max_pix=None):
    '''Renvoie (hauteur, largeur, transform, res_deg) pour une emprise lon/lat à ~scale_m.'''
    lon0, lat0, lon1, lat1 = bbox
    lat_mid = (lat0 + lat1) / 2
    res_deg = scale_m / 111_320.0
    w = max(8, int(round((lon1 - lon0) / res_deg)))
    h = max(8, int(round((lat1 - lat0) / res_deg)))
    if max_pix and max(w, h) > max_pix:
        k = max(w, h) / max_pix
        w, h = int(w / k), int(h / k)
        res_deg *= k
    res_x = (lon1 - lon0) / w
    res_y = (lat1 - lat0) / h
    return h, w, from_origin(lon0, lat1, res_x, res_y), (res_x, res_y)

def _write_tif(path, arr, transform, nodata=np.nan, tags=None):
    arr = np.asarray(arr, dtype="float32")
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1],
                       count=1, dtype="float32", crs="EPSG:4326", transform=transform,
                       nodata=nodata, compress="deflate", predictor=2, tiled=True) as dst:
        dst.write(arr, 1)
        if tags:
            dst.update_tags(**{k: str(v) for k, v in tags.items()})
    return Path(path)

# ---------- DÉMO -----------------------------------------------------------------------
def _blob(yy, xx, cy, cx, sy, sx, amp):
    return amp * np.exp(-(((yy - cy) ** 2) / (2 * sy ** 2) + ((xx - cx) ** 2) / (2 * sx ** 2)))

def _demo_fields(bbox, year, cfg, seed=17):
    '''Renvoie (ntl, pop, transform). Pays synthétique à physique réaliste.'''
    h, w, tr, (rx, ry) = _grid_for_bbox(bbox, cfg["TARGET_SCALE_M"], cfg["DEMO_MAX_PIX"])
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:h, 0:w].astype("float32")

    n_cities = 16
    cy = rng.uniform(0.08, 0.92, n_cities) * h
    cx = rng.uniform(0.08, 0.92, n_cities) * w
    # rang-taille : primauté de la capitale, puis décroissance de Zipf
    rank = np.arange(1, n_cities + 1)
    amp  = 190.0 * rank ** -1.12 * rng.uniform(0.75, 1.3, n_cities)
    sig  = (0.0105 * h) * rank ** -0.33 * rng.uniform(0.8, 1.25, n_cities)

    ntl = np.zeros((h, w), dtype="float32")
    pop = np.zeros((h, w), dtype="float32")
    for k in range(n_cities):
        ntl += _blob(yy, xx, cy[k], cx[k], sig[k], sig[k], amp[k])            # core
        ntl += _blob(yy, xx, cy[k], cx[k], sig[k]*2.8, sig[k]*2.8, amp[k]*0.055)  # blooming
        pop += _blob(yy, xx, cy[k], cx[k], sig[k]*1.5, sig[k]*1.5, amp[k]*95)

    # corridors de transport entre les quatre plus grandes villes
    for a in range(4):
        for b in range(a + 1, 4):
            t = np.linspace(0, 1, 260)
            ly, lx = cy[a] + t*(cy[b]-cy[a]), cx[a] + t*(cx[b]-cx[a])
            for py, px in zip(ly[::7], lx[::7]):
                ntl += _blob(yy, xx, py, px, 0.005*h, 0.005*h, 1.5)

    # population rurale peu éclairée — le cœur du scénario B
    for _ in range(120):
        ry_, rx_ = rng.uniform(0, h), rng.uniform(0, w)
        pop += _blob(yy, xx, ry_, rx_, 0.012*h, 0.012*h, rng.uniform(30, 260))
        if rng.random() < 0.35:
            ntl += _blob(yy, xx, ry_, rx_, 0.006*h, 0.006*h, rng.uniform(0.3, 2.2))

    # deux torchères de gaz : très brillantes, minuscules, volontairement loin des villes
    for _ in range(2):
        fy, fx = 0.5*h, 0.5*w
        for _try in range(400):
            fy, fx = rng.uniform(0.12, 0.88)*h, rng.uniform(0.12, 0.88)*w
            if np.min(np.hypot(cy - fy, cx - fx)) > 0.20 * h:
                break
        ntl += _blob(yy, xx, fy, fx, 0.0028*h, 0.0028*h, 620.0)

    # croissance entre les deux dates, plus rapide en périphérie (étalement)
    y0 = min(cfg["YEARS"]); dt = max(0, year - y0)
    core = ntl > cfg["URBAN_CORE"]
    from scipy.ndimage import zoom as _zoom, gaussian_filter as _gauss
    coarse = rng.normal(0, 1, (7, 7))
    field = _zoom(coarse, (h / 7.0, w / 7.0), order=3)
    field = _gauss(field[:h, :w], sigma=0.03 * h)
    field = field / (field.std() + 1e-9)
    g = (np.where(core, 1.018, 1.048) * np.exp(0.014 * field)) ** dt
    ntl = ntl * g
    # un district en déclin (dépeuplement, coupure, conflit) pour que la perte de
    # lumière ne soit pas structurellement nulle. La rupture survient à mi-période,
    # comme un choc réel, et les tirages aléatoires sont faits chaque année pour que
    # toutes les années partagent le même bruit de fond.
    dy, dx = rng.uniform(0.20, 0.80) * h, rng.uniform(0.20, 0.80) * w
    y_break = y0 + max(1, (max(cfg["YEARS"]) - y0) // 2)
    if year >= y_break:
        sd = 0.065 * h
        ntl = ntl * (1.0 - 0.78 * np.exp(-(((yy - dy) ** 2) + ((xx - dx) ** 2)) / (2 * sd ** 2)))
    pop = pop * (1.024 ** dt)

    # plancher de bruit log-normal + tavelure du détecteur
    ntl += rng.lognormal(mean=-2.95, sigma=0.50, size=(h, w)).astype("float32")
    ntl = np.clip(ntl + rng.normal(0, 0.03, (h, w)), 0, None).astype("float32")
    pop = np.clip(pop + rng.normal(0, 0.6, (h, w)), 0, None).astype("float32")
    # normalisation sur un total national plausible, pour que les chiffres se lisent bien
    target_pop = 9.0e6 * (1.024 ** dt)
    pop = (pop * (target_pop / max(pop.sum(), 1.0))).astype("float32")
    return ntl, pop, tr

# ---------- EARTH ENGINE ---------------------------------------------------------------
_EE_OK = False


def _ee_ready(cfg):
    '''Installe earthengine-api si besoin, authentifie, initialise (une seule fois). Renvoie ee.'''
    global _EE_OK
    try:
        import ee
    except ImportError:
        log("earthengine-api absent — installation en cours…", "warn")
        _pip_install(["earthengine-api>=0.1.380"])
        import importlib as _il
        _il.invalidate_caches()
        import ee
    if _EE_OK:
        return ee
    if cfg.get("GEE_PROJECT") in (None, ""):
        cfg["GEE_PROJECT"] = ask(
            "Identifiant de votre projet Google Cloud / Earth Engine "
            "(ex. ee-monnom) : ", default="") or None
    try:
        ee.Initialize(project=cfg["GEE_PROJECT"])
    except Exception:                                   # noqa: BLE001
        log("Authentification Earth Engine requise — suivez le lien affiché.", "warn")
        ee.Authenticate()
        ee.Initialize(project=cfg["GEE_PROJECT"])
    log("Earth Engine initialisé", "ok")
    _EE_OK = True
    return ee


def _gee_ntl(year, bbox, cfg, dest):
    ee = _ee_ready(cfg)
    region = ee.Geometry.Rectangle(list(bbox))
    # Les composites annuels EOG « VNL V2 » sont publiés en deux collections Earth Engine :
    # ANNUAL_V21 pour 2012-2021 et ANNUAL_V22 pour 2022 et après. Même famille de produits,
    # même bande (average_masked) : on cherche l'année dans V2.2, puis dans V2.1.
    # En dernier recours seulement : médiane des mensuels corrigés (produit DIFFÉRENT).
    # Le produit exact et sa famille sont inscrits dans le GeoTIFF (voir §4.2, contrôle).
    VNL_FAMILY = "EOG VNL V2 annuel (V2.1 / V2.2) · average_masked"
    img = product = family = None
    for coll in ("NOAA/VIIRS/DNB/ANNUAL_V22", "NOAA/VIIRS/DNB/ANNUAL_V21"):
        try:
            cand = (ee.ImageCollection(coll).filterDate(f"{year}-01-01", f"{year}-12-31").first()
                    .select("average_masked").rename("radiance"))
            _ = cand.bandNames().getInfo()
            img, product, family = cand, f"{coll} · average_masked", VNL_FAMILY
            break
        except Exception:                               # noqa: BLE001
            continue
    if img is None:
        log(f"NTL {year} : aucun composite annuel VNL V2 — repli sur la médiane des mensuels "
            "VCMSLCFG (produit différent)", "warn")
        _col = (ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
                .filterDate(f"{year}-01-01", f"{year}-12-31"))
        img = _col.map(lambda i: i.select("avg_rad").max(0)
                       .updateMask(i.select("cf_cvg").gt(0))).median().rename("radiance")
        product = family = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG · médiane annuelle de avg_rad"
    else:
        log(f"NTL {year} : {product}", "ok")
    # échelle adaptative pour que getDownloadURL reste sous la limite de taille
    area_deg = (bbox[2]-bbox[0]) * (bbox[3]-bbox[1])
    scale = cfg["TARGET_SCALE_M"] * max(1, math.ceil((area_deg / 40) ** 0.5))
    if scale > cfg["TARGET_SCALE_M"]:
        log(f"NTL {year} : pays étendu — téléchargement à {scale} m au lieu de "
            f"{cfg['TARGET_SCALE_M']} m (limite de taille Earth Engine)", "warn")
    url = img.getDownloadURL({"region": region, "scale": scale,
                              "crs": "EPSG:4326", "format": "GEO_TIFF"})
    path = download(url, dest, force=False)
    with rasterio.open(path, "r+") as _s:
        _s.update_tags(product=product, product_family=family, year=str(year),
                       scale_m=str(scale), provider="gee")
    return path

# ---------- NASA BLACK MARBLE ----------------------------------------------------------
BM_ARCHIVE = "https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5000"

def _bm_tiles(bbox):
    '''Grille Black Marble 10°x10° : h = (lon+180)/10, v = (90-lat)/10.'''
    lon0, lat0, lon1, lat1 = bbox
    hs = range(int((lon0 + 180) // 10), int((lon1 + 180) // 10) + 1)
    vs = range(int((90 - lat1) // 10), int((90 - lat0) // 10) + 1)
    return [(hh, vv) for vv in vs for hh in hs]

def _bm_token(cfg):
    '''Jeton NASA Earthdata : depuis CONFIG, sinon demandé en saisie masquée.'''
    if not cfg.get("EARTHDATA_TOKEN"):
        cfg["EARTHDATA_TOKEN"] = ask(
            "Jeton NASA Earthdata (saisie masquée) : ", secret=True, default="") or None
    if not cfg.get("EARTHDATA_TOKEN"):
        raise RuntimeError("Aucun jeton Earthdata : impossible d'utiliser Black Marble.")
    return cfg["EARTHDATA_TOKEN"]


def _bm_ntl(year, bbox, cfg, dest):
    '''Mosaïque VNP46A4 (annuelle), découpée sur l'emprise. Exige un jeton Earthdata.'''
    try:
        import h5py
    except ImportError:
        log("h5py absent — installation en cours…", "warn")
        _pip_install(["h5py>=3.9"])
        import importlib as _il
        _il.invalidate_caches()
        import h5py
    token = _bm_token(cfg)
    hdr = {"Authorization": f"Bearer {token}"}
    listing = http_get(f"{BM_ARCHIVE}/VNP46A4/{year}/001.csv", headers=hdr).text
    names = [ln.split(",")[0].strip().strip('"') for ln in listing.splitlines()[1:]]
    wanted = {f"h{h:02d}v{v:02d}" for h, v in _bm_tiles(bbox)}
    files = [n for n in names if any(t in n for t in wanted)]
    if not files:
        raise RuntimeError(f"Aucune tuile VNP46A4 trouvée pour {wanted} en {year}")
    h_, w_, tr, (rx, ry) = _grid_for_bbox(bbox, cfg["TARGET_SCALE_M"])
    out = np.full((h_, w_), np.nan, dtype="float32")
    for fn in files:
        local = download(f"{BM_ARCHIVE}/VNP46A4/{year}/001/{fn}", CACHE / fn, headers=hdr)
        with h5py.File(local, "r") as f:
            grp = f["HDFEOS"]["GRIDS"]
            gname = list(grp.keys())[0]
            data = grp[gname]["Data Fields"]
            key = next(k for k in data.keys() if "NearNadir_Composite_Snow_Free" in k
                       and "Quality" not in k and "Std" not in k and "Num" not in k)
            arr = np.array(data[key]).astype("float32")
            arr[arr > 6e4] = np.nan
            arr *= 0.1                              # scale factor → nW·cm⁻²·sr⁻¹
        tag = next(t for t in wanted if t in fn)
        th, tv = int(tag[1:3]), int(tag[4:6])
        t_lon0, t_lat1 = -180 + 10*th, 90 - 10*tv
        t_tr = from_origin(t_lon0, t_lat1, 10/arr.shape[1], 10/arr.shape[0])
        reproject(arr, out, src_transform=t_tr, src_crs="EPSG:4326",
                  dst_transform=tr, dst_crs="EPSG:4326", resampling=Resampling.average)
    return _write_tif(dest, np.nan_to_num(out), tr,
                      tags={"product": "VNP46A4", "year": year})

# ---------- AIGUILLAGE -----------------------------------------------------------------
def fetch_ntl(year, cfg=CONFIG, bbox=None):
    '''Point d'entrée unique. Renvoie le chemin d'un GeoTIFF float32 en nW·cm⁻²·sr⁻¹.'''
    bbox = bbox or COUNTRY_BBOX
    # Le fournisseur fait partie du nom de fichier : un raster de démo en cache ne peut
    # jamais être relu par erreur comme une donnée réelle après changement de PROVIDER.
    _tag = "gee2" if cfg["PROVIDER"] == "gee" else cfg["PROVIDER"]    # v2 : V2.1 + V2.2
    dest = OUT / "raster" / f"ntl_{cfg['ISO3']}_{year}_{_tag}.tif"
    if dest.exists():
        log(f"NTL {year} : en cache ({cfg['PROVIDER']})", "ok")
        return dest
    p = cfg["PROVIDER"]
    log(f"NTL {year} : fournisseur = {p}", "step")
    if p == "demo":
        ntl, pop, tr = _demo_fields(bbox, year, cfg, seed=cfg["RANDOM_SEED"])
        _write_tif(OUT / "raster" / f"pop_{cfg['ISO3']}_{year}.tif", pop, tr,
                   tags={"product": "SYNTHETIC-POP", "year": year})
        return _write_tif(dest, ntl, tr, tags={"product": "SYNTHETIC-NTL", "year": year,
                                               "warning": "SIMULATED - NOT FOR DISSEMINATION"})
    if p == "local":
        src = cfg["LOCAL_RASTERS"].get(str(year)) or cfg["LOCAL_RASTERS"].get(year)
        if not src:
            raise RuntimeError(f"CONFIG['LOCAL_RASTERS'] has no entry for {year}")
        return Path(src)
    if p == "gee":
        return _gee_ntl(year, bbox, cfg, dest)
    if p == "blackmarble":
        return _bm_ntl(year, bbox, cfg, dest)
    raise ValueError(f"Unknown PROVIDER {p!r}")

NTL_PATHS = {y: fetch_ntl(y) for y in CONFIG["YEARS"]}


def ntl_product(path, family=False):
    '''Produit NTL inscrit dans le GeoTIFF : exact, ou sa famille (pour la comparabilité).'''
    with rasterio.open(path) as _s:
        t = _s.tags()
    exact = t.get("product") or f"fichier local : {Path(path).name}"
    return (t.get("product_family") or exact) if family else exact


NTL_PRODUCT = {y: ntl_product(p) for y, p in NTL_PATHS.items()}
NTL_FAMILY = {y: ntl_product(p, family=True) for y, p in NTL_PATHS.items()}
print("Produit utilisé pour chaque année :")
for y, pr in NTL_PRODUCT.items():
    print(f"  {y} : {pr}")
if len(set(NTL_FAMILY.values())) == 1 and len(set(NTL_PRODUCT.values())) > 1:
    print(f"  → versions différentes d'une même famille : {next(iter(NTL_FAMILY.values()))}")
if len(set(NTL_FAMILY.values())) > 1:
    msg = ("Les années ne proviennent pas du même produit NTL : toute comparaison entre elles "
           "mélangerait une rupture de produit et un changement réel (artefact n° 5, §3.2).")
    if not CONFIG["ALLOW_MIXED_PRODUCTS"]:
        raise RuntimeError(msg + " Choisissez des années couvertes par un même produit, ou "
                           "mettez CONFIG['ALLOW_MIXED_PRODUCTS'] = True en connaissance de cause.")
    log(msg + " Poursuite autorisée par ALLOW_MIXED_PRODUCTS.", "warn")
for y, p in NTL_PATHS.items():
    with rasterio.open(p) as s:
        print(f"  NTL {y}: {s.width}×{s.height} px  ·  res {s.res[0]*111.32*1000:.0f} m  ·  {p.name}")

### 4.3 · Population — WorldPop, et pourquoi elle n'est pas facultative

Sans population, vous pouvez dire « la région A est plus lumineuse que la région B ». Avec la population, vous
pouvez dire « 3,1 millions de personnes de la région B vivent dans des cellules habitées qui n'émettent aucune
lumière détectable » — c'est-à-dire une *phrase sur laquelle un ministre peut agir*. La population permet aussi de
normaliser : une région peut être lumineuse simplement parce que beaucoup de gens y vivent, et la lumière par
habitant est un indicateur très différent de la lumière totale.

Cette étape réutilise les rasters **WorldPop** introduits au jour 3 : les deux laboratoires partagent donc la même
couche de population.

**Une population par année.** La lumière par habitant de 2015 se calcule avec la population de 2015. Le notebook
télécharge donc un raster par année étudiée — l'année WorldPop disponible la plus proche — et signale tout écart
de plus d'un an entre l'année demandée et l'année fournie.

In [ ]:
# --------------------------------------------------------------------------------------
# 4.3 · Raster de population, aligné pixel à pixel sur la grille NTL
# --------------------------------------------------------------------------------------
WP_REST = "https://www.worldpop.org/rest/data/pop/wpgp?iso3={iso}"

POP_META = {}   # année demandée -> année de population réellement utilisée


def fetch_population(cfg=CONFIG, year=None):
    '''Raster de population pour `year` : l'année WorldPop disponible la plus proche.'''
    year = year or max(cfg["YEARS"])
    if cfg["PROVIDER"] == "demo":
        POP_META[year] = year
        p = OUT / "raster" / f"pop_{cfg['ISO3']}_{year}.tif"
        if p.exists():
            return p
        return OUT / "raster" / f"pop_{cfg['ISO3']}_{min(cfg['YEARS'])}.tif"
    if cfg["LOCAL_POP"]:
        # LOCAL_POP : un chemin unique, ou un dictionnaire {année: chemin}
        lp = cfg["LOCAL_POP"]
        src = lp.get(str(year)) or lp.get(year) if isinstance(lp, dict) else lp
        POP_META[year] = f"fichier local ({Path(src).name})"
        return Path(src)
    dest = OUT / "raster" / f"worldpop_{cfg['ISO3']}_{year}.tif"
    if dest.exists():
        with rasterio.open(dest) as _s:
            POP_META[year] = int(_s.tags().get("popyear", year))
        return dest
    meta = http_get(WP_REST.format(iso=cfg["ISO3"])).json()["data"]
    rec = min(meta, key=lambda d: abs(int(d.get("popyear", 0) or 0) - year))
    url = rec["files"][0] if isinstance(rec.get("files"), list) else rec["files"]
    popyear = int(rec.get("popyear") or year)
    log(f"WorldPop {popyear} (demandé : {year}) → {url.split('/')[-1]}")
    path = download(url, dest)
    with rasterio.open(path, "r+") as _s:
        _s.update_tags(popyear=str(popyear))
    POP_META[year] = popyear
    return path

def align_to(src_path, ref_path, dest, resampling=Resampling.sum):
    '''Reprojette src sur la grille exacte de ref. `sum` préserve les totaux de population.'''
    dest = Path(dest)
    with rasterio.open(ref_path) as ref:
        prof = ref.profile.copy()
        prof.update(dtype="float32", count=1, nodata=np.nan,
                    compress="deflate", predictor=2, tiled=True)
        out = np.zeros((ref.height, ref.width), dtype="float32")
        with rasterio.open(src_path) as src:
            band = src.read(1, masked=True).filled(0).astype("float32")
            # `sum` exige une source plus fine que la cible ; repli sinon
            rs = resampling
            if abs(src.res[0]) > abs(ref.res[0]):
                rs = Resampling.bilinear
            reproject(band, out,
                      src_transform=src.transform, src_crs=src.crs or "EPSG:4326",
                      dst_transform=ref.transform, dst_crs=ref.crs,
                      resampling=rs)
    with rasterio.open(dest, "w", **prof) as dst:
        dst.write(out, 1)
    return dest

# Une population PAR ANNÉE : la lumière par habitant de 2015 doit se calculer avec la
# population de 2015, pas avec celle de la fin de période.
REF_YEAR = max(CONFIG["YEARS"])
POP_PATHS = {}
for y in CONFIG["YEARS"]:
    _raw = fetch_population(year=y)
    POP_PATHS[y] = align_to(_raw, NTL_PATHS[y],
                            OUT / "raster" / f"pop_aligned_{CONFIG['ISO3']}_{y}.tif")
POP_PATH = POP_PATHS[REF_YEAR]

print("Rasters de population alignés :")
for y, pth in POP_PATHS.items():
    with rasterio.open(pth) as s:
        _pop = s.read(1)
    src_y = POP_META.get(y, "?")
    print(f"  {y} : population {src_y} · total {np.nansum(_pop):,.0f} · "
          f"{(np.nan_to_num(_pop) > CONFIG['POP_LIT_MIN']).sum():,} pixels peuplés")
    if isinstance(src_y, int) and abs(src_y - y) > 1:
        log(f"population {y} approchée par l'année {src_y} (écart de {abs(src_y - y)} ans)", "warn")
_src = [POP_META.get(y) for y in CONFIG["YEARS"]]
if len(set(map(str, _src))) == 1 and len(_src) > 1:
    log("Même raster de population pour toutes les années : les indicateurs par habitant ne "
        "reflètent alors que l'évolution de la lumière.", "warn")

## 5 · Regarder le raster avant de calculer dessus

C'est la discipline sur laquelle insiste l'agenda du jour 4, et c'est celle que l'on saute le plus souvent.
Calculer des statistiques zonales sur un raster que l'on n'a jamais affiché, c'est ainsi qu'une torchère de gaz
devient un « pôle industriel émergent » dans un bulletin publié. Trois contrôles, dans cet ordre :

1. **La distribution.** La radiance est log-normale sur cinq ordres de grandeur. Tout histogramme en axe linéaire
   est inutile ; toute palette en échelle linéaire vous montre un pays noir avec quatre points blancs.
2. **La carte.** Regardez-la. Votre géographie nationale est dans votre tête : un barrage hydroélectrique, un
   champ pétrolier, un camp de réfugiés, un nouvel aéroport. L'œil attrape en deux secondes ce qu'une statistique
   dissimule.
3. **Les artefacts.** Quantifiés, et non estimés à vue : part saturée, part sous le plancher de bruit, torchères
   candidates.

In [ ]:
# --------------------------------------------------------------------------------------
# 5.1 · Distribution et carte — la figure de contrôle en deux volets
# --------------------------------------------------------------------------------------
def read_ntl(path):
    with rasterio.open(path) as s:
        a = s.read(1).astype("float32")
        a[~np.isfinite(a)] = 0.0
        return a, s.transform, s.bounds

def qa_panel(year, path, cfg=CONFIG):
    arr, tr, bnds = read_ntl(path)
    pos = arr[arr > 1e-4]
    fig = plt.figure(figsize=(13.2, 5.0))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.25], wspace=0.22)

    # ---- histogramme, x logarithmique ----
    ax = fig.add_subplot(gs[0, 0])
    bins = np.logspace(np.log10(max(pos.min(), 1e-3)), np.log10(pos.max()), 70)
    ax.hist(pos, bins=bins, color=PAL["green"], alpha=.85, edgecolor="none")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.axvline(cfg["NOISE_FLOOR"], color=PAL["ochre"], ls="--", lw=1.8)
    ax.text(cfg["NOISE_FLOOR"], ax.get_ylim()[1]*.45, " plancher\n de bruit",
            color=PAL["ochre"], fontsize=8.5, fontweight="bold")
    ax.axvline(cfg["URBAN_CORE"], color=PAL["terracotta"], ls="--", lw=1.8)
    ax.text(cfg["URBAN_CORE"], ax.get_ylim()[1]*.06, " cœur\n urbain",
            color=PAL["terracotta"], fontsize=8.5, fontweight="bold")
    ax.set_xlabel("Radiance  nW·cm⁻²·sr⁻¹  (log)"); ax.set_ylabel("Pixels (log)")
    ax.set_title(f"Distribution · {year}", loc="left", fontsize=11.5)

    # ---- carte, couleur logarithmique ----
    ax2 = fig.add_subplot(gs[0, 1])
    show = np.where(arr <= 0, np.nan, arr)
    im = ax2.imshow(show, cmap=NTL_CMAP, norm=LogNorm(vmin=max(cfg["NOISE_FLOOR"]*.4, 1e-2),
                                                      vmax=np.nanpercentile(show, 99.97)),
                    extent=[bnds.left, bnds.right, bnds.bottom, bnds.top], interpolation="nearest")
    ADMIN.get("ADM1", ADMIN["ADM0"]).boundary.plot(ax=ax2, color="#FFFFFF", lw=.55, alpha=.45)
    ax2.set_title(f"{cfg['COUNTRY_NAME']} de nuit · {year}", loc="left", fontsize=11.5)
    ax2.set_xlabel("lon"); ax2.set_ylabel("lat")
    cb = plt.colorbar(im, ax=ax2, fraction=.036, pad=.02)
    cb.set_label("nW·cm⁻²·sr⁻¹", fontsize=9, color=PAL["slate"])
    if cfg["PROVIDER"] == "demo":
        ax2.text(.5, .5, "SIMULATED", transform=ax2.transAxes, ha="center", va="center",
                 fontsize=34, color="white", alpha=.13, rotation=28, fontweight="bold")
    plt.savefig(OUT / "figures" / f"02_qa_{year}.png", dpi=145, bbox_inches="tight")
    plt.show()
    return arr

ARR = {y: qa_panel(y, p) for y, p in NTL_PATHS.items()}

In [ ]:
# --------------------------------------------------------------------------------------
# 5.2 · Diagnostic des artefacts — quantifié, pas estimé à vue
# --------------------------------------------------------------------------------------
from scipy import ndimage

def artefact_report(arr, pop, cfg=CONFIG):
    rep = {}
    finite = arr[np.isfinite(arr)]
    n = finite.size
    rep["pixels"] = n
    rep["share_below_noise_floor"] = float((finite < cfg["NOISE_FLOOR"]).mean())
    rep["share_lit"] = float((finite >= cfg["NOISE_FLOOR"]).mean())
    rep["share_urban_core"] = float((finite >= cfg["URBAN_CORE"]).mean())
    top = np.nanpercentile(finite, 99.99)
    rep["p50"], rep["p90"], rep["p99"], rep["p9999"], rep["max"] = [
        float(np.nanpercentile(finite, q)) for q in (50, 90, 99, 99.99)] + [float(np.nanmax(finite))]
    # concentration : part de la SoL nationale détenue par le 1 % de pixels les plus brillants
    s = np.sort(finite)[::-1]
    k = max(1, int(.01 * n))
    rep["sol_share_top1pct_pixels"] = float(s[:k].sum() / max(s.sum(), 1e-9))
    # indicateur de saturation : pixels à moins de 1 % du maximum observé
    rep["share_near_max"] = float((finite >= .99 * rep["max"]).mean())

    # --- torchères candidates : très brillantes, isolées, sans population --------------
    # Une torchère est « aussi brillante qu'un cœur urbain, sans la population d'une ville ».
    core_mask = arr >= cfg["URBAN_CORE"]
    ref_density = float(np.median(pop[core_mask])) if core_mask.any() else 0.0
    rep["core_pop_per_pixel"] = ref_density
    hot = arr >= max(cfg["URBAN_CORE"] * 6, np.nanpercentile(finite, 99.9))
    lab, nlab = ndimage.label(hot)
    flares, flare_ids = [], []
    if nlab:
        sizes = ndimage.sum(hot, lab, range(1, nlab + 1))
        peaks = ndimage.maximum(arr, lab, range(1, nlab + 1))
        popsum = ndimage.sum(np.nan_to_num(pop), lab, range(1, nlab + 1))
        cents = ndimage.center_of_mass(hot, lab, range(1, nlab + 1))
        for i in range(nlab):
            dens = popsum[i] / max(sizes[i], 1)
            if (sizes[i] <= 200 and peaks[i] > cfg["URBAN_CORE"] * 8
                    and dens < 0.30 * max(ref_density, 1.0)):
                flare_ids.append(i + 1)
                flares.append({"px": int(sizes[i]), "peak": round(float(peaks[i]), 1),
                               "pop": round(float(popsum[i])),
                               "pop_per_px": round(float(dens), 1),
                               "row": int(cents[i][0]), "col": int(cents[i][1])})
    rep["flare_candidates"] = sorted(flares, key=lambda d: -d["peak"])[:10]
    rep["_mask"] = np.isin(lab, flare_ids) if flare_ids else np.zeros(arr.shape, dtype=bool)
    return rep

POPA = {}
for y in CONFIG["YEARS"]:
    with rasterio.open(POP_PATHS[y]) as s:
        POPA[y] = np.nan_to_num(s.read(1).astype("float32"))
POP = POPA[REF_YEAR]

AQ = {y: artefact_report(ARR[y], POPA[y]) for y in CONFIG["YEARS"]}

print("DIAGNOSTIC DES ARTEFACTS")
print("=" * 78)
for y, r in AQ.items():
    print(f"\n▶ {y}")
    print(f"  sous le plancher de bruit   : {r['share_below_noise_floor']:6.1%}"
          f"   → candidate 'rural low-light noise'")
    print(f"  pixels éclairés             : {r['share_lit']:6.1%}")
    print(f"  cœur urbain                 : {r['share_urban_core']:6.1%}")
    print(f"  SoL du 1 % de pixels le + vif: {r['sol_share_top1pct_pixels']:6.1%}"
          f"   → concentration / risque de dominance")
    print(f"  pixels à 1 % du maximum     : {r['share_near_max']:6.3%}"
          f"   → saturation check")
    print(f"  p50 {r['p50']:.3f}  p90 {r['p90']:.2f}  p99 {r['p99']:.1f}  max {r['max']:.0f}")
    if r["flare_candidates"]:
        print(f"  ⚠️  {len(r['flare_candidates'])} torchère(s) suspectée(s) :")
        for f in r["flare_candidates"][:4]:
            print(f"        peak {f['peak']:>7.1f}  ·  {f['px']:>3} px  ·  {f['pop_per_px']:>6.1f} pers/px"
                  f"  (urban cores: {r['core_pop_per_pixel']:.1f})  ·  (r{f['row']}, c{f['col']})")
        print("      → vérifier sur une carte pétrolière ou gazière avant d'interpréter\n"
              "        une croissance dans cette zone.")
    else:
        print("  ✅ aucune torchère suspectée")

### 5.3 · Neutraliser les torchères avant tout calcul

Détecter une torchère ne suffit pas : si elle reste dans le raster, elle continue de peser dans la somme des lumières
de son district, dans le taux de croissance national et dans le Gini. Le notebook **retire donc les torchères
détectées** — le noyau brillant et un halo de `FLARE_BUFFER_PX` pixels — de toutes les années, avant les statistiques
zonales. Le masque est l'**union** des détections de toutes les années : une torchère repérée en 2024 est aussi
retirée en 2015, sinon son apparition passerait pour une croissance réelle.

Les pixels retirés sont mis à zéro et comptés dans le manifeste et dans la déclaration de limites. Pour désactiver
cette correction — par exemple pour étudier précisément le torchage — mettez `CONFIG["MASK_FLARES"] = False`.

In [ ]:
# --------------------------------------------------------------------------------------
# 5.3 · Masquage des torchères (union des années, avec halo)
# --------------------------------------------------------------------------------------
_shapes = {ARR[y].shape for y in CONFIG["YEARS"]}
_buf = int(CONFIG["FLARE_BUFFER_PX"])
def _grow(m):
    return ndimage.binary_dilation(m, iterations=_buf) if (_buf > 0 and m.any()) else m

if len(_shapes) == 1:
    # même grille : union des détections de toutes les années
    _u = np.zeros(next(iter(_shapes)), dtype=bool)
    for y in CONFIG["YEARS"]:
        _u |= AQ[y]["_mask"]
    MASKS = {y: _grow(_u) for y in CONFIG["YEARS"]}
else:
    # grilles différentes : masque propre à chaque année (union impossible pixel à pixel)
    log("Grilles différentes entre années : masque de torchères calculé année par année.", "warn")
    MASKS = {y: _grow(AQ[y]["_mask"]) for y in CONFIG["YEARS"]}
FLARE_MASK = MASKS[CONFIG["YEARS"][-1]]

FLARE_STATS = {"masked_pixels": int(FLARE_MASK.sum()), "applied": False, "sol_removed": {}}

if CONFIG["MASK_FLARES"] and FLARE_MASK.any():
    for y in CONFIG["YEARS"]:
        a = ARR[y]
        removed = float(a[MASKS[y]].sum())
        FLARE_STATS["sol_removed"][str(y)] = {
            "sol": removed, "share_of_national": removed / max(float(a.sum()), 1e-9)}
        a = a.copy(); a[MASKS[y]] = 0.0
        src = NTL_PATHS[y]
        dst = Path(src).with_name(Path(src).stem + "_noflare.tif")
        with rasterio.open(src) as s:
            prof = s.profile.copy(); tags = s.tags()
        prof.update(dtype="float32", count=1, nodata=None)
        with rasterio.open(dst, "w", **prof) as d:
            d.write(a.astype("float32"), 1)
            d.update_tags(**tags, flares_masked=str(int(MASKS[y].sum())))
        NTL_PATHS[y] = dst
        ARR[y] = a
    FLARE_STATS["applied"] = True
    print(f"Torchères neutralisées : {FLARE_MASK.sum():,} pixels (halo compris), sur toutes les années.")
    for y, r in FLARE_STATS["sol_removed"].items():
        print(f"  {y} : {r['sol']:,.0f} de SoL retirés, soit {r['share_of_national']:.1%} du total national")
elif FLARE_MASK.any():
    log("Torchères détectées mais NON masquées (MASK_FLARES = False) : elles pèsent dans les agrégats.",
        "warn")
else:
    print("Aucune torchère à neutraliser.")

> 🎓 **Exercice 2 — le test de sensibilité qui a sa place dans votre note méthodologique**
>
> Le scénario B calcule déjà la sensibilité de la population non éclairée. Étendez-la à la part éclairée :
> relancez le §5.2 avec `NOISE_FLOOR` à 0,15 puis 0,25 puis 0,50, et tabulez `share_lit` pour chaque valeur. Si la
> part éclairée nationale passe de 12 % à 31 % sur cette plage, votre indicateur de surface éclairée porte une
> incertitude de ±19 points et **ne doit pas** être publié à la décimale près. Publiez l'intervalle, ou publiez un
> rang plutôt qu'un niveau.

## 6 · Prétraitement et moteur zonal

Deux idées portent ici tout le reste.

**(1) L'écrêtage (*top-coding*).** Une seule torchère de gaz, ou une capitale macrocéphale, peut concentrer 40 %
de la SoL nationale. Si vous publiez alors un taux de croissance national, vous publiez en réalité la croissance
d'un seul objet. L'écrêtage plafonne la radiance au percentile `TOPCODE_PCT` : il ne supprime pas les lieux
brillants, il les empêche de *piloter seuls l'agrégat*. Chaque indicateur est calculé **des deux façons** et
l'écart est exposé, parce que cet écart est lui-même un résultat. Les scénarios utilisent ensuite la version
choisie par `USE_TOPCODED` (écrêtée par défaut) ; les deux versions sont exportées, la brute avec le suffixe
`_brut`.

**(2) La surface réelle du pixel, corrigée de la latitude.** En EPSG:4326, un pixel occupe un nombre constant de
*degrés*, pas un nombre constant de *kilomètres carrés* : à 35°N il est environ 18 % plus étroit qu'à l'équateur.
Une « surface éclairée en km² » obtenue en comptant des pixels est fausse d'autant, et l'erreur est systématique —
elle biaise les régions du nord contre celles du sud à l'intérieur d'un même pays. Nous calculons donc la surface
véritable de chaque ligne de pixels.

In [ ]:
# --------------------------------------------------------------------------------------
# 6.1 · Mesures d'inégalité et de concentration
# --------------------------------------------------------------------------------------
def gini(x, w=None):
    '''Coefficient de Gini pondéré (formule de Brown). x >= 0.'''
    x = np.asarray(x, dtype="float64").ravel()
    w = np.ones_like(x) if w is None else np.asarray(w, dtype="float64").ravel()
    m = np.isfinite(x) & np.isfinite(w) & (w > 0) & (x >= 0)
    x, w = x[m], w[m]
    if x.size < 2 or x.sum() <= 0:
        return np.nan
    o = np.argsort(x); x, w = x[o], w[o]
    w = w / w.sum()
    s = np.cumsum(w * x); s = s / s[-1]
    s_prev = np.concatenate(([0.0], s[:-1]))
    return float(1.0 - np.sum(w * (s_prev + s)))

def lorenz(x, w=None, n=101):
    '''Renvoie (part cumulée des poids, part cumulée des valeurs) pour une courbe de Lorenz.'''
    x = np.asarray(x, dtype="float64").ravel()
    w = np.ones_like(x) if w is None else np.asarray(w, dtype="float64").ravel()
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[m], w[m]
    if x.size == 0:
        return np.array([0, 1]), np.array([0, 1])
    o = np.argsort(x); x, w = x[o], w[o]
    cw = np.cumsum(w) / w.sum()
    cv = np.cumsum(x * w); cv = cv / cv[-1]
    q = np.linspace(0, 1, n)
    return q, np.interp(q, np.concatenate(([0], cw)), np.concatenate(([0], cv)))

def theil_T(x, w=None):
    '''Indice de Theil T de x pondéré par w (population). 0 = égalité parfaite.'''
    x = np.asarray(x, dtype="float64").ravel()
    w = np.ones_like(x) if w is None else np.asarray(w, dtype="float64").ravel()
    m = np.isfinite(x) & np.isfinite(w) & (w > 0) & (x > 0)
    x, w = x[m], w[m]
    if x.size < 2:
        return np.nan
    s = (x * w) / np.sum(x * w)        # value share
    p = w / w.sum()                    # weight share
    return float(np.sum(s * np.log(s / p)))

def theil_decomposition(df, value_col, weight_col, group_col):
    '''Décompose le Theil T en une composante inter-groupes et une composante intra-groupe.'''
    d = df[[value_col, weight_col, group_col]].dropna()
    d = d[(d[weight_col] > 0) & (d[value_col] > 0)]
    if d[group_col].nunique() < 2:
        return dict(total=np.nan, between=np.nan, within=np.nan, between_share=np.nan)
    total = theil_T(d[value_col], d[weight_col])
    g = d.groupby(group_col).apply(
        lambda t: pd.Series({"V": (t[value_col] * t[weight_col]).sum(),
                             "W": t[weight_col].sum(),
                             "T": theil_T(t[value_col], t[weight_col])}),
        include_groups=False)
    sg = g["V"] / g["V"].sum()
    pg = g["W"] / g["W"].sum()
    between = float(np.sum(sg * np.log(sg / pg)))
    within = float(np.sum(sg * g["T"].fillna(0)))
    return dict(total=total, between=between, within=within,
                between_share=between / total if total and np.isfinite(total) and total > 0 else np.nan)

def cagr(v0, v1, years):
    with np.errstate(divide="ignore", invalid="ignore"):
        r = (np.asarray(v1, float) / np.asarray(v0, float)) ** (1.0 / years) - 1.0
    return np.where(np.isfinite(r), r, np.nan)

# vérification rapide
_eq = gini(np.ones(100)); _un = gini(np.concatenate([np.zeros(99), [100]]))
print(f"Contrôle du Gini — réparti : {_eq:.3f} (≈0)   ·   tout concentré : {_un:.3f} (≈1)")

In [ ]:
# --------------------------------------------------------------------------------------
# 6.2 · Moteur zonal — tous les indicateurs, toutes les zones, en un seul passage
# --------------------------------------------------------------------------------------
R_EARTH_LAT_KM = 110.574     # 1° of latitude, km
R_EARTH_LON_KM = 111.320     # 1° of longitude at the equator, km

def pixel_area_km2(transform, shape):
    '''Surface réelle de chaque pixel d'une grille EPSG:4326, corrigée de la latitude.'''
    rows = np.arange(shape[0], dtype="float64")
    lat = transform.f + (rows + 0.5) * transform.e      # transform.e is negative
    w_km = abs(transform.a) * R_EARTH_LON_KM * np.cos(np.radians(lat))
    h_km = abs(transform.e) * R_EARTH_LAT_KM
    return np.repeat((w_km * h_km)[:, None], shape[1], axis=1)

def zonal_profile(gdf, ntl_path, pop_path, cfg=CONFIG, topcode=None):
    '''
    One row per administrative unit, with the full indicator battery.
    Une ligne par unité administrative, avec toute la batterie d'indicateurs.
    '''
    nf, uc, pmin = cfg["NOISE_FLOOR"], cfg["URBAN_CORE"], cfg["POP_LIT_MIN"]
    rows = []
    with rasterio.open(ntl_path) as nsrc, rasterio.open(pop_path) as psrc:
        for _, rec in gdf.iterrows():
            geom = [mapping(rec.geometry)]
            try:
                n, tr = rio_mask(nsrc, geom, crop=True, filled=True, nodata=np.nan)
                p, _  = rio_mask(psrc, geom, crop=True, filled=True, nodata=np.nan)
            except Exception:                          # geometry fully outside the raster
                continue
            n = n[0].astype("float64"); p = np.nan_to_num(p[0].astype("float64"))
            if n.shape != p.shape:                     # defensive: crop to the overlap
                h = min(n.shape[0], p.shape[0]); w = min(n.shape[1], p.shape[1])
                n, p = n[:h, :w], p[:h, :w]
            A = pixel_area_km2(tr, n.shape)
            valid = np.isfinite(n)
            if valid.sum() == 0:
                continue
            rad, area, pop = n[valid], A[valid], p[valid]
            rad = np.clip(rad, 0, None)
            if topcode is not None:
                rad = np.minimum(rad, topcode)

            lit  = rad >= nf
            core = rad >= uc
            inh  = pop >= pmin
            tot_area = area.sum()
            tot_pop  = pop.sum()
            sol      = rad.sum()

            rows.append({
                "shapeID":  rec["shapeID"],
                "name":     rec["shapeName"],
                # --- étendue ---
                "area_km2": tot_area,
                "n_pixels": int(valid.sum()),
                # --- lumière ---
                "sol":            sol,                                   # sum of radiance
                "rad_km2":        float((rad * area).sum()),             # area-weighted
                "sol_per_km2":    sol / tot_area if tot_area else np.nan,
                "mean_rad":       float(rad.mean()),
                "median_rad":     float(np.median(rad)),
                "p90_rad":        float(np.percentile(rad, 90)),
                "max_rad":        float(rad.max()),
                # --- étendue éclairée ---
                "lit_area_km2":   float(area[lit].sum()),
                "lit_share":      float(area[lit].sum() / tot_area) if tot_area else np.nan,
                "core_area_km2":  float(area[core].sum()),
                "core_share":     float(area[core].sum() / tot_area) if tot_area else np.nan,
                # --- population ---
                "pop":            tot_pop,
                "pop_inhabited":  float(pop[inh].sum()),
                "pop_lit":        float(pop[inh & lit].sum()),
                "pop_dark":       float(pop[inh & ~lit].sum()),
                "pop_lit_share":  float(pop[inh & lit].sum() / pop[inh].sum()) if pop[inh].sum() > 0 else np.nan,
                "pop_density":    tot_pop / tot_area if tot_area else np.nan,
                "sol_per_capita": sol / tot_pop if tot_pop > 0 else np.nan,
                # --- distribution interne ---
                "gini_pixels":    gini(rad, area),
                "gini_pop_light": gini(rad, np.maximum(pop, 1e-6)),
                "cv_rad":         float(rad.std() / rad.mean()) if rad.mean() > 0 else np.nan,
            })
    out = pd.DataFrame(rows)
    if len(out):
        out["sol_share_nat"] = out["sol"] / out["sol"].sum()
        out["pop_share_nat"] = out["pop"] / out["pop"].sum()
        # >1 : la zone capte plus de lumière que son poids démographique
        out["light_pop_ratio"] = out["sol_share_nat"] / out["pop_share_nat"].replace(0, np.nan)
    return out

# ---- seuil d'écrêtage, dérivé du raster de fin de période -------------------------
TOPCODE = float(np.nanpercentile(ARR[REF_YEAR][ARR[REF_YEAR] > 0], CONFIG["TOPCODE_PCT"]))
log(f"Écrêtage au percentile {CONFIG['TOPCODE_PCT']} = {TOPCODE:,.1f} nW·cm⁻²·sr⁻¹", "info")

ZS = {}        # (level, year) -> DataFrame, raw
ZT = {}        # (level, year) -> DataFrame, top-coded
t0 = time.time()
for lvl, g in ADMIN.items():
    for y in CONFIG["YEARS"]:
        ZS[(lvl, y)] = zonal_profile(g, NTL_PATHS[y], POP_PATHS[y])
        ZT[(lvl, y)] = zonal_profile(g, NTL_PATHS[y], POP_PATHS[y], topcode=TOPCODE)
        log(f"{lvl} {y} : {len(ZS[(lvl,y)])} zones")
log(f"statistiques zonales terminées en {time.time()-t0:.1f} s", "ok")

display_cols = ["name", "area_km2", "pop", "sol", "lit_share", "pop_lit_share",
                "sol_per_capita", "gini_pixels"]
print("\nÉchantillon ADM1, fin de période :")
ZS[("ADM1", REF_YEAR)][display_cols].sort_values("sol", ascending=False).head(8)

In [ ]:
# --------------------------------------------------------------------------------------
# 6.3 · Dans quelle mesure l'écrêtage déplace-t-il le tableau national ?
# --------------------------------------------------------------------------------------
lvl = "ADM1"
cmp_rows = []
for y in CONFIG["YEARS"]:
    raw, tc = ZS[(lvl, y)], ZT[(lvl, y)]
    cmp_rows.append({
        "year": y,
        "SoL raw": raw["sol"].sum(),
        "SoL top-coded": tc["sol"].sum(),
        "Δ %": 100 * (tc["sol"].sum() / raw["sol"].sum() - 1),
        "Gini(zones) raw": gini(raw["sol"], raw["pop"]),
        "Gini(zones) top-coded": gini(tc["sol"], tc["pop"]),
        "top zone raw": raw.loc[raw["sol"].idxmax(), "name"],
        "top zone share raw": raw["sol"].max() / raw["sol"].sum(),
    })
TOPCODE_EFFECT = pd.DataFrame(cmp_rows)
print("Effet de l'écrêtage au percentile {:.1f}".format(CONFIG["TOPCODE_PCT"]))
print("=" * 78)
for _, r in TOPCODE_EFFECT.iterrows():
    print(f"{int(r['year'])}:  SoL {r['SoL raw']:>14,.0f} → {r['SoL top-coded']:>14,.0f}  "
          f"({r['Δ %']:+.1f} %)   Gini {r['Gini(zones) raw']:.3f} → {r['Gini(zones) top-coded']:.3f}")
    print(f"        zone la plus lumineuse : {r['top zone raw']}"
          f"  = {r['top zone share raw']:.1%} of national SoL")
print("\nInterprétation :")
print("  Si Δ% est élevé (>5 %), une poignée de pixels domine votre agrégat national.")

---

<div style="background:linear-gradient(120deg,#00553A,#00704A 55%,#00A86A);border-radius:12px;
            padding:26px 32px;color:#fff;font-family:Calibri,sans-serif;">
  <div style="font-size:52px;font-weight:700;color:#F5C242;line-height:1;">07</div>
  <div style="font-size:11px;letter-spacing:3px;font-weight:700;color:#F5C242;margin-top:6px;">
    SEPT SCÉNARIOS</div>
  <div style="font-size:30px;font-weight:700;margin-top:10px;">De la radiance à la décision</div>
  <div style="font-size:16px;font-style:italic;color:#E6F6EE;margin-top:4px;">
    Que peut légitimement dire un raster à un gouvernement ?</div>
</div>

Chaque scénario ci-dessous suit les mêmes quatre temps : *la question*, *la mesure*, *le résultat*, *ce qu'il ne
permet pas de soutenir*. Ce dernier temps n'est pas décoratif : c'est lui qui distingue un produit statistique
d'une jolie visualisation.

In [ ]:
# --------------------------------------------------------------------------------------
# 7.0 · Utilitaires — registre de figures, rattachement spatial, mise en forme
# --------------------------------------------------------------------------------------
FIGS, RESULTS, INSIGHTS = {}, {}, {}

def show_fig(fig, key=None, height=430):
    fig.update_layout(**PLOTLY_LAYOUT, height=height)
    if key:
        FIGS[key] = fig
    try:
        fig.show()
    except Exception:                                 # headless execution
        pass
    return fig

def fmt_n(v, d=0):
    return "—" if v is None or not np.isfinite(v) else f"{v:,.{d}f}"

def fmt_p(v, d=1):
    return "—" if v is None or not np.isfinite(v) else f"{100*v:.{d}f} %"

def assign_parent(child_gdf, parent_gdf, parent_col="parent"):
    '''Rattache chaque unité fille au polygone parent contenant son point représentatif.'''
    c = child_gdf.copy()
    c["geometry"] = c.geometry.representative_point()
    j = gpd.sjoin(c, parent_gdf[["shapeName", "geometry"]].rename(
        columns={"shapeName": parent_col}), how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    return child_gdf.assign(**{parent_col: j[parent_col].values})

if "ADM2" in ADMIN and "ADM1" in ADMIN:
    ADMIN["ADM2"] = assign_parent(ADMIN["ADM2"], ADMIN["ADM1"])
    PARENT = ADMIN["ADM2"].set_index("shapeID")["parent"].to_dict()
    for y in CONFIG["YEARS"]:
        for D in (ZS, ZT):
            D[("ADM2", y)]["parent"] = D[("ADM2", y)]["shapeID"].map(PARENT)
    log("Unités ADM2 rattachées à leur parent ADM1", "ok")

# Jeu de valeurs utilisé par tous les scénarios : écrêté (recommandé) ou brut.
# Les seuils de B et C (plancher de bruit, cœur urbain) sont très en dessous du seuil
# d'écrêtage : ces deux scénarios donnent le même résultat dans les deux cas.
Z = ZT if CONFIG["USE_TOPCODED"] else ZS
Z_LABEL = (f"valeurs écrêtées au {CONFIG['TOPCODE_PCT']}ᵉ percentile"
           if CONFIG["USE_TOPCODED"] else "valeurs brutes, non écrêtées")
log(f"Scénarios calculés sur les {Z_LABEL}", "info")

Y0, Y1 = min(CONFIG["YEARS"]), max(CONFIG["YEARS"])
DT = max(1, Y1 - Y0)
print(f"Référence {Y0} → fin de période {Y1}   ({DT} ans)")

### 🅐 Scénario A · Activité économique

**La question** — Où l'activité économique mesurable se concentre-t-elle, et se concentre-t-elle davantage ?

**La mesure** — La *somme des lumières* (SoL) par zone, et le **ratio lumière/population**
`part de SoL ÷ part de population`. Un ratio supérieur à 1 signifie que la zone émet plus de lumière que son poids
démographique — typiquement l'industrie, les ports, le tourisme, les capitales administratives. En dessous de 1,
c'est l'inverse : des habitants sans activité éclairée correspondante.

**Pourquoi c'est crédible** — La littérature empirique (Henderson, Storeygard et Weil 2012 ; Chen et Nordhaus
2011) situe l'élasticité de la lumière au PIB entre **0,3 à court terme et 1,0 à long terme**, et d'autant plus
forte que la comptabilité nationale est fragile — c'est-à-dire précisément le cas infranational africain. La
lumière est un bon *complément* du PIB là où le PIB est mal mesuré, et un mauvais *substitut* là où il est bien
mesuré.

**Lecture de l'élasticité.** β est publiée avec son intervalle de confiance à 95 %. Le notebook ne conclut à une
concentration (β > 1) ou à une diffusion (β < 1) que si l'intervalle entier est du même côté de 1.

In [ ]:
# --------------------------------------------------------------------------------------
# 🅐 · Activité économique
# --------------------------------------------------------------------------------------
import plotly.express as px

a1 = Z[("ADM1", Y1)].copy()
a0 = Z[("ADM1", Y0)].set_index("shapeID")
a1["sol_0"] = a1["shapeID"].map(a0["sol"])
a1["sol_cagr"] = cagr(a1["sol_0"], a1["sol"], DT)
a1 = a1.sort_values("sol", ascending=False)

nat = dict(
    sol_0=Z[("ADM1", Y0)]["sol"].sum(), sol_1=a1["sol"].sum(),
    pop=a1["pop"].sum(), area=a1["area_km2"].sum())
nat["sol_cagr"] = float(cagr(nat["sol_0"], nat["sol_1"], DT))

# --- élasticité de la lumière à la population (log-log, MCO) -------------------------
d2 = Z[("ADM2", Y1)]
mask = (d2["sol"] > 0) & (d2["pop"] > 0)
X = np.log(d2.loc[mask, "pop"].values); Yv = np.log(d2.loc[mask, "sol"].values)
from scipy import stats as sstats
_lr = sstats.linregress(X, Yv)
beta, alpha = float(_lr.slope), float(_lr.intercept)
beta_ci = (beta - 1.96 * _lr.stderr, beta + 1.96 * _lr.stderr)
r2 = float(_lr.rvalue ** 2)

RESULTS["A"] = dict(national=nat, adm1=a1, elasticity=float(beta), elasticity_ci=beta_ci,
                    r2=r2, n=int(mask.sum()))

print(f"NIVEAU NATIONAL — {CONFIG['COUNTRY_NAME']}")
print("=" * 78)
print(f"  Somme des lumières {Y0} → {Y1} : {nat['sol_0']:,.0f} → {nat['sol_1']:,.0f}"
      f"   ({nat['sol_cagr']:+.2%}/yr)")
print(f"  Population                     : {nat['pop']:,.0f}")
print(f"  Élasticité log(SoL)~log(pop) sur {RESULTS['A']['n']} unités ADM2 :"
      f" β = {beta:.2f}  (IC 95 % {beta_ci[0]:.2f} – {beta_ci[1]:.2f})   R² = {r2:.2f}")
print(f"  → β > 1 : la lumière se concentre plus vite que la population (agglomération, industrie)")
print(f"    β < 1 : la lumière se répartit plus uniformément que la population")

top = a1.head(14).iloc[::-1]
fig = go.Figure()
fig.add_bar(y=top["name"], x=top["sol_share_nat"] * 100, orientation="h",
            marker_color=[PAL["green"] if v >= 1 else PAL["teal"] for v in top["light_pop_ratio"]],
            name="SoL share %", customdata=np.c_[top["pop_share_nat"]*100, top["light_pop_ratio"]],
            hovertemplate="<b>%{y}</b><br>SoL: %{x:.1f}%<br>Pop: %{customdata[0]:.1f}%"
                          "<br>Ratio: %{customdata[1]:.2f}<extra></extra>")
fig.add_scatter(y=top["name"], x=top["pop_share_nat"] * 100, mode="markers",
                marker=dict(symbol="line-ns", size=16, line=dict(color=PAL["ochre"], width=3)),
                name="Population share %")
fig.update_layout(xaxis_title="Share of national total · Part du total national (%)",
                  yaxis_title="")
show_fig(fig, "A_share", height=480)

fig2 = go.Figure()
fig2.add_scatter(x=d2.loc[mask, "pop"], y=d2.loc[mask, "sol"], mode="markers",
                 marker=dict(size=8, color=PAL["green"], opacity=.62,
                             line=dict(width=.5, color="white")),
                 text=d2.loc[mask, "name"],
                 hovertemplate="<b>%{text}</b><br>pop %{x:,.0f}<br>SoL %{y:,.0f}<extra></extra>",
                 name="ADM2")
xs = np.linspace(X.min(), X.max(), 40)
fig2.add_scatter(x=np.exp(xs), y=np.exp(alpha + beta * xs), mode="lines",
                 line=dict(color=PAL["ochre"], width=3, dash="dash"),
                 name=f"β = {beta:.2f} · R² = {r2:.2f}")
fig2.update_xaxes(type="log", title="Population (log)")
fig2.update_yaxes(type="log", title="Sum of Lights (log)")
show_fig(fig2, "A_elasticity", height=430)

INSIGHTS["A"] = {
 "en": (f"National Sum of Lights grew {nat['sol_cagr']:+.1%} per year between {Y0} and {Y1}. "
        f"The light-to-population elasticity across ADM2 units is {beta:.2f} "
        f"(95% CI {beta_ci[0]:.2f}–{beta_ci[1]:.2f}, R²={r2:.2f}): "
        + ("light concentrates faster than people, a signature of agglomeration."
           if beta_ci[0] > 1 else
           "light is spread more evenly than population, suggesting broad-based service coverage."
           if beta_ci[1] < 1 else
           "light cannot be distinguished from proportional to population.")),
 "fr": (f"La somme des lumières nationale a progressé de {nat['sol_cagr']:+.1%} par an entre {Y0} et {Y1}. "
        f"L'élasticité lumière-population entre unités ADM2 vaut {beta:.2f} "
        f"(IC 95 % {beta_ci[0]:.2f}–{beta_ci[1]:.2f}, R²={r2:.2f}) : "
        + ("la lumière se concentre plus vite que la population, signature d'agglomération."
           if beta_ci[0] > 1 else
           "la lumière est répartie plus uniformément que la population."
           if beta_ci[1] < 1 else
           "on ne peut pas distinguer la lumière d'une simple proportionnalité à la population.")),
}
print("\n" + INSIGHTS["A"]["fr"])

> **🚫 Ce que le scénario A ne permet pas de soutenir**
>
> Il ne produit pas un *niveau* de PIB. Il ne capte pas l'activité qui n'émet pas de lumière : agriculture
> vivrière, commerce informel diurne, l'essentiel de l'économie du soin, et — ce qui compte — tout ce qui se
> déroule à l'intérieur d'un bâtiment bien isolé. Une région de petits exploitants n'est pas « économiquement
> inactive » : elle est *non éclairée*. Publier un classement lumineux comme un « classement économique » sans
> cette phrase attachée constitue un détournement de l'indicateur.

### 🅑 Scénario B · Électrification et accès à l'énergie

**La question** — Combien de personnes vivent dans des cellules **habitées mais sans lumière détectable**, et où
se trouvent-elles ?

**La mesure** — Croiser deux rasters : population ≥ `POP_LIT_MIN` et radiance < `NOISE_FLOOR`, puis sommer la
population de l'intersection. C'est le chiffre le plus directement actionnable de tout le notebook, parce que
c'est un *effectif de personnes*, localisé, et directement comparable à la liste-cible d'un programme
d'électrification rurale.

**Ce que c'est, honnêtement** — un indicateur de l'**éclairage extérieur observable**, pas du raccordement
électrique d'un ménage. Un foyer raccordé au réseau mais n'utilisant que des LED intérieures de faible puissance
ne contribue presque rien à la radiance VIIRS. Attendez-vous donc à ce que l'indicateur **sous-estime** l'accès,
et le sous-estime le plus exactement là où l'éclairage est le plus efficace. C'est pour cette raison que le
scénario G le confronte à votre taux d'électrification officiel avant toute publication.

**Marge d'incertitude calculée automatiquement.** Le chiffre dépend presque entièrement du plancher de bruit. Le
notebook le recalcule pour chaque valeur de `NOISE_SENSITIVITY` (0,15 · 0,25 · 0,50 par défaut) et publie
l'intervalle obtenu à côté de la valeur centrale.

In [ ]:
# --------------------------------------------------------------------------------------
# 🅑 · Indicateur d'électrification
# --------------------------------------------------------------------------------------
b1 = Z[("ADM1", Y1)].sort_values("pop_lit_share")
b0 = Z[("ADM1", Y0)].set_index("shapeID")
b1 = b1.assign(pop_lit_share_0=b1["shapeID"].map(b0["pop_lit_share"]),
               pop_dark_0=b1["shapeID"].map(b0["pop_dark"]))
b1["delta_pts"] = 100 * (b1["pop_lit_share"] - b1["pop_lit_share_0"])

nat_dark_1 = Z[("ADM1", Y1)]["pop_dark"].sum()
nat_dark_0 = Z[("ADM1", Y0)]["pop_dark"].sum()
nat_inh    = Z[("ADM1", Y1)]["pop_inhabited"].sum()
nat_share  = 1 - nat_dark_1 / nat_inh if nat_inh else np.nan

worst2 = Z[("ADM2", Y1)].nlargest(12, "pop_dark")[
    ["name", "parent", "pop", "pop_dark", "pop_lit_share", "pop_density"]] \
    if "parent" in Z[("ADM2", Y1)] else Z[("ADM2", Y1)].nlargest(12, "pop_dark")

RESULTS["B"] = dict(adm1=b1, national_dark=nat_dark_1, national_dark_0=nat_dark_0,
                    national_lit_share=nat_share, worst_adm2=worst2)

print("INDICATEUR D'ÉLECTRIFICATION")
print("=" * 78)
print(f"  Population habitante                 : {nat_inh:,.0f}")
print(f"  En cellules éclairées {Y1}           : {nat_inh - nat_dark_1:,.0f}   ({nat_share:.1%})")
print(f"  ⚠️  En cellules habitées NON éclairées {Y1} : {nat_dark_1:,.0f}")
print(f"      même chiffre en {Y0}             : {nat_dark_0:,.0f}   "
      f"({nat_dark_1 - nat_dark_0:+,.0f} people / personnes)")

fig = go.Figure()
fig.add_bar(y=b1["name"], x=b1["pop_lit_share_0"] * 100, orientation="h",
            marker_color=PAL["sage"], name=f"{Y0}")
fig.add_bar(y=b1["name"], x=(b1["pop_lit_share"] - b1["pop_lit_share_0"]) * 100,
            orientation="h", base=b1["pop_lit_share_0"] * 100,
            marker_color=[PAL["green"] if v >= 0 else PAL["brick"] for v in b1["delta_pts"]],
            name=f"Δ {Y0}→{Y1}",
            customdata=np.c_[b1["pop_dark"]],
            hovertemplate="<b>%{y}</b><br>Δ %{x:+.1f} pts<br>unlit pop: %{customdata[0]:,.0f}<extra></extra>")
fig.update_layout(barmode="overlay",
                  xaxis_title="Population in lit inhabited cells · Population en cellules habitées éclairées (%)",
                  yaxis_title="")
show_fig(fig, "B_access", height=480)

INSIGHTS["B"] = {
 "en": (f"{nat_dark_1:,.0f} people — {1-nat_share:.1%} of the inhabited population — live in cells that are "
        f"inhabited but emit no detectable night-time light, against {nat_dark_0:,.0f} in {Y0}. "
        f"The largest concentrations are listed in the table; they are the natural starting point for a "
        f"targeted off-grid or grid-extension programme."),
 "fr": (f"{nat_dark_1:,.0f} personnes — {1-nat_share:.1%} de la population habitante — vivent dans des cellules "
        f"habitées sans lumière nocturne détectable, contre {nat_dark_0:,.0f} en {Y0}. "
        f"Les plus fortes concentrations figurent dans le tableau : ce sont les points de départ naturels d'un "
        f"programme ciblé d'électrification hors réseau ou d'extension du réseau."),
}

# ---- sensibilité au plancher de bruit : la vraie marge d'incertitude de l'indicateur -----
_a1 = ARR[Y1]
_rows = []
if _a1.shape == POP.shape:
    _inh = POP >= CONFIG["POP_LIT_MIN"]
    _tot = float(POP[_inh].sum())
    for _nf in CONFIG["NOISE_SENSITIVITY"]:
        _d = float(POP[_inh & (_a1 < _nf)].sum())
        _rows.append({"noise_floor": _nf, "pop_dark": _d, "share_dark": _d / max(_tot, 1e-9)})
SENS_B = pd.DataFrame(_rows)
RESULTS["B"]["sensitivity"] = SENS_B
if len(SENS_B):
    print("\nSENSIBILITÉ AU PLANCHER DE BRUIT (population en cellules habitées non éclairées)")
    for _, r in SENS_B.iterrows():
        tag = "  ← valeur retenue" if abs(r["noise_floor"] - CONFIG["NOISE_FLOOR"]) < 1e-9 else ""
        print(f"  plancher {r['noise_floor']:.2f} : {r['pop_dark']:>12,.0f}  ({r['share_dark']:.1%}){tag}")
    _lo, _hi = SENS_B["pop_dark"].min(), SENS_B["pop_dark"].max()
    _nf0, _nf1 = min(CONFIG["NOISE_SENSITIVITY"]), max(CONFIG["NOISE_SENSITIVITY"])
    RESULTS["B"]["range"] = (float(_lo), float(_hi))
    INSIGHTS["B"]["en"] += (f" Depending on the noise floor ({_nf0}–{_nf1} nW·cm⁻²·sr⁻¹), this figure ranges "
                            f"from {_lo:,.0f} to {_hi:,.0f}: that is the indicator's real uncertainty.")
    INSIGHTS["B"]["fr"] += (f" Selon le plancher de bruit retenu ({_nf0} à {_nf1} nW·cm⁻²·sr⁻¹), ce chiffre "
                            f"varie de {_lo:,.0f} à {_hi:,.0f} : c'est la marge d'incertitude réelle de l'indicateur.")

print("\nADM2 à plus forte population habitée non éclairée :")
worst2.head(8)

In [ ]:
# --------------------------------------------------------------------------------------
# 🅑.2 · Où se trouvent précisément les cellules habitées non éclairées ?
# --------------------------------------------------------------------------------------
ntl1, tr1, bn1 = read_ntl(NTL_PATHS[Y1])
dark_inhab = (POP >= CONFIG["POP_LIT_MIN"]) & (ntl1 < CONFIG["NOISE_FLOOR"])
lit_inhab  = (POP >= CONFIG["POP_LIT_MIN"]) & (ntl1 >= CONFIG["NOISE_FLOOR"])

rgb = np.zeros(ntl1.shape + (4,), dtype="float32")
rgb[..., 3] = 0.0
rgb[lit_inhab]  = [0.0, 0.659, 0.416, 0.85]     # AfDB green
rgb[dark_inhab] = [0.722, 0.231, 0.180, 0.95]   # brick red

fig, ax = plt.subplots(figsize=(8.6, 7.4))
ax.imshow(np.where(ntl1 <= 0, np.nan, ntl1), cmap="Greys_r",
          norm=LogNorm(vmin=0.05, vmax=np.nanpercentile(ntl1, 99.9)),
          extent=[bn1.left, bn1.right, bn1.bottom, bn1.top], alpha=.35)
ax.imshow(rgb, extent=[bn1.left, bn1.right, bn1.bottom, bn1.top], interpolation="nearest")
ADMIN.get("ADM1", ADMIN["ADM0"]).boundary.plot(ax=ax, color=PAL["ink"], lw=.6, alpha=.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#00A86A", label="Habitée et éclairée"),
                   Patch(color="#B83B2E", label="Habitée et NON éclairée")],
          loc="lower left", fontsize=9, frameon=True, facecolor="white", framealpha=.9)
ax.set_title(f"Carte du déficit d'accès — {CONFIG['COUNTRY_NAME']} {Y1}",
             loc="left", fontsize=12.5)
ax.set_xlabel("lon"); ax.set_ylabel("lat")
plt.tight_layout(); plt.savefig(OUT / "figures" / "03_access_gap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Pixels habités non éclairés : {dark_inhab.sum():,}  ·  population {POP[dark_inhab].sum():,.0f}")

### 🅒 Scénario C · Urbanisation et empreinte éclairée

**La question** — L'empreinte éclairée du pays s'est-elle étendue, et l'a-t-elle fait en **densifiant les villes
existantes** ou en **s'étalant vers l'extérieur** ?

**La mesure** — Une matrice de transition des pixels en quatre classes entre l'année de référence et l'année
finale :

| Classe | Référence | Fin de période | Se lit comme |
|---|---|---|---|
| **Éclairé stable** | éclairé | éclairé | zone urbaine consolidée, desservie |
| **Nouvellement éclairé** | non éclairé | éclairé | expansion — nouvel habitat, nouvelle électrification, nouvelle infrastructure |
| **Lumière perdue** | éclairé | non éclairé | dépeuplement, conflit, coupure, site désaffecté, **ou changement de capteur** |
| **Sombre stable** | non éclairé | non éclairé | non desservi |

Puis un **ratio d'étalement** : croissance de la surface éclairée divisée par croissance de la surface du cœur
urbain. Au-dessus de 1, le pays s'étale ; en dessous, il se densifie. Densification et étalement ont des
implications opposées sur le coût unitaire de l'eau, de la voirie et du raccordement au réseau — c'est pourquoi
les aménageurs s'intéressent bien davantage à ce ratio qu'à un total.

> ⚠️ **Lire « lumière perdue » avec méfiance** — un pixel peut perdre de la lumière parce qu'une ville s'est
> vidée, **ou** parce que la version du produit a changé, **ou** parce que ce mois-là était plus nuageux. Ne
> publiez jamais un constat de perte de lumière sans avoir vérifié la version du produit et le nombre
> d'observations sans nuage pour les deux dates.

In [ ]:
# --------------------------------------------------------------------------------------
# 🅒 · Urbanisation — matrice de transition des pixels
# --------------------------------------------------------------------------------------
# On garantit que les deux années partagent exactement la même grille avant de différencier
ALIGNED_Y0 = OUT / "raster" / f"ntl_{CONFIG['ISO3']}_{Y0}_on_{Y1}grid.tif"
with rasterio.open(NTL_PATHS[Y0]) as s0, rasterio.open(NTL_PATHS[Y1]) as s1:
    same_grid = (s0.width, s0.height, s0.transform) == (s1.width, s1.height, s1.transform)
P0 = NTL_PATHS[Y0] if same_grid else align_to(NTL_PATHS[Y0], NTL_PATHS[Y1], ALIGNED_Y0,
                                              resampling=Resampling.bilinear)
ntl0, tr0, bn0 = read_ntl(P0)
nf = CONFIG["NOISE_FLOOR"]

with rasterio.open(NTL_PATHS[Y1]) as s1:
    AREA = pixel_area_km2(s1.transform, (s1.height, s1.width))

lit0, lit1 = ntl0 >= nf, ntl1 >= nf
CLASSES = {
    "stable_lit":  lit0 & lit1,
    "newly_lit":   (~lit0) & lit1,
    "lost_light":  lit0 & (~lit1),
    "stable_dark": (~lit0) & (~lit1),
}
trans = pd.DataFrame([{
    "class": k,
    "pixels": int(v.sum()),
    "area_km2": float(AREA[v].sum()),
    "share_area": float(AREA[v].sum() / AREA.sum()),
    "population": float(POP[v].sum()),
} for k, v in CLASSES.items()])

core_area_0 = float(AREA[ntl0 >= CONFIG["URBAN_CORE"]].sum())
core_area_1 = float(AREA[ntl1 >= CONFIG["URBAN_CORE"]].sum())
lit_area_0  = float(AREA[lit0].sum()); lit_area_1 = float(AREA[lit1].sum())
sprawl = ((lit_area_1 / lit_area_0) - 1) / max((core_area_1 / core_area_0) - 1, 1e-9) \
    if lit_area_0 > 0 and core_area_0 > 0 else np.nan

RESULTS["C"] = dict(transition=trans, sprawl_ratio=float(sprawl),
                    lit_area=(lit_area_0, lit_area_1), core_area=(core_area_0, core_area_1))

print("TRANSITION DE L'EMPREINTE ÉCLAIRÉE")
print("=" * 78)
for _, r in trans.iterrows():
    print(f"  {r['class']:<12} {r['area_km2']:>12,.0f} km²  ({r['share_area']:>5.1%})"
          f"   pop {r['population']:>12,.0f}")
print(f"\n  Surface éclairée {Y0}→{Y1} : {lit_area_0:,.0f} → {lit_area_1:,.0f} km²"
      f"  ({(lit_area_1/lit_area_0-1):+.1%})")
print(f"  Cœur urbain      {Y0}→{Y1} : {core_area_0:,.0f} → {core_area_1:,.0f} km²"
      f"  ({(core_area_1/core_area_0-1):+.1%})")
print(f"  Ratio d'étalement : {sprawl:.2f}"
      f"   → {'SPREADING / étalement' if sprawl > 1 else 'DENSIFYING / densification'}")

colr = {"stable_lit": PAL["green"], "newly_lit": PAL["gold"],
        "lost_light": PAL["brick"], "stable_dark": PAL["sage"]}
fig = go.Figure()
for _, r in trans.iterrows():
    fig.add_bar(x=[r["area_km2"]], y=["Area · Surface (km²)"], orientation="h",
                marker_color=colr[r["class"]], name=r["class"],
                hovertemplate=f"<b>{r['class']}</b><br>%{{x:,.0f}} km²<extra></extra>")
for _, r in trans.iterrows():
    fig.add_bar(x=[r["population"]], y=["Population"], orientation="h",
                marker_color=colr[r["class"]], name=r["class"], showlegend=False,
                hovertemplate=f"<b>{r['class']}</b><br>%{{x:,.0f}} hab.<extra></extra>")
fig.update_layout(barmode="stack", xaxis_title="", yaxis_title="")
show_fig(fig, "C_transition", height=300)

INSIGHTS["C"] = {
 "en": (f"The lit footprint moved from {lit_area_0:,.0f} to {lit_area_1:,.0f} km² "
        f"({(lit_area_1/lit_area_0-1):+.1%}), with {trans.loc[trans['class']=='newly_lit','area_km2'].iloc[0]:,.0f} km² "
        f"newly lit and {trans.loc[trans['class']=='lost_light','area_km2'].iloc[0]:,.0f} km² losing detectable light. "
        f"The sprawl ratio of {sprawl:.2f} indicates a pattern of "
        + ("outward spread rather than densification." if sprawl > 1 else "densification rather than outward spread.")),
 "fr": (f"L'empreinte éclairée est passée de {lit_area_0:,.0f} à {lit_area_1:,.0f} km² "
        f"({(lit_area_1/lit_area_0-1):+.1%}), dont {trans.loc[trans['class']=='newly_lit','area_km2'].iloc[0]:,.0f} km² "
        f"nouvellement éclairés et {trans.loc[trans['class']=='lost_light','area_km2'].iloc[0]:,.0f} km² ayant perdu "
        f"toute lumière détectable. Le ratio d'étalement de {sprawl:.2f} traduit "
        + ("un étalement plutôt qu'une densification." if sprawl > 1 else "une densification plutôt qu'un étalement.")),
}
print("\n" + INSIGHTS["C"]["fr"])

In [ ]:
# --------------------------------------------------------------------------------------
# 🅒.2 · Carte de transition — où le changement a eu lieu
# --------------------------------------------------------------------------------------
cls = np.zeros(ntl1.shape, dtype="uint8")
cls[CLASSES["stable_dark"]] = 0
cls[CLASSES["stable_lit"]]  = 1
cls[CLASSES["newly_lit"]]   = 2
cls[CLASSES["lost_light"]]  = 3
cmap4 = matplotlib.colors.ListedColormap(["#0B100E", PAL["green"], PAL["gold"], PAL["brick"]])

fig, ax = plt.subplots(figsize=(8.8, 7.4))
ax.imshow(cls, cmap=cmap4, vmin=0, vmax=3, interpolation="nearest",
          extent=[bn1.left, bn1.right, bn1.bottom, bn1.top])
ADMIN.get("ADM1", ADMIN["ADM0"]).boundary.plot(ax=ax, color="white", lw=.6, alpha=.45)
ax.legend(handles=[Patch(color="#00A86A", label="Éclairé stable"),
                   Patch(color="#F5C242", label="Nouvellement éclairé"),
                   Patch(color="#B83B2E", label="Lumière perdue"),
                   Patch(color="#0B100E", label="Sombre stable")],
          loc="lower left", fontsize=9, frameon=True, facecolor="white", framealpha=.92)
ax.set_title(f"Transition de l'empreinte éclairée {Y0} → {Y1}",
             loc="left", fontsize=12.5)
ax.set_xlabel("lon"); ax.set_ylabel("lat")
plt.tight_layout(); plt.savefig(OUT / "figures" / "04_transition.png", dpi=150, bbox_inches="tight")
plt.show()

### 🅓 Scénario D · Inégalités spatiales

**La question** — La lumière, et donc l'activité qu'elle approxime, se concentre-t-elle ou se diffuse-t-elle ? Et
cette inégalité se joue-t-elle *entre* les régions ou *à l'intérieur* de chacune ?

**Les mesures**

- Le **Gini de la lumière par habitant**, pondéré par la population, sur les unités ADM2. La valeur 0 signifie que
  chacun vit dans un lieu également éclairé ; la valeur 1, que toute la lumière est concentrée en un seul point.
- La **courbe de Lorenz** de la lumière cumulée contre la population cumulée — l'image derrière le Gini.
- La **décomposition de Theil T** en une composante *entre ADM1* et une composante *intra-ADM1*. C'est la partie
  qui change la politique publique : si 80 % de l'inégalité est *intra-régionale*, une formule de péréquation
  entre régions n'y changera rien.
- La **β-convergence** : régresser le taux de croissance de chaque unité ADM2 sur son niveau initial. Une pente
  négative signifie que les lieux les moins éclairés ont crû plus vite — il y a convergence.

**Significativité.** La variation du Gini est accompagnée d'un intervalle de confiance par bootstrap apparié
(500 tirages d'unités) et la pente de convergence de son intervalle à 95 %. Si l'intervalle contient zéro, le
notebook écrit « écart non significatif » plutôt que « se sont creusées » ou « convergence ».

In [ ]:
# --------------------------------------------------------------------------------------
# 🅓 · Inégalités spatiales
# --------------------------------------------------------------------------------------
lvl_ineq = "ADM2" if ("ADM2", Y1) in Z and len(Z[("ADM2", Y1)]) > 6 else "ADM1"
d_1 = Z[(lvl_ineq, Y1)].copy(); d_0 = Z[(lvl_ineq, Y0)].copy()

gini_1 = gini(d_1["sol_per_capita"], d_1["pop"])
gini_0 = gini(d_0["sol_per_capita"], d_0["pop"])
theil_1 = theil_decomposition(d_1, "sol_per_capita", "pop", "parent") if "parent" in d_1 else \
          dict(total=theil_T(d_1["sol_per_capita"], d_1["pop"]), between=np.nan,
               within=np.nan, between_share=np.nan)

q0, l0 = lorenz(d_0["sol_per_capita"], d_0["pop"])
q1, l1 = lorenz(d_1["sol_per_capita"], d_1["pop"])

# --- β-convergence -------------------------------------------------------------------
m = d_1[["shapeID", "sol_per_capita"]].merge(
    d_0[["shapeID", "sol_per_capita", "pop"]], on="shapeID", suffixes=("_1", "_0"))
m = m[(m["sol_per_capita_0"] > 0) & (m["sol_per_capita_1"] > 0)]
m["growth"] = cagr(m["sol_per_capita_0"], m["sol_per_capita_1"], DT)
bx = np.log(m["sol_per_capita_0"].values); by = m["growth"].values
ok = np.isfinite(bx) & np.isfinite(by)
from scipy import stats as sstats
_lc = sstats.linregress(bx[ok], by[ok])
beta_conv, a_conv, r_conv = float(_lc.slope), float(_lc.intercept), float(_lc.rvalue)
conv_ci = (beta_conv - 1.96 * _lc.stderr, beta_conv + 1.96 * _lc.stderr)

# Variation du Gini : bootstrap apparié sur les unités (500 tirages)
_pair = d_1[["shapeID", "sol_per_capita", "pop"]].merge(
    d_0[["shapeID", "sol_per_capita", "pop"]], on="shapeID", suffixes=("_1", "_0"))
_rng = np.random.default_rng(CONFIG["RANDOM_SEED"])
_diffs = []
for _ in range(500):
    _b = _pair.iloc[_rng.integers(0, len(_pair), len(_pair))]
    _diffs.append(gini(_b["sol_per_capita_1"].values, _b["pop_1"].values)
                  - gini(_b["sol_per_capita_0"].values, _b["pop_0"].values))
gini_diff_ci = tuple(float(v) for v in np.nanpercentile(_diffs, [2.5, 97.5]))
gini_change_sig = not (gini_diff_ci[0] <= 0 <= gini_diff_ci[1])

RESULTS["D"] = dict(level=lvl_ineq, gini_0=gini_0, gini_1=gini_1, theil=theil_1,
                    gini_diff_ci=gini_diff_ci, gini_change_significant=gini_change_sig,
                    beta_convergence=float(beta_conv), convergence_ci=conv_ci, r_convergence=r_conv)

print(f"INÉGALITÉS SPATIALES au niveau {lvl_ineq}")
print("=" * 78)
print(f"  Gini(lumière/hab., pondéré population)  {Y0} : {gini_0:.3f}   {Y1} : {gini_1:.3f}"
      f"   ({gini_1-gini_0:+.3f} ; IC 95 % {gini_diff_ci[0]:+.3f} à {gini_diff_ci[1]:+.3f}"
      f" → {'significatif' if gini_change_sig else 'non significatif'})")
if np.isfinite(theil_1.get("between_share", np.nan)):
    print(f"  Theil T {Y1} : {theil_1['total']:.4f}"
          f"   between ADM1 {theil_1['between']:.4f} ({theil_1['between_share']:.0%})"
          f"   within {theil_1['within']:.4f} ({1-theil_1['between_share']:.0%})")
print(f"  Pente de β-convergence : {beta_conv:+.4f}  (IC 95 % {conv_ci[0]:+.4f} à {conv_ci[1]:+.4f},"
      f" r = {r_conv:+.2f})  → "
      f"{'CONVERGENCE' if conv_ci[1] < 0 else 'DIVERGENCE' if conv_ci[0] > 0 else 'NON SIGNIFICATIVE'}")

fig = go.Figure()
fig.add_scatter(x=q1 * 100, y=q1 * 100, mode="lines", name="Perfect equality · égalité parfaite",
                line=dict(color=PAL["sage"], dash="dot", width=2))
fig.add_scatter(x=q0 * 100, y=l0 * 100, mode="lines", name=f"{Y0}",
                line=dict(color=PAL["teal"], width=3))
fig.add_scatter(x=q1 * 100, y=l1 * 100, mode="lines", name=f"{Y1}",
                line=dict(color=PAL["green"], width=3.5), fill="tonexty",
                fillcolor="rgba(0,168,106,0.10)")
fig.update_layout(xaxis_title="Cumulative population · Population cumulée (%)",
                  yaxis_title="Cumulative light · Lumière cumulée (%)")
show_fig(fig, "D_lorenz", height=430)

fig2 = go.Figure()
fig2.add_scatter(x=np.exp(bx[ok]), y=by[ok] * 100, mode="markers",
                 marker=dict(size=9, color=PAL["green"], opacity=.62,
                             line=dict(width=.5, color="white")),
                 text=m.loc[ok, "shapeID"], name=lvl_ineq,
                 hovertemplate="%{text}<br>initial %{x:,.3f}<br>growth %{y:+.1f} %/yr<extra></extra>")
xs = np.linspace(bx[ok].min(), bx[ok].max(), 30)
fig2.add_scatter(x=np.exp(xs), y=(a_conv + beta_conv * xs) * 100, mode="lines",
                 line=dict(color=PAL["ochre"], width=3, dash="dash"),
                 name=f"slope {beta_conv:+.3f}")
fig2.update_xaxes(type="log", title=f"Light per capita in {Y0} (log) · Lumière par habitant")
fig2.update_yaxes(title="Annual growth · Croissance annuelle (%)")
show_fig(fig2, "D_convergence", height=430)

if gini_change_sig:
    _dir_en = "narrowed" if gini_1 < gini_0 else "widened"
    _dir_fr = "resserrées" if gini_1 < gini_0 else "creusées"
else:
    _dir_en = "did not change significantly, moving"
    _dir_fr = "maintenues (écart non significatif à 95 %)"
INSIGHTS["D"] = {
 "en": (f"Population-weighted inequality in light per capita across {lvl_ineq} units {_dir_en} from "
        f"{gini_0:.3f} to {gini_1:.3f} (Gini). "
        + (f"{theil_1['between_share']:.0%} of total inequality sits between ADM1 regions and "
           f"{1-theil_1['between_share']:.0%} within them. "
           if np.isfinite(theil_1.get('between_share', np.nan)) else "")
        + ("Initially dimmer units grew faster, indicating convergence."
           if conv_ci[1] < 0 else "Initially brighter units grew faster, indicating divergence."
           if conv_ci[0] > 0 else "There is no significant convergence or divergence.")),
 "fr": (f"Les inégalités de lumière par habitant pondérées par la population entre unités {lvl_ineq} se sont "
        f"{_dir_fr}, de {gini_0:.3f} à {gini_1:.3f} (Gini). "
        + (f"{theil_1['between_share']:.0%} de l'inégalité totale se situe entre régions ADM1 et "
           f"{1-theil_1['between_share']:.0%} à l'intérieur de celles-ci. "
           if np.isfinite(theil_1.get('between_share', np.nan)) else "")
        + ("Les unités initialement moins éclairées ont crû plus vite : convergence."
           if conv_ci[1] < 0 else "Les unités initialement plus éclairées ont crû plus vite : divergence."
           if conv_ci[0] > 0 else "Aucune convergence ni divergence significative.")),
}
print("\n" + INSIGHTS["D"]["fr"])

### 🅔 Scénario E · Détection de changement

**La question** — Quels districts ont changé, et ce changement se distingue-t-il du **bruit ordinaire** ?

**La mesure** — Le taux de croissance annuel composé de la SoL par unité ADM2, puis un **z-score robuste**
construit sur la *médiane* et l'*écart absolu médian* plutôt que sur la moyenne et l'écart-type. La raison est
méthodologique et non esthétique : une poignée de districts extrêmes — la capitale, un champ pétrolier, une mine
nouvelle — gonfle tellement l'écart-type que plus rien d'autre ne paraît significatif. L'écart absolu médian y est
insensible.

Nous signalons |z| > 2,5 comme changement candidat. **Candidat**, et non confirmé : le signalement ouvre une
enquête, il ne la conclut pas.

**Toutes les années, pas seulement deux.** Une croissance entre deux dates dépend entièrement de ces deux
dates : une année de référence anormalement sombre fabrique une « forte hausse ». Avec `TREND_ALL_YEARS = True`, le
notebook télécharge toutes les années intermédiaires et mesure la croissance entre la **médiane des trois
premières années** et la **médiane des trois dernières** : une année aberrante ne pèse plus, et une rupture durable
reste visible.

Une **pente de Theil-Sen** (médiane de toutes les pentes entre paires d'années) et son test de Kendall sont publiés
à côté, dans les colonnes `trend_*`. Elle décrit la tendance de fond mais n'est pas utilisée pour signaler les
changements : elle masquerait une rupture survenue juste après l'année de référence, puisque la plupart des paires
d'années se situent après la rupture. La croissance entre les deux dates extrêmes reste dans `growth_2dates`.

**Bases trop faibles.** Une unité presque sombre au départ peut afficher +300 % pour trois lampadaires. Sous le
percentile `CHANGE_MIN_BASE_PCT` de la SoL initiale (10ᵉ par défaut), l'unité est affichée — en gris — mais
jamais signalée.

In [ ]:
# --------------------------------------------------------------------------------------
# 🅔.0 · Série annuelle complète — tendance robuste plutôt que deux dates
# --------------------------------------------------------------------------------------
# Une croissance calculée entre deux années dépend entièrement de ces deux années : une
# année de référence exceptionnellement sombre suffit à fabriquer une « forte hausse ».
# On télécharge donc toutes les années intermédiaires et on calcule, pour chaque unité :
#   · la croissance entre la MÉDIANE des k premières et des k dernières années (k = 3) :
#     robuste à une année aberrante, et elle capte une rupture durable ;
#   · une pente de Theil-Sen sur log(SoL), avec test de Kendall : la tendance de fond.
# La détection de changement (E) utilise la première ; la pente est publiée à côté.
from scipy import stats as sstats
from rasterio import features

TREND, NAT_SERIES, TREND_YEARS_USED = None, None, []
lvl_tr = "ADM2" if "ADM2" in ADMIN else "ADM1"
TREND_YEARS = list(range(Y0, Y1 + 1)) if CONFIG["TREND_ALL_YEARS"] else []

if len(TREND_YEARS) >= 4:
    g2 = ADMIN[lvl_tr].reset_index(drop=True)
    with rasterio.open(NTL_PATHS[Y1]) as _s:
        _shape, _tr = (_s.height, _s.width), _s.transform
    _ids = {sid: i + 1 for i, sid in enumerate(g2["shapeID"])}
    LAB = features.rasterize(((geom, _ids[sid]) for geom, sid in zip(g2.geometry, g2["shapeID"])),
                             out_shape=_shape, transform=_tr, fill=0, dtype="int32")
    _sol = {}
    for y in TREND_YEARS:
        if y in ARR:
            a = ARR[y]                                   # déjà débarrassé des torchères
        else:
            try:
                a, _, _ = read_ntl(fetch_ntl(y))
            except Exception as e:                       # noqa: BLE001
                log(f"année {y} ignorée pour la tendance : {e}", "warn")
                continue
            if a.shape != _shape:
                log(f"année {y} ignorée : grille différente de {Y1}", "warn")
                continue
            if CONFIG["MASK_FLARES"] and FLARE_STATS["applied"]:
                a = a.copy(); a[FLARE_MASK] = 0.0
        if CONFIG["USE_TOPCODED"]:
            a = np.minimum(a, TOPCODE)
        _sol[y] = np.bincount(LAB.ravel(), weights=a.ravel(), minlength=len(_ids) + 1)[1:]

    TREND_YEARS_USED = sorted(_sol)
    if len(TREND_YEARS_USED) >= 4:
        yrs = np.array(TREND_YEARS_USED, dtype=float)
        M = np.vstack([_sol[y] for y in TREND_YEARS_USED]).T          # unités × années
        KWIN = 3 if len(yrs) >= 6 else 1
        span = float(np.median(yrs[-KWIN:]) - np.median(yrs[:KWIN]))
        rows = []
        for k, sid in enumerate(g2["shapeID"]):
            v = M[k]; ok = v > 0
            s0, s1 = np.median(v[:KWIN]), np.median(v[-KWIN:])
            pg = (s1 / s0) ** (1 / span) - 1 if (s0 > 0 and s1 > 0 and span > 0) else np.nan
            if ok.sum() < 4:
                rows.append((sid, pg, np.nan, np.nan, np.nan, np.nan, int(ok.sum())))
                continue
            ts = sstats.theilslopes(np.log(v[ok]), yrs[ok])
            _tau, _p = sstats.kendalltau(yrs[ok], v[ok])
            rows.append((sid, pg, np.exp(ts[0]) - 1, np.exp(ts[2]) - 1, np.exp(ts[3]) - 1,
                         float(_p), int(ok.sum())))
        TREND = pd.DataFrame(rows, columns=["shapeID", "period_growth", "trend_growth", "trend_lo",
                                            "trend_hi", "trend_p", "trend_n"])
        NAT_SERIES = pd.Series(M.sum(axis=0), index=TREND_YEARS_USED)

        print(f"Tendance estimée sur {len(TREND_YEARS_USED)} années ({TREND_YEARS_USED[0]}–"
              f"{TREND_YEARS_USED[-1]}) pour {TREND['trend_growth'].notna().sum()} unités {lvl_tr}")
        print(f"  tendance significative (Kendall, p < 0,05) : {(TREND['trend_p'] < .05).sum()} unités")

        fig = go.Figure()
        fig.add_scatter(x=NAT_SERIES.index, y=NAT_SERIES.values, mode="lines+markers",
                        line=dict(color=PAL["green"], width=3), marker=dict(size=8),
                        hovertemplate="%{x} : %{y:,.0f}<extra></extra>", name="SoL")
        for yy in (Y0, Y1):
            fig.add_vline(x=yy, line=dict(color=PAL["ochre"], dash="dot", width=1.4))
        fig.update_layout(xaxis_title="", yaxis_title="Sum of Lights · Somme des lumières",
                          xaxis=dict(dtick=1), showlegend=False)
        show_fig(fig, "E_series", height=330)
    else:
        log("Pas assez d'années disponibles : la tendance est remplacée par la croissance entre deux dates.",
            "warn")
else:
    log("Tendance annuelle désactivée (TREND_ALL_YEARS = False) ou période trop courte.", "info")

In [ ]:
# --------------------------------------------------------------------------------------
# 🅔 · Détection de changement par z-score robuste
# --------------------------------------------------------------------------------------
lvl_chg = "ADM2" if ("ADM2", Y1) in Z and len(Z[("ADM2", Y1)]) > 6 else "ADM1"
c1 = Z[(lvl_chg, Y1)][["shapeID", "name", "sol", "pop", "area_km2"] +
                       (["parent"] if "parent" in Z[(lvl_chg, Y1)] else [])].copy()
c0 = Z[(lvl_chg, Y0)].set_index("shapeID")
c1["sol_0"] = c1["shapeID"].map(c0["sol"])
c1["growth"] = cagr(c1["sol_0"], c1["sol"], DT)
c1["abs_change"] = c1["sol"] - c1["sol_0"]
c1["growth_2dates"] = c1["growth"]
GROWTH_METHOD = f"croissance annuelle composée entre {Y0} et {Y1}"
if TREND is not None and lvl_tr == lvl_chg:
    c1 = c1.merge(TREND, on="shapeID", how="left")
    c1["growth"] = c1["period_growth"].where(c1["period_growth"].notna(), c1["growth_2dates"])
    _k = KWIN
    GROWTH_METHOD = (f"croissance annuelle entre la médiane des années "
                     f"{TREND_YEARS_USED[0]}–{TREND_YEARS_USED[_k-1]} et celle des années "
                     f"{TREND_YEARS_USED[-_k]}–{TREND_YEARS_USED[-1]}"
                     if _k > 1 else f"croissance annuelle composée entre {Y0} et {Y1}")

g = c1["growth"].replace([np.inf, -np.inf], np.nan)
med = np.nanmedian(g)
mad = np.nanmedian(np.abs(g - med))
sigma = 1.4826 * mad if mad > 0 else np.nanstd(g)
c1["z_robust"] = (g - med) / (sigma if sigma > 0 else np.nan)

# Une unité presque sombre au départ peut afficher +300 % sur trois lampadaires : son taux
# de croissance n'a pas de sens statistique. Sous le percentile CHANGE_MIN_BASE_PCT de la
# SoL initiale, l'unité est affichée mais jamais signalée.
base_min = float(np.nanpercentile(c1["sol_0"], CONFIG["CHANGE_MIN_BASE_PCT"]))
c1["eligible"] = c1["sol_0"] >= base_min
c1["z_signal"] = c1["z_robust"].where(c1["eligible"])
c1["flag"] = np.select(
    [~c1["eligible"], c1["z_signal"] >= 2.5, c1["z_signal"] <= -2.5],
    ["low initial base · base initiale trop faible",
     "significant gain · hausse significative", "significant loss · baisse significative"],
    default="within normal range · dans la plage normale")

RESULTS["E"] = dict(level=lvl_chg, table=c1, median_growth=float(med), sigma=float(sigma),
                    method=GROWTH_METHOD, years=TREND_YEARS_USED,
                    base_min=base_min, n_low_base=int((~c1["eligible"]).sum()))

print(f"DÉTECTION DE CHANGEMENT au niveau {lvl_chg}")
print("=" * 78)
print(f"  mesure de croissance          : {GROWTH_METHOD}")
print(f"  croissance médiane            : {med:+.2%} par an")
print(f"  sigma robuste (1,4826 × MAD)  : {sigma:.4f}")
print(f"  hausses signalées : {(c1['z_signal'] >= 2.5).sum()}")
print(f"  baisses signalées : {(c1['z_signal'] <= -2.5).sum()}")
print(f"  exclues du signalement (SoL initiale < p{CONFIG['CHANGE_MIN_BASE_PCT']} = {base_min:,.0f}) : "
      f"{(~c1['eligible']).sum()}")
print("\nLes 6 plus fortes hausses :")
print(c1.nlargest(6, "z_signal")[["name", "growth", "z_robust", "pop"]].to_string(index=False))
print("\nLes 6 plus fortes baisses :")
print(c1.nsmallest(6, "z_signal")[["name", "growth", "z_robust", "pop"]].to_string(index=False))

srt = c1.dropna(subset=["growth"]).sort_values("growth")
colors = np.where(~srt["eligible"], PAL["sage"],
          np.where(srt["z_signal"] >= 2.5, PAL["gold"],
          np.where(srt["z_signal"] <= -2.5, PAL["brick"], PAL["green"])))
fig = go.Figure()
fig.add_bar(x=srt["name"], y=srt["growth"] * 100, marker_color=colors,
            customdata=np.c_[srt["z_robust"], srt["pop"]],
            hovertemplate="<b>%{x}</b><br>%{y:+.1f} %/yr<br>z = %{customdata[0]:+.2f}"
                          "<br>pop %{customdata[1]:,.0f}<extra></extra>")
fig.add_hline(y=med * 100, line=dict(color=PAL["slate"], dash="dot"))
fig.add_hline(y=(med + 2.5 * sigma) * 100, line=dict(color=PAL["ochre"], dash="dash", width=1.4))
fig.add_hline(y=(med - 2.5 * sigma) * 100, line=dict(color=PAL["ochre"], dash="dash", width=1.4))
fig.update_layout(xaxis_title="", yaxis_title=f"SoL growth {Y0}–{Y1} · croissance (%/yr)",
                  xaxis=dict(tickangle=-60, tickfont=dict(size=9)))
show_fig(fig, "E_growth", height=470)

_ng, _nl = int((c1['z_signal'] >= 2.5).sum()), int((c1['z_signal'] <= -2.5).sum())
_nb = int((~c1["eligible"]).sum())
INSIGHTS["E"] = {
 "en": (f"Median {lvl_chg} growth in Sum of Lights is {med:+.1%} per year. "
        f"Growth is measured {'between the median of the first and last three years' if TREND is not None else 'as a two-date compound rate'}. "
        f"{_ng} unit(s) exceed the robust +2.5σ threshold and {_nl} fall below −2.5σ; these are "
        f"candidates for investigation, not confirmed findings — each should be checked against the "
        f"gas-flare list, the product version and any known local event. {_nb} unit(s) with a very "
        f"low initial base are shown but never flagged."),
 "fr": (f"La croissance médiane de la SoL au niveau {lvl_chg} est de {med:+.1%} par an. "
        f"Croissance mesurée par {GROWTH_METHOD}. "
        f"{_ng} unité(s) dépassent le seuil robuste de +2,5σ et {_nl} passent sous −2,5σ ; ce sont des "
        f"candidats à investiguer, pas des résultats confirmés — chacun doit être confronté à la liste des "
        f"torchères, à la version du produit et à tout événement local connu. {_nb} unité(s) à base "
        f"initiale très faible sont affichées mais jamais signalées."),
}

### 🅕 Scénario F · Suivi des chocs et des anomalies

**La question** — Une région a-t-elle connu une **anomalie mensuelle** — cyclone, effondrement du réseau
électrique, épisode de conflit, arrêt industriel brutal — qu'une comparaison de janvier à janvier ne révélerait
jamais ?

**La mesure** — Construire une série mensuelle de SoL par ADM1. Pour chaque mois, comparer l'observation au
**même mois calendaire des autres années**, et non au mois précédent. C'est la discipline qui sépare un choc réel
de l'artefact saisonnier identifié au jour 4 : couverture neigeuse, phénologie de la végétation, cycle lunaire,
Ramadan, Noël, éclairage des récoltes. On prend ensuite un z-score robuste du résidu.

**Contrôle nuageux.** Dans les composites mensuels, un pixel sans aucune observation sans nuage vaut zéro : un mois
de saison des pluies ressemblerait à une coupure géante. Le notebook masque ces pixels, mesure la part de la
région réellement observée, écarte les mois-région sous `MONTHLY_MIN_COVERAGE` (80 %) et corrige les mois
partiellement couverts.

> ⚠️ **L'erreur la plus fréquente** — comparer décembre à novembre et conclure que l'économie s'est contractée.
> Presque tous les pays du continent présentent une amplitude saisonnière de 10 à 30 % dans les lumières
> nocturnes, sans le moindre rapport avec l'activité économique.

In [ ]:
# --------------------------------------------------------------------------------------
# 🅕 · Série mensuelle et détection d'anomalies
# --------------------------------------------------------------------------------------
def monthly_series(cfg=CONFIG, level="ADM1"):
    '''DataFrame propre : name | date | sol. Acquisition propre à chaque fournisseur.'''
    gdf = ADMIN[level]
    years = cfg["MONTHLY_YEARS"]

    # ---- DÉMO : série paramétrique amorcée sur la SoL zonale observée ----------------
    if cfg["PROVIDER"] == "demo":
        base = Z[(level, Y1)].set_index("name")["sol"] / 12.0
        rng = np.random.default_rng(cfg["RANDOM_SEED"] + 5)
        shock_zone = base.index[int(rng.integers(0, len(base)))]
        shock_start = pd.Timestamp(f"{years[-1]}-03-01")
        rows = []
        rng_c = np.random.default_rng(cfg["RANDOM_SEED"] + 11)   # couverture nuageuse simulée
        for nm, b in base.items():
            amp   = rng.uniform(.07, .20)          # seasonal amplitude
            phase = rng.uniform(0, 2 * np.pi)
            trend = rng.uniform(.005, .035) / 12
            for i, dt_ in enumerate(pd.date_range(f"{years[0]}-01-01",
                                                  f"{years[-1]}-12-01", freq="MS")):
                v = b * (1 + trend) ** i
                v *= 1 + amp * np.sin(2 * np.pi * dt_.month / 12 + phase)
                v *= 1 + rng.normal(0, .06)
                if nm == shock_zone and shock_start <= dt_ < shock_start + pd.DateOffset(months=4):
                    v *= 0.55                       # injected shock
                cov = float(rng_c.uniform(.45, .95)) if rng_c.random() < .08 else 1.0
                # les pixels nuageux manquent : la somme observée est amputée d'autant
                rows.append({"name": nm, "date": dt_, "sol": max(v, 0) * cov, "coverage": cov})
        log(f"série mensuelle de démo · choc injecté dans « {shock_zone} » à partir de {shock_start:%Y-%m}", "warn")
        return pd.DataFrame(rows)

    # ---- EARTH ENGINE : reduceRegions côté serveur (rapide, sans téléchargement) -----
    if cfg["PROVIDER"] == "gee":
        ee = _ee_ready(cfg)
        fc = ee.FeatureCollection(json.loads(gdf[["shapeName", "geometry"]].to_json()))
        col = (ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
               .filterDate(f"{years[0]}-01-01", f"{years[-1]}-12-31")
               .select(["avg_rad", "cf_cvg"]))
        # Un pixel sans aucune observation sans nuage vaut 0 dans avg_rad : il ressemblerait
        # à une coupure de courant. On le masque, et on mesure la part de pixels couverts.
        def reduce_img(img):
            valid = img.select("cf_cvg").gt(0)
            sol = img.select("avg_rad").max(0).updateMask(valid).rename("sol")
            both = sol.addBands(valid.rename("cov"))
            red = both.reduceRegions(
                collection=fc, scale=500,
                reducer=ee.Reducer.sum().combine(ee.Reducer.mean(), sharedInputs=True))
            d = img.date().format("YYYY-MM-dd")
            return red.map(lambda f: f.set("date", d))
        flat = col.map(reduce_img).flatten()
        feats = flat.select(["shapeName", "date", "sol_sum", "cov_mean"], None, False) \
                    .getInfo()["features"]
        df = pd.DataFrame([f["properties"] for f in feats]).rename(
            columns={"shapeName": "name", "sol_sum": "sol", "cov_mean": "coverage"})
        df["date"] = pd.to_datetime(df["date"])
        return df.dropna(subset=["sol"])

    # ---- autres fournisseurs ---------------------------------------------------------
    log("Série mensuelle non implémentée pour ce fournisseur — scénario F ignoré.", "warn")
    return pd.DataFrame(columns=["name", "date", "sol"])

MS = monthly_series()
MONTHLY_QC = {"dropped": 0, "rescaled": 0, "min_coverage": CONFIG["MONTHLY_MIN_COVERAGE"]}
if len(MS) and "coverage" in MS:
    _low = MS["coverage"] < CONFIG["MONTHLY_MIN_COVERAGE"]
    MONTHLY_QC["dropped"] = int(_low.sum())
    MS = MS[~_low].copy()
    # mois partiellement couverts : SoL estimée = SoL observée ÷ part de pixels couverts
    _part = MS["coverage"] < 0.999
    MONTHLY_QC["rescaled"] = int(_part.sum())
    MS.loc[_part, "sol"] = MS.loc[_part, "sol"] / MS.loc[_part, "coverage"]
    print(f"Contrôle nuageux : {MONTHLY_QC['dropped']} mois-région écartés (couverture < "
          f"{CONFIG['MONTHLY_MIN_COVERAGE']:.0%}), {MONTHLY_QC['rescaled']} corrigés de leur "
          f"couverture partielle.")

if len(MS):
    MS["month"] = MS["date"].dt.month
    MS["year"] = MS["date"].dt.year
    MS = MS.sort_values(["name", "date"]).reset_index(drop=True)
    # Moyenne du mois calendaire hors observation : un point ne définit jamais sa propre référence.
    _g = MS.groupby(["name", "month"])["sol"]
    _sum, _cnt = _g.transform("sum"), _g.transform("count")
    MS["expected"] = np.where(_cnt > 1, (_sum - MS["sol"]) / (_cnt - 1), _sum / _cnt)
    MS["resid"] = MS["sol"] / MS["expected"].replace(0, np.nan) - 1
    _med = MS.groupby("name")["resid"].transform("median")
    _sig = 1.4826 * MS.assign(_ad=(MS["resid"] - _med).abs()) \
                      .groupby("name")["_ad"].transform("median")
    MS["z"] = (MS["resid"] - _med) / _sig.replace(0, np.nan)
    ANOM = MS[MS["z"].abs() >= 2.5].sort_values("z")
    RESULTS["F"] = dict(series=MS, anomalies=ANOM)

    print("SAISONNALITÉ ET ANOMALIES")
    print("=" * 78)
    nat = MS.groupby("date")["sol"].sum()
    seas = MS.groupby("month")["sol"].sum()
    print(f"  amplitude saisonnière nationale (max/min des totaux mensuels) : "
          f"{seas.max()/seas.min():.2f}×")
    print(f"  mois le plus lumineux : {seas.idxmax()}"
          f"   ·  dimmest / le moins lumineux : {seas.idxmin()}")
    print(f"  anomalies signalées (|z| ≥ 2,5) : {len(ANOM)}")
    if len(ANOM):
        print("\n  Anomalies les plus négatives :")
        print(ANOM.nsmallest(6, "z")[["name", "date", "sol", "expected", "resid", "z"]]
              .to_string(index=False))

    top5 = MS.groupby("name")["sol"].mean().nlargest(6).index
    fig = go.Figure()
    for i, nm in enumerate(top5):
        d = MS[MS["name"] == nm].sort_values("date")
        fig.add_scatter(x=d["date"], y=d["sol"], mode="lines", name=str(nm),
                        line=dict(width=2.2, color=SECTION_COLORS[i % len(SECTION_COLORS)]))
    if len(ANOM):
        aa = ANOM[ANOM["name"].isin(top5)]
        fig.add_scatter(x=aa["date"], y=aa["sol"], mode="markers", name="anomaly · anomalie",
                        marker=dict(size=11, color=PAL["brick"], symbol="x",
                                    line=dict(width=1, color="white")))
    fig.update_layout(xaxis_title="", yaxis_title="Monthly SoL · SoL mensuelle")
    show_fig(fig, "F_monthly", height=430)

    INSIGHTS["F"] = {
     "en": (f"Monthly Sum of Lights shows a seasonal amplitude of {seas.max()/seas.min():.2f}× between the "
            f"brightest and dimmest calendar month — a reminder that month-on-month comparisons are meaningless "
            f"without seasonal adjustment. {len(ANOM)} observation(s) deviate by more than 2.5 robust sigma from "
            f"their own month-of-year baseline."),
     "fr": (f"La SoL mensuelle présente une amplitude saisonnière de {seas.max()/seas.min():.2f}× entre le mois "
            f"calendaire le plus lumineux et le moins lumineux — rappel que les comparaisons de mois à mois n'ont "
            f"aucun sens sans correction saisonnière. {len(ANOM)} observation(s) s'écartent de plus de 2,5 sigma "
            f"robustes de leur propre référence mensuelle."),
    }
else:
    RESULTS["F"] = None
    INSIGHTS["F"] = {"en": "Monthly series unavailable for this provider.",
                     "fr": "Série mensuelle indisponible pour ce fournisseur."}
    print("Scénario F ignoré")

### 🅖 Scénario G · Validation — le scénario qui décide si vous avez le droit de publier

Tout ce qui précède est un calcul. Ceci est la seule partie qui constitue un *argument statistique*. L'agenda du
jour 4 pose l'exigence sans détour : *décider, preuves à l'appui, si l'indicateur est diffusable dans votre pays
ou s'il reste un simple outil de diagnostic*.

Le test est une corrélation entre l'indicateur NTL et une **statistique officielle indépendante, mesurée au même
niveau administratif** : population recensée, taux d'électrification issu de votre enquête ménages, PIB régional,
nombre d'entreprises, registres de raccordement au réseau. La population est utilisée par défaut parce qu'elle est
toujours disponible ; c'est aussi le test le *plus faible*, puisque la population détermine mécaniquement une part
de la lumière. **Apportez une seconde variable.**

**La règle de décision appliquée ici**

Seules les **statistiques indépendantes** que vous fournissez fondent le verdict. Les corrélations avec la
population sont affichées comme *contrôles de cohérence* : elles vérifient que la chaîne n'est pas cassée, elles
ne valident rien.

| Éléments de preuve | Verdict |
|---|---|
| ρ ≥ 0,85, borne basse de l'IC à 95 % ≥ 0,60, n ≥ `VALIDATION_MIN_N` unités | **Diffusable comme indicateur**, accompagné de la déclaration de limites |
| 0,60 ≤ ρ, ou ρ ≥ 0,85 avec trop peu d'unités ou un intervalle trop large | **Diffusable comme statistique expérimentale**, clairement étiquetée, aux côtés de la source officielle |
| ρ < 0,60 | **Diagnostic et usage interne uniquement** — ne pas diffuser comme statistique |
| Aucune statistique indépendante | Au mieux **expérimental**, quelle que soit la corrélation avec la population |

**Trois précautions intégrées au calcul.** Un *taux* officiel (électrification, pauvreté) est comparé à un taux —
lumière par habitant, part de la population éclairée — et jamais à un total. Un *total* officiel (PIB régional,
nombre d'entreprises) est comparé à la somme des lumières, mais on calcule aussi la **corrélation partielle, une
fois la population neutralisée** : deux totaux corrèlent toujours parce que les grandes unités sont grandes ; si
le lien disparaît sans l'effet de taille (ρ partiel < `VALIDATION_MIN_PARTIAL`), le verdict est plafonné à
« expérimental ». Enfin, la validation peut se faire au niveau **ADM2** (`OFFICIAL_STATS_LEVEL`) : un pays qui
n'a que 5 ou 14 régions ne peut jamais atteindre 20 unités en ADM1.

Le ρ de Spearman (corrélation de rangs) est retenu comme chiffre principal plutôt que le r de Pearson, parce
qu'une corrélation de rangs résiste à la log-normalité de la radiance et ne se laisse pas détourner par la seule
capitale.

In [ ]:
# --------------------------------------------------------------------------------------
# 🅖 · Validation contre les statistiques officielles
# --------------------------------------------------------------------------------------
from scipy import stats as sstats
import unicodedata, re as _re


def _norm_name(x):
    '''Clé d'appariement tolérante : sans accents, sans ponctuation, en minuscules.'''
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode()
    return _re.sub(r"[^a-z0-9]", "", x.lower())


def spearman_ci(rho, n, z=1.96):
    '''IC à 95 % du ρ de Spearman (transformation de Fisher, erreur type 1,03/√(n−3)).'''
    if n <= 4 or not np.isfinite(rho) or abs(rho) >= 1:
        return (np.nan, np.nan)
    f, se = np.arctanh(rho), 1.03 / np.sqrt(n - 3)
    return (float(np.tanh(f - z * se)), float(np.tanh(f + z * se)))


def validate(level="ADM1", cfg=CONFIG):
    d = Z[(level, Y1)].copy()
    tests = []

    def _test(label_en, label_fr, x, y, independent, note_en="", note_fr="",
              var_type="", size=None):
        m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        if size is not None:
            m &= np.isfinite(size) & (size > 0)
        if m.sum() < 5:
            return None
        rho, p_s = sstats.spearmanr(x[m], y[m])
        partial = np.nan
        if size is not None:
            # corrélation partielle de rangs, population neutralisée : deux totaux corrèlent
            # mécaniquement parce que les grandes unités sont grandes
            rxz = sstats.spearmanr(x[m], size[m])[0]; ryz = sstats.spearmanr(y[m], size[m])[0]
            den = np.sqrt(max((1 - rxz ** 2) * (1 - ryz ** 2), 1e-12))
            partial = float((rho - rxz * ryz) / den)
        r, p_p = sstats.pearsonr(np.log(x[m]), np.log(y[m]))
        el = np.polyfit(np.log(x[m]), np.log(y[m]), 1)[0]
        lo, hi = spearman_ci(float(rho), int(m.sum()))
        return dict(label_en=label_en, label_fr=label_fr, n=int(m.sum()), independent=independent,
                    var_type=var_type, partial_rho=partial,
                    spearman=float(rho), spearman_p=float(p_s), ci_low=lo, ci_high=hi,
                    pearson_log=float(r), r2_log=float(r ** 2),
                    elasticity=float(el), note_en=note_en, note_fr=note_fr)

    # ---- contrôles de cohérence : toujours disponibles, jamais suffisants ------------
    t = _test("Sum of Lights vs population", "Somme des lumières vs population",
              d["pop"].values, d["sol"].values, False,
              "Consistency check only: population mechanically drives light.",
              "Contrôle de cohérence seulement : la population détermine mécaniquement la lumière.")
    if t: tests.append(t)
    t = _test("Lit area vs population density", "Surface éclairée vs densité de population",
              d["pop_density"].values, d["lit_share"].values, False,
              "Consistency check only.", "Contrôle de cohérence seulement.")
    if t: tests.append(t)

    # ---- statistiques officielles indépendantes : les seules qui fondent le verdict --
    if cfg["OFFICIAL_STATS_CSV"]:
        off = pd.read_csv(cfg["OFFICIAL_STATS_CSV"])
        jc = cfg["OFFICIAL_JOIN_COL"] if cfg["OFFICIAL_JOIN_COL"] in off.columns else off.columns[0]
        keys = off[jc].astype(str)
        # jointure sur l'identifiant geoBoundaries si la colonne en contient, sinon sur le nom
        if keys.isin(d["shapeID"].astype(str)).mean() > 0.5:
            off["_key"], d["_key"], how = keys, d["shapeID"].astype(str), "shapeID"
        else:
            off["_key"], d["_key"], how = keys.map(_norm_name), d["name"].map(_norm_name), "nom"
        mg = d.merge(off, on="_key", how="inner")
        miss = sorted(set(d["_key"]) - set(off["_key"]))
        log(f"statistiques officielles appariées par {how} sur {len(mg)}/{len(d)} unités {level}",
            "ok" if len(mg) >= .9 * len(d) else "warn")
        if miss:
            print("  Unités sans correspondance dans le CSV :",
                  ", ".join(d.loc[d["_key"].isin(miss), "name"].astype(str).head(12)),
                  "…" if len(miss) > 12 else "")
        types = cfg.get("OFFICIAL_STATS_TYPES") or {}
        for c in off.select_dtypes("number").columns:
            v = mg[c].astype(float)
            typ = types.get(c) or ("rate" if (v.min() >= 0 and v.max() <= 100) else "total")
            if typ == "rate":
                # un taux se compare à un taux : jamais à un total qui dépend de la taille
                pairs = (("Light per capita", "Lumière par habitant", "sol_per_capita"),
                         ("Share of population in lit cells", "Part de population éclairée", "pop_lit_share"))
                for lab_en, lab_fr, col in pairs:
                    t = _test(f"{lab_en} vs {c}", f"{lab_fr} vs {c}", v.values, mg[col].values,
                              True, var_type="taux")
                    if t: tests.append(t)
            else:
                t = _test(f"Sum of Lights vs {c}", f"Somme des lumières vs {c}", v.values,
                          mg["sol"].values, True, var_type="total", size=mg["pop"].values.astype(float))
                if t: tests.append(t)
    else:
        log("Aucun OFFICIAL_STATS_CSV fourni — le verdict ne peut pas dépasser « expérimental ».", "warn")
    return pd.DataFrame(tests)


VAL_LEVEL = CONFIG["OFFICIAL_STATS_LEVEL"] if CONFIG["OFFICIAL_STATS_CSV"] else "ADM1"
if (VAL_LEVEL, Y1) not in Z:
    log(f"Niveau {VAL_LEVEL} indisponible — validation au niveau ADM1.", "warn")
    VAL_LEVEL = "ADM1"
VAL = validate(VAL_LEVEL)
MIN_N = int(CONFIG["VALIDATION_MIN_N"])

indep = VAL[VAL["independent"]] if len(VAL) else VAL
cons = VAL[~VAL["independent"]] if len(VAL) else VAL
has_independent = len(indep) > 0
basis = indep if has_independent else cons
if len(basis):
    bi = basis["spearman"].abs().idxmax()
    best, n_units = float(abs(basis.loc[bi, "spearman"])), int(basis.loc[bi, "n"])
    ci_low = float(np.sign(basis.loc[bi, "spearman"]) * basis.loc[bi, "ci_low"]) \
        if basis.loc[bi, "spearman"] > 0 else float(-basis.loc[bi, "ci_high"])
    best_label = basis.loc[bi, "label_fr"]
else:
    best, n_units, ci_low, best_label = np.nan, 0, np.nan, "—"

reasons = []
if not np.isfinite(best):
    verdict = "unknown"
    reasons.append("validation non calculable (moins de 5 unités exploitables)")
elif not has_independent:
    verdict = "experimental" if best >= 0.60 else "diagnostic"
    reasons.append("aucune statistique officielle indépendante : la corrélation avec la population "
                   "est un contrôle de cohérence, pas une validation")
elif (best >= 0.60 and basis.loc[bi, "var_type"] == "total"
      and np.isfinite(basis.loc[bi, "partial_rho"])
      and basis.loc[bi, "partial_rho"] < CONFIG["VALIDATION_MIN_PARTIAL"]):
    verdict = "experimental"
    reasons.append(f"la corrélation (ρ = {best:.2f}) s'explique surtout par la taille des unités : "
                   f"une fois la population neutralisée, ρ partiel = {basis.loc[bi, 'partial_rho']:.2f} "
                   f"(< {CONFIG['VALIDATION_MIN_PARTIAL']:.2f})")
elif best >= 0.85 and n_units >= MIN_N and np.isfinite(ci_low) and ci_low >= 0.60:
    verdict = "publishable"
    reasons.append(f"ρ = {best:.2f} (IC 95 % ≥ {ci_low:.2f}) sur {n_units} unités {VAL_LEVEL}, "
                   f"contre une statistique indépendante")
elif best >= 0.60:
    verdict = "experimental"
    if best >= 0.85 and n_units < MIN_N:
        reasons.append(f"corrélation forte mais seulement {n_units} unités {VAL_LEVEL} "
                       f"(minimum {MIN_N}) — fournir la statistique au niveau ADM2 si elle existe")
    elif best >= 0.85:
        reasons.append(f"corrélation forte mais intervalle de confiance trop large "
                       f"(borne basse {ci_low:.2f} < 0,60)")
    else:
        reasons.append(f"corrélation modérée : ρ = {best:.2f}")
else:
    verdict = "diagnostic"
    reasons.append(f"corrélation faible avec la statistique indépendante : ρ = {best:.2f}")

_TXT = {
    "publishable": ("Publishable as a subnational indicator, with the limitations statement attached.",
                    "Diffusable comme indicateur infranational, accompagné de la déclaration de limites."),
    "experimental": ("Publishable only as an EXPERIMENTAL statistic, clearly labelled as such and presented "
                     "alongside the official source it complements.",
                     "Diffusable uniquement comme statistique EXPÉRIMENTALE, explicitement étiquetée comme "
                     "telle et présentée à côté de la source officielle qu'elle complète."),
    "diagnostic": ("DIAGNOSTIC AND INTERNAL USE ONLY — do not disseminate as a statistic.",
                   "USAGE DIAGNOSTIQUE ET INTERNE UNIQUEMENT — ne pas diffuser comme statistique."),
    "unknown": ("Validation could not be computed.", "Validation non calculable."),
}
v_en, v_fr = _TXT[verdict]
v_fr = v_fr + " Motif : " + " ; ".join(reasons) + "."
v_en = v_en + (" No independent official statistic was supplied." if not has_independent else "")

RESULTS["G"] = dict(table=VAL, verdict=verdict, best_rho=best, ci_low=ci_low, n=n_units,
                    level=VAL_LEVEL, basis=best_label, reasons=reasons,
                    verdict_en=v_en, verdict_fr=v_fr, independent=has_independent)

print("VALIDATION")
print("=" * 78)
if len(VAL):
    _v = VAL.assign(type=np.where(VAL["independent"], "indépendante", "cohérence"))
    _cols = ["label_fr", "type", "n", "spearman", "ci_low", "ci_high"] + \
            (["partial_rho"] if "partial_rho" in _v and _v["partial_rho"].notna().any() else [])
    print(_v[_cols].round(3).to_string(index=False))
print(f"\n  Base du verdict : {best_label}")
print(f"  ρ = {best:.3f}   ·   borne basse IC 95 % = {ci_low:.3f}   ·   {n_units} unités {VAL_LEVEL}")
print(f"  Statistique officielle indépendante : {'OUI' if has_independent else 'NON'}")
print("\n" + "▂" * 78)
print(f"  VERDICT [{verdict.upper()}]")
print(f"  {v_fr}")
print("▂" * 78)

if len(VAL):
    fig = go.Figure()
    _lab = [f"{l} (cohérence)" if not ind else l for l, ind in zip(VAL["label_fr"], VAL["independent"])]
    _col = [PAL["sage"] if not ind else
            (PAL["green"] if v >= .85 else PAL["ochre"] if v >= .6 else PAL["brick"])
            for v, ind in zip(VAL["spearman"], VAL["independent"])]
    fig.add_bar(x=VAL["spearman"], y=_lab, orientation="h", marker_color=_col,
                error_x=dict(type="data", symmetric=False,
                             array=(VAL["ci_high"] - VAL["spearman"]).fillna(0),
                             arrayminus=(VAL["spearman"] - VAL["ci_low"]).fillna(0),
                             color=PAL["slate"], thickness=1.4, width=4),
                hovertemplate="<b>%{y}</b><br>ρ = %{x:.3f}<extra></extra>")
    for thr, c in ((0.60, PAL["ochre"]), (0.85, PAL["green"])):
        fig.add_vline(x=thr, line=dict(color=c, dash="dash", width=1.6))
    fig.update_layout(xaxis_title="Spearman ρ (IC 95 %)", yaxis_title="", xaxis=dict(range=[-0.05, 1]))
    show_fig(fig, "G_validation", height=340)

INSIGHTS["G"] = {"en": v_en, "fr": v_fr}

### 🅗 Contrôle transversal · les unités voisines se ressemblent-elles ?

Tous les intervalles de confiance ci-dessus supposent que chaque unité ADM2 apporte une information
indépendante. C'est rarement vrai : deux districts voisins partagent un réseau électrique, un marché, un climat,
et le halo lumineux de la même ville. Quand les voisins se ressemblent — ce qu'on appelle l'**autocorrélation
spatiale** — le nombre d'observations *réellement indépendantes* est plus petit que le nombre d'unités, et les
intervalles sont trop étroits.

Le test utilisé est l'**I de Moran** : proche de 0, pas de structure spatiale ; positif, les voisins se
ressemblent. Sa significativité est évaluée par 499 permutations aléatoires des valeurs entre unités. Quand il est
significatif, le notebook indique un **effectif équivalent** approximatif, `n × (1 − I) / (1 + I)`, à garder en tête
en lisant les intervalles.

In [ ]:
# --------------------------------------------------------------------------------------
# 🅗 · Autocorrélation spatiale (I de Moran, contiguïté, test par permutation)
# --------------------------------------------------------------------------------------
SPATIAL = {}

def _moran(values, W0, perms=499, seed=CONFIG["RANDOM_SEED"]):
    '''I de Moran sur les unités à valeur finie ; poids de contiguïté standardisés en ligne.'''
    x = np.asarray(values, dtype=float); ok = np.isfinite(x)
    W = W0[np.ix_(ok, ok)].astype(float); x = x[ok]; n = len(x)
    rs = W.sum(axis=1); keep = rs > 0
    W, x, rs = W[np.ix_(keep, keep)], x[keep], None
    rs = W.sum(axis=1); W = W / np.where(rs > 0, rs, 1)[:, None]
    n = len(x)
    if n < 10:
        return None
    z = x - x.mean(); den = float(z @ z)
    if den <= 0:
        return None
    I = float((n / W.sum()) * (z @ W @ z) / den)
    rng = np.random.default_rng(seed)
    sims = np.empty(perms)
    for k in range(perms):
        zp = rng.permutation(z)
        sims[k] = (n / W.sum()) * (zp @ W @ zp) / den
    p = float((1 + (sims >= I).sum()) / (perms + 1))
    n_eff = int(round(n * (1 - I) / (1 + I))) if I > 0 else n
    return {"I": I, "p": p, "n": n, "n_eff": max(n_eff, 2), "expected": -1 / (n - 1)}

if CONFIG["SPATIAL_CHECK"] and "ADM2" in ADMIN and len(ADMIN["ADM2"]) >= 10:
    g2 = ADMIN["ADM2"][["shapeID", "geometry"]].reset_index(drop=True)
    # contiguïté « reine », avec une marge de ~50 m pour absorber les micro-écarts des limites
    gb = g2.assign(geometry=g2.geometry.buffer(0.0005))
    _nb = gpd.sjoin(gb, gb, predicate="intersects", how="inner")
    _nb = _nb[_nb.index != _nb["index_right"]]
    W0 = np.zeros((len(g2), len(g2)), dtype=np.int8)
    W0[_nb.index.values, _nb["index_right"].values] = 1
    order = g2["shapeID"]

    tgt = {}
    if RESULTS.get("E") and RESULTS["E"]["level"] == "ADM2":
        tgt["Croissance de la SoL (scénario E)"] = \
            order.map(RESULTS["E"]["table"].set_index("shapeID")["growth"])
    _d2 = Z[("ADM2", Y1)].set_index("shapeID")
    _ok = (_d2["sol"] > 0) & (_d2["pop"] > 0)
    _res = pd.Series(np.nan, index=_d2.index)
    _res[_ok] = np.log(_d2.loc[_ok, "sol"]) - (RESULTS["A"]["elasticity"] * np.log(_d2.loc[_ok, "pop"])
                                              + float(np.mean(np.log(_d2.loc[_ok, "sol"]))
                                                      - RESULTS["A"]["elasticity"]
                                                      * np.mean(np.log(_d2.loc[_ok, "pop"]))))
    tgt["Résidus de l'élasticité (scénario A)"] = order.map(_res)
    tgt["Lumière par habitant (scénario D)"] = order.map(_d2["sol_per_capita"])

    print("AUTOCORRÉLATION SPATIALE — I de Moran")
    print("=" * 78)
    for lab, vals in tgt.items():
        r = _moran(vals.values, W0)
        if r is None:
            continue
        SPATIAL[lab] = r
        sig = r["p"] < 0.05
        print(f"  {lab:<40} I = {r['I']:+.3f}   p = {r['p']:.3f}   "
              f"{'→ n effectif ≈ ' + str(r['n_eff']) + ' sur ' + str(r['n']) if sig else '→ pas de structure significative'}")
    _e = SPATIAL.get("Croissance de la SoL (scénario E)")
    if _e and _e["p"] < 0.05 and "E" in INSIGHTS:
        INSIGHTS["E"]["fr"] += (f" Les croissances des districts voisins se ressemblent (I de Moran = {_e['I']:.2f}) : "
                                f"les hausses et baisses signalées forment souvent des grappes, à lire comme des "
                                f"phénomènes régionaux plutôt que locaux.")
        INSIGHTS["E"]["en"] += (f" Neighbouring districts grow alike (Moran's I = {_e['I']:.2f}): flagged changes "
                                f"often cluster and should be read as regional rather than local phenomena.")
else:
    print("Contrôle spatial non exécuté (désactivé, ou moins de 10 unités ADM2).")

---

<div style="background:linear-gradient(120deg,#00553A,#00704A 55%,#00A86A);border-radius:12px;
            padding:26px 32px;color:#fff;font-family:Calibri,sans-serif;">
  <div style="font-size:52px;font-weight:700;color:#F5C242;line-height:1;">08</div>
  <div style="font-size:11px;letter-spacing:3px;font-weight:700;color:#F5C242;margin-top:6px;">
    TABLEAU DE BORD</div>
  <div style="font-size:30px;font-weight:700;margin-top:10px;">Un fichier HTML, aucun serveur</div>
  <div style="font-size:16px;font-style:italic;color:#E6F6EE;margin-top:4px;">
    Le livrable que vos collègues ouvriront encore dans cinq ans</div>
</div>

Le tableau de bord est un **unique `index.html` autonome**. Ce choix est délibéré, et c'est celui qui a le plus de
chances de survivre à l'atelier : aucune étape de compilation, aucun framework, aucun serveur, aucune base de
données, aucune péremption. Il s'ouvre depuis une clé USB, un lecteur partagé, GitHub Pages ou une pièce jointe.
Dans cinq ans, il s'ouvrira encore.

**Langues du tableau de bord.** Le notebook est en français, mais la page produite peut être bilingue : chaque
chaîne est portée par des attributs `data-en` / `data-fr` et un bouton bascule l'ensemble, y compris les titres
d'axes Plotly via `Plotly.relayout`. Le réglage se fait dans `CONFIG["DASHBOARD_LANGS"]` :

- `["fr", "en"]` (par défaut) — page bilingue, bouton FR/EN, français affiché en premier ;
- `["fr"]` — page en français seul, sans bouton ;
- `["en"]` — page en anglais seul.

Le bilinguisme est géré **au niveau du contenu et non de la page** : un seul fichier, pas de répertoire `/fr/` à
maintenir en parallèle, donc pas de désynchronisation entre versions — le mode de défaillance le plus courant des
sites institutionnels bilingues.

In [ ]:
# --------------------------------------------------------------------------------------
# 8.1 · Table de libellés du tableau de bord — source unique de vérité
#        Chaque entrée est un couple (anglais, français) ; la page affiche la ou les
#        langues choisies dans CONFIG['DASHBOARD_LANGS'].
# --------------------------------------------------------------------------------------
CN = CONFIG["COUNTRY_NAME"]
I18N = {
 "title":        (f"{CN} — Night-Time Lights", f"{CN} — Lumières nocturnes"),
 "subtitle":     ("Subnational indicators derived from VIIRS night-time lights",
                  "Indicateurs infranationaux dérivés des lumières nocturnes VIIRS"),
 "period":       (f"Period {Y0}–{Y1}", f"Période {Y0}–{Y1}"),
 "nav_overview": ("Overview", "Vue d'ensemble"),
 "nav_a": ("Economic activity", "Activité économique"),
 "nav_b": ("Electrification", "Électrification"),
 "nav_c": ("Urbanisation", "Urbanisation"),
 "nav_d": ("Inequality", "Inégalités"),
 "nav_e": ("Change", "Changement"),
 "nav_f": ("Shocks", "Chocs"),
 "nav_g": ("Validation", "Validation"),
 "nav_m": ("Method & limits", "Méthode & limites"),
 "kpi_sol":   ("Sum of Lights", "Somme des lumières"),
 "kpi_growth":("Annual growth of SoL", "Croissance annuelle de la SoL"),
 "kpi_pop":   ("Population", "Population"),
 "kpi_dark":  ("People in unlit inhabited cells", "Personnes en cellules habitées non éclairées"),
 "kpi_lit":   ("Lit area", "Surface éclairée"),
 "kpi_gini":  ("Gini of light per capita", "Gini de la lumière par habitant"),
 "h_a": ("Where is measurable activity concentrated?", "Où se concentre l'activité mesurable ?"),
 "h_b": ("Who lives without detectable light?", "Qui vit sans lumière détectable ?"),
 "h_c": ("How did the lit footprint change?", "Comment l'empreinte éclairée a-t-elle évolué ?"),
 "h_d": ("Is light concentrating or spreading?", "La lumière se concentre-t-elle ou se diffuse-t-elle ?"),
 "h_e": ("Which districts changed significantly?", "Quels districts ont significativement changé ?"),
 "h_f": ("Any month-level anomaly?", "Des anomalies au niveau mensuel ?"),
 "h_g": ("Does the proxy survive official statistics?",
         "L'indicateur résiste-t-il aux statistiques officielles ?"),
 "c_share":   ("Share of national Sum of Lights vs share of population",
               "Part de la somme des lumières nationale vs part de la population"),
 "c_elast":   ("Light against population, ADM2 units, log–log",
               "Lumière contre population, unités ADM2, échelle log–log"),
 "c_access":  ("Population living in lit inhabited cells, by region",
               "Population vivant en cellules habitées éclairées, par région"),
 "c_trans":   ("Lit-footprint transition between the two dates",
               "Transition de l'empreinte éclairée entre les deux dates"),
 "c_lorenz":  ("Lorenz curve of light against population",
               "Courbe de Lorenz de la lumière contre la population"),
 "c_conv":    ("Growth against initial level (β-convergence)",
               "Croissance contre niveau initial (β-convergence)"),
 "c_series":  ("National Sum of Lights, every year", "Somme des lumières nationale, année par année"),
 "c_growth":  ("Annual growth of Sum of Lights by district",
               "Croissance annuelle de la somme des lumières par district"),
 "c_monthly": ("Monthly Sum of Lights, largest regions",
               "Somme des lumières mensuelle, principales régions"),
 "c_valid":   ("Rank correlation with independent statistics",
               "Corrélation de rangs avec des statistiques indépendantes"),
 "c_map":     ("Regional map · click a region for details",
               "Carte régionale · cliquez sur une région pour le détail"),
 "map_metric":("Sum of Lights per capita", "Somme des lumières par habitant"),
 "tbl_region":("Region", "Région"), "tbl_pop": ("Population", "Population"),
 "tbl_sol":   ("Sum of Lights", "Somme des lumières"),
 "tbl_lit":   ("Lit area (%)", "Surface éclairée (%)"),
 "tbl_access":("In lit cells (%)", "En cellules éclairées (%)"),
 "tbl_dark":  ("Unlit population", "Population non éclairée"),
 "tbl_growth":("Growth (%/yr)", "Croissance (%/an)"),
 "verdict":   ("Dissemination verdict", "Verdict de diffusion"),
 "limits_h":  ("Limitations — read before citing", "Limites — à lire avant citation"),
 "method_h":  ("Method in one paragraph", "La méthode en un paragraphe"),
 "sources_h": ("Sources and licences", "Sources et licences"),
 "download":  ("Download the data (CSV)", "Télécharger les données (CSV)"),
 "repro":     ("Reproduce this analysis", "Reproduire cette analyse"),
 "demo_warn": ("⚠ SIMULATED DATA — demonstration only, not official statistics",
               "⚠ DONNÉES SIMULÉES — démonstration seulement, pas des statistiques officielles"),
}

# Titres d'axes permutés à l'exécution par Plotly.relayout
AXIS_I18N = {
 "A_share":      {"xaxis.title.text": ("Share of national total (%)", "Part du total national (%)")},
 "A_elasticity": {"xaxis.title.text": ("Population (log)", "Population (log)"),
                  "yaxis.title.text": ("Sum of Lights (log)", "Somme des lumières (log)")},
 "B_access":     {"xaxis.title.text": ("Population in lit inhabited cells (%)",
                                       "Population en cellules habitées éclairées (%)")},
 "D_lorenz":     {"xaxis.title.text": ("Cumulative population (%)", "Population cumulée (%)"),
                  "yaxis.title.text": ("Cumulative light (%)", "Lumière cumulée (%)")},
 "D_convergence":{"xaxis.title.text": (f"Light per capita in {Y0} (log)",
                                       f"Lumière par habitant en {Y0} (log)"),
                  "yaxis.title.text": ("Annual growth (%)", "Croissance annuelle (%)")},
 "E_series":     {"yaxis.title.text": ("Sum of Lights", "Somme des lumières")},
 "E_growth":     {"yaxis.title.text": (f"SoL growth {Y0}–{Y1} (%/yr)",
                                       f"Croissance SoL {Y0}–{Y1} (%/an)")},
 "F_monthly":    {"yaxis.title.text": ("Monthly Sum of Lights", "Somme des lumières mensuelle")},
 "G_validation": {"xaxis.title.text": ("Spearman ρ (95 % CI)", "ρ de Spearman (IC 95 %)")},
}
print(f"{len(I18N)} libellés · {len(AXIS_I18N)} figures à axes permutables")

In [ ]:
# --------------------------------------------------------------------------------------
# 8.2 · L'explorateur cartographique — données embarquées pour la carte interactive
# --------------------------------------------------------------------------------------
# La carte du tableau de bord est un composant autonome : les limites (simplifiées) et
# tous les indicateurs, pour chaque niveau administratif et chaque année, sont embarqués
# dans la page. Le navigateur recalcule la carte instantanément quand on change de
# niveau, d'indicateur ou d'année — sans serveur, sans clé d'API.
import shapely


def simplified_geojson(gdf, keep_cols=("shapeID", "shapeName"), max_kb=420):
    '''Simplifie progressivement, puis arrondit les coordonnées à ~1 m (1e-5 degré).'''
    g = gdf[list(keep_cols) + ["geometry"]].copy()
    span = max(g.total_bounds[2] - g.total_bounds[0], g.total_bounds[3] - g.total_bounds[1])
    tol = span / 3000
    for _ in range(10):
        gs = g.copy()
        gs["geometry"] = shapely.set_precision(
            g.geometry.simplify(tol, preserve_topology=True).values, 1e-5)
        gs = gs[~gs.geometry.is_empty]
        js = json.loads(gs.to_json(drop_id=True))
        size = len(json.dumps(js, separators=(",", ":"))) / 1024
        if size <= max_kb:
            log(f"GeoJSON simplifié à {size:.0f} ko (tol={tol:.5f}°)", "ok")
            return js, size
        tol *= 1.8
    log(f"GeoJSON encore à {size:.0f} ko après simplification", "warn")
    return js, size


def _r(v):
    '''Arrondi à 4 chiffres significatifs ; NaN -> None (JSON null).'''
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(v) else float(f"{v:.4g}")


# ---- catalogue des indicateurs cartographiables -------------------------------------------
# kind : "year" (une valeur par année) ou "change" (une valeur pour la période)
# scale : night (lumière), green (favorable), brick (défavorable), teal, div (écart à mid)
MAP_METRICS = [
    dict(key="sol_per_capita", kind="year", fmt="dec3", scale="night", group="light",
         en="Light per capita", fr="Lumière par habitant",
         den="Sum of Lights divided by population — the closest proxy for activity per person.",
         dfr="Somme des lumières rapportée à la population : l'indicateur le plus proche d'une activité par personne."),
    dict(key="sol_per_km2", kind="year", fmt="dec2", scale="night", group="light",
         en="Light intensity (SoL per km²)", fr="Intensité lumineuse (SoL par km²)",
         den="Sum of Lights per km² of territory: where light is dense.",
         dfr="Somme des lumières par km² de territoire : là où la lumière est dense."),
    dict(key="lit_share", kind="year", fmt="pct", scale="night", group="light",
         en="Share of territory lit", fr="Part du territoire éclairée",
         den="Share of the unit's area above the noise floor.",
         dfr="Part de la surface de l'unité au-dessus du plancher de bruit."),
    dict(key="pop_lit_share", kind="year", fmt="pct", scale="green", group="access",
         en="Population in lit cells", fr="Population en cellules éclairées",
         den="Share of inhabitants living in cells that emit detectable light (apparent access).",
         dfr="Part des habitants vivant dans des cellules émettant une lumière détectable (accès apparent)."),
    dict(key="pop_dark", kind="year", fmt="int", scale="brick", group="access", ext=True,
         en="People without detectable light", fr="Personnes sans lumière détectable",
         den="Inhabitants of inhabited cells with no detectable night-time light.",
         dfr="Habitants de cellules habitées sans lumière nocturne détectable."),
    dict(key="light_pop_ratio", kind="year", fmt="ratio", scale="div", mid=1.0, group="light",
         en="Light-to-population ratio", fr="Ratio lumière / population",
         den="Share of national light ÷ share of national population. Above 1: brighter than its weight.",
         dfr="Part de la lumière nationale ÷ part de la population nationale. Au-dessus de 1 : plus lumineuse que son poids."),
    dict(key="pop_density", kind="year", fmt="dec1", scale="teal", group="people",
         en="Population density (per km²)", fr="Densité de population (hab./km²)",
         den="Inhabitants per km² (WorldPop).", dfr="Habitants par km² (WorldPop)."),
    dict(key="growth", kind="change", fmt="pctyr", scale="div", mid=0.0, group="change",
         en="Annual growth of light", fr="Croissance annuelle de la lumière",
         den="Annual growth of the Sum of Lights over the period (scenario E method).",
         dfr="Croissance annuelle de la somme des lumières sur la période (méthode du scénario E)."),
    dict(key="d_pop_lit", kind="change", fmt="pts", scale="div", mid=0.0, group="change",
         en="Change in apparent access", fr="Évolution de l'accès apparent",
         den="Change in the share of population living in lit cells, in percentage points.",
         dfr="Variation de la part de la population en cellules éclairées, en points de pourcentage."),
    dict(key="z_change", kind="change", fmt="z", scale="div", mid=0.0, group="change",
         en="Change signal (robust z-score)", fr="Signal de changement (z-score robuste)",
         den="How unusual the unit's growth is. Beyond ±2.5: flagged by scenario E.",
         dfr="À quel point la croissance de l'unité est inhabituelle. Au-delà de ±2,5 : signalée par le scénario E."),
]
MAP_GROUPS = {"light": ("Light", "Lumière"), "access": ("Access", "Accès"),
              "people": ("Population", "Population"), "change": (f"Change {Y0}–{Y1}", f"Évolution {Y0}–{Y1}")}

MAP_LEVELS = [l for l in ("ADM1", "ADM2") if l in ADMIN and (l, Y1) in Z]
_LEVEL_LABEL = {"ADM1": ("Regions · ADM1", "Régions · ADM1"), "ADM2": ("Districts · ADM2", "Districts · ADM2")}
_E = RESULTS.get("E") or {}
MAPDATA = {"years": [int(Y0), int(Y1)], "levels": {}, "national": {}, "iso3": CONFIG["ISO3"],
           "country": CONFIG["COUNTRY_NAME"], "groups": MAP_GROUPS,
           "metrics": [{k: v for k, v in m.items()} for m in MAP_METRICS],
           "bounds": [float(b) for b in ADMIN[MAP_LEVELS[0]].total_bounds], "demo": bool(IS_DEMO)}

for lvl in MAP_LEVELS:
    g = ADMIN[lvl].reset_index(drop=True)
    gj, kb = simplified_geojson(g, max_kb=300 if lvl == "ADM1" else 900)
    ids = [f["properties"]["shapeID"] for f in gj["features"]]
    gi = g.set_index("shapeID")
    rp = gi.geometry.representative_point()
    t0, t1 = Z[(lvl, Y0)].set_index("shapeID"), Z[(lvl, Y1)].set_index("shapeID")
    vals = {}
    for m in MAP_METRICS:
        k = m["key"]
        if m["kind"] == "year":
            if k not in t1:
                continue
            vals[k] = {str(Y0): [_r(t0[k].get(i)) for i in ids], str(Y1): [_r(t1[k].get(i)) for i in ids]}
        elif k == "growth":
            if _E and _E.get("level") == lvl:
                s_ = _E["table"].set_index("shapeID")["growth"]
            else:
                s_ = pd.Series(np.asarray(cagr(t0["sol"].reindex(t1.index), t1["sol"], DT), dtype=float),
                               index=t1.index)
            vals[k] = {"chg": [_r(s_.get(i)) for i in ids]}
        elif k == "d_pop_lit":
            vals[k] = {"chg": [_r((t1["pop_lit_share"].get(i, np.nan) - t0["pop_lit_share"].get(i, np.nan)) * 100)
                               for i in ids]}
        elif k == "z_change" and _E and _E.get("level") == lvl:
            s_ = _E["table"].set_index("shapeID")["z_signal"]
            vals[k] = {"chg": [_r(s_.get(i)) for i in ids]}
    names = [str(gi.loc[i, "shapeName"]) for i in ids]
    parents = ([str(t1["parent"].get(i, "")) if "parent" in t1 else "" for i in ids]
               if lvl != "ADM1" else [CONFIG["COUNTRY_NAME"]] * len(ids))
    MAPDATA["levels"][lvl] = {
        "label": _LEVEL_LABEL[lvl], "geojson": gj, "ids": ids, "names": names, "parents": parents,
        "pop": [_r(t1["pop"].get(i)) for i in ids],
        "center": [[round(float(rp[i].x), 4), round(float(rp[i].y), 4)] for i in ids],
        "bbox": [[round(float(v), 4) for v in gi.loc[i, "geometry"].bounds] for i in ids],
        "values": vals,
    }

# repères nationaux, pour situer chaque unité
if ("ADM0", Y1) in Z:
    n0, n1 = Z[("ADM0", Y0)].iloc[0], Z[("ADM0", Y1)].iloc[0]
    for m in MAP_METRICS:
        k = m["key"]
        if m["kind"] == "year" and k in n1:
            MAPDATA["national"][k] = {str(Y0): _r(n0[k]), str(Y1): _r(n1[k])}
    MAPDATA["national"]["growth"] = {"chg": _r(RESULTS["A"]["national"]["sol_cagr"])}
    MAPDATA["national"]["d_pop_lit"] = {"chg": _r((n1["pop_lit_share"] - n0["pop_lit_share"]) * 100)}
_prod_short = " / ".join(sorted(set(NTL_PRODUCT.values())))
MAPDATA["foot"] = [
    f"Sources: {_prod_short} · geoBoundaries · WorldPop — " + ("top-coded values" if CONFIG["USE_TOPCODED"] else "raw values")
    + (" · SIMULATED DATA" if IS_DEMO else ""),
    f"Sources : {_prod_short} · geoBoundaries · WorldPop — " + ("valeurs écrêtées" if CONFIG["USE_TOPCODED"] else "valeurs brutes")
    + (" · DONNÉES SIMULÉES" if IS_DEMO else "")]
if "ADM0" in ADMIN:
    MAPDATA["outline"], _ = simplified_geojson(ADMIN["ADM0"], max_kb=150)
MAPDATA_JSON = json.dumps(MAPDATA, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
print(f"Explorateur cartographique : niveaux {', '.join(MAP_LEVELS)} · "
      f"{sum(len(v['values']) for v in MAPDATA['levels'].values())} couches · "
      f"{len(MAPDATA_JSON)/1024:,.0f} ko embarqués")

# aperçu dans le notebook : la couche par défaut, sur fond sombre
_lv = MAP_LEVELS[-1]; _L = MAPDATA["levels"][_lv]
_zv = _L["values"]["sol_per_capita"][str(Y1)]
_b = MAPDATA["bounds"]
mapfig = go.Figure(go.Choroplethmap(
    geojson=_L["geojson"], locations=_L["ids"], z=_zv, featureidkey="properties.shapeID",
    colorscale=[[0, "#1d2b4f"], [.35, "#2a6f97"], [.6, "#3fb6a8"], [.85, "#f5c242"], [1, "#fff4c2"]],
    zmin=float(np.nanpercentile([v for v in _zv if v is not None], 2)),
    zmax=float(np.nanpercentile([v for v in _zv if v is not None], 98)),
    marker_opacity=.85, marker_line_width=.5, marker_line_color="rgba(255,255,255,.5)",
    text=_L["names"], hovertemplate="<b>%{text}</b><br>%{z:.3f}<extra></extra>",
    colorbar=dict(thickness=10, len=.6, outlinewidth=0)))
mapfig.update_layout(map=dict(style="carto-darkmatter",
                              center=dict(lon=(_b[0] + _b[2]) / 2, lat=(_b[1] + _b[3]) / 2),
                              zoom=max(1.0, 8.4 - np.log2(max(_b[2] - _b[0], _b[3] - _b[1]) * 5.2))),
                     margin=dict(l=0, r=0, t=0, b=0))
show_fig(mapfig, "MAP", height=520)

In [ ]:
# --------------------------------------------------------------------------------------
# 8.3 · Exports de données — les tables derrière chaque graphique
# --------------------------------------------------------------------------------------
DATA_DIR = OUT / "site" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
exported = []

for (lvl, y), df in Z.items():
    f = DATA_DIR / f"ntl_{CONFIG['ISO3']}_{lvl}_{y}.csv"
    df.to_csv(f, index=False, float_format="%.6g")
    exported.append(f)
    if CONFIG["USE_TOPCODED"]:
        fr = DATA_DIR / f"ntl_{CONFIG['ISO3']}_{lvl}_{y}_brut.csv"
        ZS[(lvl, y)].to_csv(fr, index=False, float_format="%.6g")
        exported.append(fr)

if RESULTS.get("E") is not None:
    RESULTS["E"]["table"].to_csv(DATA_DIR / f"change_{CONFIG['ISO3']}_{Y0}_{Y1}.csv",
                                 index=False, float_format="%.6g")
    exported.append(DATA_DIR / f"change_{CONFIG['ISO3']}_{Y0}_{Y1}.csv")
if RESULTS.get("F"):
    RESULTS["F"]["series"].to_csv(DATA_DIR / f"monthly_{CONFIG['ISO3']}.csv",
                                  index=False, float_format="%.6g")
    exported.append(DATA_DIR / f"monthly_{CONFIG['ISO3']}.csv")
if len(RESULTS["G"]["table"]):
    RESULTS["G"]["table"].to_csv(DATA_DIR / "validation.csv", index=False, float_format="%.6g")
    exported.append(DATA_DIR / "validation.csv")

for _lv, _L in MAPDATA["levels"].items():
    _f = DATA_DIR / f"{_lv.lower()}_{CONFIG['ISO3']}.geojson"
    with open(_f, "w", encoding="utf-8") as f:
        json.dump(_L["geojson"], f)
    exported.append(_f)

# ---- manifeste d'exécution : la trace de reproductibilité ---------------------------
MANIFEST = {
    "run_id": RUN_ID,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "country": {"iso3": CONFIG["ISO3"], "name": CONFIG["COUNTRY_NAME"], "bbox": list(COUNTRY_BBOX)},
    "provider": CONFIG["PROVIDER"],
    "simulated_data": IS_DEMO,
    "years": {"baseline": Y0, "endline": Y1, "monthly": CONFIG["MONTHLY_YEARS"]},
    "admin_levels": {k: int(len(v)) for k, v in ADMIN.items()},
    "parameters": {k: CONFIG[k] for k in ("NOISE_FLOOR", "URBAN_CORE", "TOPCODE_PCT",
                                          "POP_LIT_MIN", "TARGET_SCALE_M", "USE_TOPCODED",
                                          "MASK_FLARES", "FLARE_BUFFER_PX", "NOISE_SENSITIVITY",
                                          "CHANGE_MIN_BASE_PCT", "MONTHLY_MIN_COVERAGE",
                                          "ALLOW_MIXED_PRODUCTS", "OFFICIAL_STATS_LEVEL",
                                          "VALIDATION_MIN_N")},
    "ntl_products": {str(y): v for y, v in NTL_PRODUCT.items()},
    "ntl_product_families": {str(y): v for y, v in NTL_FAMILY.items()},
    "population_years": {str(y): v for y, v in POP_META.items()},
    "growth_method": RESULTS["E"].get("method"),
    "trend_years": RESULTS["E"].get("years"),
    "spatial_autocorrelation": SPATIAL,
    "values_used": Z_LABEL,
    "topcode_value": TOPCODE,
    "flare_masking": FLARE_STATS,
    "monthly_quality": MONTHLY_QC,
    "electrification_range": RESULTS["B"].get("range"),
    "artefacts": {str(y): {k: v for k, v in AQ[y].items()
                           if k != "flare_candidates" and not k.startswith("_")}
                  for y in CONFIG["YEARS"]},
    "flare_candidates": {str(y): AQ[y]["flare_candidates"] for y in CONFIG["YEARS"]},
    "verdict": RESULTS["G"]["verdict"],
    "verdict_basis": RESULTS["G"]["basis"],
    "verdict_reasons": RESULTS["G"]["reasons"],
    "validation_level": RESULTS["G"]["level"],
    "best_spearman": RESULTS["G"]["best_rho"],
    "spearman_ci_low": RESULTS["G"]["ci_low"],
    "software": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "geopandas": gpd.__version__,
                 "rasterio": rasterio.__version__},
    "licences": {"boundaries": "geoBoundaries gbOpen, CC BY 4.0",
                 "population": "WorldPop, CC BY 4.0",
                 "ntl": "VIIRS / NASA Black Marble — US Government work, attribution requested",
                 "code": CONFIG["LICENSE_CODE"], "derived_data": CONFIG["LICENSE_DATA"]},
}
with open(OUT / "site" / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2, ensure_ascii=False, default=str)

print(f"{len(exported)} fichiers de données exportés vers {DATA_DIR}")
for f in exported[:8]:
    print(f"   {f.name:<44} {f.stat().st_size/1024:>8.1f} kB")

In [ ]:
# --------------------------------------------------------------------------------------
# 8.4 · La déclaration de limites — produite par cette exécution, pas par un modèle
# --------------------------------------------------------------------------------------
G = RESULTS["G"]
flares = sum(len(AQ[y]["flare_candidates"]) for y in CONFIG["YEARS"])
lost_km2 = float(RESULTS["C"]["transition"].query("`class`=='lost_light'")["area_km2"].iloc[0])
noise_share = AQ[Y1]["share_below_noise_floor"]
top1 = AQ[Y1]["sol_share_top1pct_pixels"]

# ---- éléments des garde-fous méthodologiques, écrits d'après cette exécution -----------
_prod = sorted(set(NTL_FAMILY.values()))
_vers = sorted(set(NTL_PRODUCT.values()))
_vtxt = (" (versions : " + " / ".join(_vers) + ")") if len(_vers) > 1 else ""
_prod_fr = ("un seul produit pour toutes les années : " + _prod[0] + _vtxt) if len(_prod) == 1 else \
           ("PRODUITS DIFFÉRENTS selon les années (" + " / ".join(_prod) + ") — comparaisons à interpréter comme "
            "mêlant rupture de produit et changement réel")
_prod_en = ("a single product for all years: " + _prod[0] + _vtxt.replace("versions :", "versions:")) if len(_prod) == 1 else \
           ("DIFFERENT PRODUCTS across years (" + " / ".join(_prod) + ") — comparisons mix a product break with "
            "real change")
if FLARE_STATS["applied"]:
    _fl = "; ".join(f"{y} : {r['share_of_national']:.1%}" for y, r in FLARE_STATS["sol_removed"].items())
    _flare_fr = (f"{FLARE_STATS['masked_pixels']:,} pixels de torchères (halo compris) neutralisés avant calcul ; "
                 f"part de la SoL nationale retirée — {_fl}.")
    _flare_en = (f"{FLARE_STATS['masked_pixels']:,} gas-flare pixels (with halo) neutralised before any computation; "
                 f"share of national SoL removed — {_fl.replace(' : ', ': ')}.")
elif flares:
    _flare_fr = "torchères détectées mais NON neutralisées (MASK_FLARES = False) : elles pèsent dans les agrégats."
    _flare_en = "gas flares detected but NOT neutralised (MASK_FLARES = False): they weigh on aggregates."
else:
    _flare_fr = _flare_en = "—"
_rng_b = RESULTS["B"].get("range")
_sens_fr = (f"population non éclairée comprise entre {_rng_b[0]:,.0f} et {_rng_b[1]:,.0f} selon le plancher "
            f"de bruit ({min(CONFIG['NOISE_SENSITIVITY'])} à {max(CONFIG['NOISE_SENSITIVITY'])}).") if _rng_b else "non calculée."
_sens_en = (f"unlit inhabited population between {_rng_b[0]:,.0f} and {_rng_b[1]:,.0f} depending on the noise "
            f"floor ({min(CONFIG['NOISE_SENSITIVITY'])}–{max(CONFIG['NOISE_SENSITIVITY'])}).") if _rng_b else "not computed."
_mon_fr = (f"{MONTHLY_QC['dropped']} mois-région écartés pour couverture sans nuage inférieure à "
           f"{CONFIG['MONTHLY_MIN_COVERAGE']:.0%}, {MONTHLY_QC['rescaled']} corrigés de leur couverture partielle.")
_mon_en = (f"{MONTHLY_QC['dropped']} region-months dropped for cloud-free coverage below "
           f"{CONFIG['MONTHLY_MIN_COVERAGE']:.0%}, {MONTHLY_QC['rescaled']} rescaled for partial coverage.")
_ci = f"{G['ci_low']:.2f}" if np.isfinite(G.get("ci_low", np.nan)) else "—"
_pop_fr = "; ".join(f"{y} ← population {v}" for y, v in POP_META.items())
_pop_en = "; ".join(f"{y} ← population {v}" for y, v in POP_META.items())
_sig = {k: v for k, v in SPATIAL.items() if v["p"] < 0.05}
_sp_fr = ("significative pour " + " ; ".join(f"{k} (I = {v['I']:.2f}, n effectif ≈ {v['n_eff']} sur {v['n']})"
          for k, v in _sig.items()) + " — les intervalles de confiance correspondants sont trop étroits.") \
    if _sig else ("non significative." if SPATIAL else "non évaluée.")
_sp_en = ("significant for " + "; ".join(f"{k} (I = {v['I']:.2f}, effective n ≈ {v['n_eff']} of {v['n']})"
          for k, v in _sig.items()) + " — the corresponding confidence intervals are too narrow.") \
    if _sig else ("not significant." if SPATIAL else "not assessed.")

L_EN = f'''# Limitations statement — {CONFIG['COUNTRY_NAME']} ({CONFIG['ISO3']})

Run `{RUN_ID}` · provider `{CONFIG['PROVIDER']}` · baseline {Y0} · endline {Y1}

## 1. What is measured
Upward radiance in the visible and near-infrared, captured at approximately 01:30 local solar time, in
nW·cm⁻²·sr⁻¹, aggregated to administrative units. **Nothing in this product is a direct measurement of
electricity access, economic output or population.** Every such reading is an inference.

## 2. Dissemination verdict
**{G['verdict'].upper()}** — {G['verdict_en']}
Basis: {G['basis']} · ρ = {G['best_rho']:.3f} (lower 95% bound {_ci}) on {G['n']} {G['level']} units.
Independent official statistic supplied: {'yes' if G['independent'] else 'NO — the population check is a consistency test, not a validation'}.

## 2b. Methodological safeguards applied in this run
- NTL product: {_prod_en}
- Indicators computed on {'top-coded' if CONFIG['USE_TOPCODED'] else 'raw, non top-coded'} values (p{CONFIG['TOPCODE_PCT']}).
- Gas flares: {_flare_en}
- Electrification uncertainty: {_sens_en}
- Change detection: units below the p{CONFIG['CHANGE_MIN_BASE_PCT']} of initial SoL are shown but never flagged ({RESULTS['E'].get('n_low_base', 0)} unit(s)).
- Monthly series: {_mon_en}
- Population: {_pop_en}
- Growth (scenario E): {'between the median of the first and last three years; Theil-Sen trend reported alongside' if RESULTS['E'].get('years') else 'two-date compound rate'}.
- Spatial autocorrelation: {_sp_en}

## 3. Thresholds and their sensitivity
- Noise floor: **{CONFIG['NOISE_FLOOR']} nW·cm⁻²·sr⁻¹**. {noise_share:.1%} of national pixels fall below it in {Y1}.
  Every "unlit" figure in this product depends on this single number; a sensitivity test across
  0.15–0.50 is required before any figure is cited as a level rather than a rank.
- Urban-core threshold: **{CONFIG['URBAN_CORE']} nW·cm⁻²·sr⁻¹**. Cross-country comparison is valid only at an
  identical threshold.
- Top-coding at p{CONFIG['TOPCODE_PCT']} = {TOPCODE:,.1f}. The brightest 1 % of pixels hold {top1:.1%} of the
  national Sum of Lights, so national aggregates are sensitive to a small number of locations.

## 4. Known artefacts in this country
- **Gas-flare candidates detected: {flares}.** {'Each must be verified against a petroleum infrastructure map before any growth in its district is interpreted economically.' if flares else 'None detected at the thresholds used; this does not prove absence.'}
- **Blooming.** Lit-area figures overstate the true built footprint of cities; small ADM2 units adjacent to a
  large city inherit light that is not theirs.
- **Lost light: {lost_km2:,.0f} km².** May reflect depopulation, outage or conflict — or a change in product
  version or cloud-free observation count. Not interpretable without that check.
- **Seasonality.** {'Month-of-year amplitude observed: ' + f"{RESULTS['F']['series'].groupby('month')['sol'].sum().max() / RESULTS['F']['series'].groupby('month')['sol'].sum().min():.2f}×." if RESULTS.get('F') else 'Not assessed in this run.'} Month-on-month comparisons are invalid without seasonal adjustment.
- **Sensor era.** VIIRS only. Any comparison with DMSP-OLS years (1992–2013) would be a comparison between two
  different physical quantities and is not made here.

## 5. What this product must not be used for
- Producing a level of GDP, of household income, or of the electrification rate.
- Ranking administrative units for budget allocation without a second, independent indicator.
- Any statement about activity that does not emit light: subsistence agriculture, daytime informal trade,
  indoor activity in well-insulated buildings, the care economy.
- Monitoring at a finer level than the units published here.

## 6. Boundaries and population
Administrative boundaries: {CONFIG['BOUNDARY_SOURCE']}. If these differ from the official national boundaries,
the zonal figures differ accordingly. Population: WorldPop modelled surfaces, themselves an estimate with their
own error structure, which propagates into every per-capita and every "unlit population" figure here.

{'## 7. SIMULATED DATA' + chr(10) + 'This run used the built-in synthetic country generator. All figures are fictional and exist only to demonstrate the method. They must never be published, cited or presented as statistics for ' + CONFIG['COUNTRY_NAME'] + '.' if IS_DEMO else ''}
'''

L_FR = f'''# Déclaration de limites — {CONFIG['COUNTRY_NAME']} ({CONFIG['ISO3']})

Exécution `{RUN_ID}` · fournisseur `{CONFIG['PROVIDER']}` · référence {Y0} · fin de période {Y1}

## 1. Ce qui est mesuré
Une radiance ascendante dans le visible et le proche infrarouge, captée vers 01h30 heure solaire locale, en
nW·cm⁻²·sr⁻¹, agrégée par unité administrative. **Rien dans ce produit n'est une mesure directe de l'accès à
l'électricité, de la production économique ou de la population.** Chacune de ces lectures est une inférence.

## 2. Verdict de diffusion
**{G['verdict'].upper()}** — {G['verdict_fr']}
Base : {G['basis']} · ρ = {G['best_rho']:.3f} (borne basse IC 95 % : {_ci}) sur {G['n']} unités {G['level']}.
Statistique officielle indépendante fournie : {'oui' if G['independent'] else 'NON — la corrélation avec la population est un contrôle de cohérence, pas une validation'}.

## 2 bis. Garde-fous méthodologiques appliqués dans cette exécution
- Produit NTL : {_prod_fr}
- Indicateurs calculés sur les valeurs {'écrêtées' if CONFIG['USE_TOPCODED'] else 'brutes, non écrêtées'} (p{CONFIG['TOPCODE_PCT']}).
- Torchères : {_flare_fr}
- Incertitude sur l'électrification : {_sens_fr}
- Détection de changement : les unités sous le p{CONFIG['CHANGE_MIN_BASE_PCT']} de la SoL initiale sont affichées mais jamais signalées ({RESULTS['E'].get('n_low_base', 0)} unité(s)).
- Série mensuelle : {_mon_fr}
- Population : {_pop_fr}
- Croissance (scénario E) : {RESULTS['E'].get('method', '—')}.
- Autocorrélation spatiale : {_sp_fr}

## 3. Seuils et sensibilité
- Plancher de bruit : **{CONFIG['NOISE_FLOOR']} nW·cm⁻²·sr⁻¹**. {noise_share:.1%} des pixels nationaux passent
  en dessous en {Y1}. Tout chiffre « non éclairé » dépend de ce seul nombre ; un test de sensibilité sur
  0,15–0,50 est exigé avant de citer un niveau plutôt qu'un rang.
- Seuil de cœur urbain : **{CONFIG['URBAN_CORE']} nW·cm⁻²·sr⁻¹**. Une comparaison entre pays n'est valide qu'à
  seuil identique.
- Écrêtage au p{CONFIG['TOPCODE_PCT']} = {TOPCODE:,.1f}. Les 1 % de pixels les plus brillants concentrent
  {top1:.1%} de la somme des lumières nationale : les agrégats nationaux sont sensibles à un très petit nombre
  de lieux.

## 4. Artefacts identifiés dans ce pays
- **Torchères suspectées : {flares}.** {'Chacune doit être vérifiée sur une carte des infrastructures pétrolières avant toute interprétation économique de la croissance du district concerné.' if flares else 'Aucune détectée aux seuils utilisés, ce qui ne prouve pas leur absence.'}
- **Halo lumineux.** Les surfaces éclairées surestiment l'emprise bâtie réelle ; une petite unité ADM2 voisine
  d'une grande ville hérite d'une lumière qui n'est pas la sienne.
- **Lumière perdue : {lost_km2:,.0f} km².** Peut traduire une dépopulation, une coupure ou un conflit — ou un
  changement de version du produit ou du nombre d'observations sans nuage. Non interprétable sans cette vérification.
- **Saisonnalité.** {'Amplitude mensuelle observée : ' + f"{RESULTS['F']['series'].groupby('month')['sol'].sum().max() / RESULTS['F']['series'].groupby('month')['sol'].sum().min():.2f}×." if RESULTS.get('F') else 'Non évaluée dans cette exécution.'} Les comparaisons de mois à mois sont invalides sans correction saisonnière.
- **Ère du capteur.** VIIRS uniquement. Toute comparaison avec les années DMSP-OLS (1992–2013) opposerait deux
  grandeurs physiques différentes et n'est pas faite ici.

## 5. Usages proscrits
- Produire un niveau de PIB, de revenu des ménages ou de taux d'électrification.
- Classer des unités administratives pour une allocation budgétaire sans un second indicateur indépendant.
- Toute affirmation sur une activité qui n'émet pas de lumière : agriculture vivrière, commerce informel diurne,
  activité intérieure en bâtiment bien isolé, économie du soin.
- Un suivi à un niveau plus fin que les unités publiées ici.

## 6. Limites administratives et population
Limites : {CONFIG['BOUNDARY_SOURCE']}. Si elles diffèrent des limites officielles nationales, les chiffres zonaux
diffèrent d'autant. Population : surfaces modélisées WorldPop, elles-mêmes une estimation dotée de sa propre
structure d'erreur, qui se propage dans chaque chiffre par habitant et chaque « population non éclairée ».

{'## 7. DONNÉES SIMULÉES' + chr(10) + "Cette exécution a utilisé le générateur de pays synthétique intégré. Tous les chiffres sont fictifs et n'existent que pour démontrer la méthode. Ils ne doivent jamais être publiés, cités ou présentés comme des statistiques relatives à " + CONFIG['COUNTRY_NAME'] + '.' if IS_DEMO else ''}
'''

(OUT / "site" / "LIMITATIONS.md").write_text(L_EN + "\n\n---\n\n" + L_FR, encoding="utf-8")
(OUT / "LIMITATIONS.md").write_text(L_EN + "\n\n---\n\n" + L_FR, encoding="utf-8")
print(L_FR[:1500] + "\n…")

In [ ]:
# --------------------------------------------------------------------------------------
# 8.5 · Assemblage de index.html
# --------------------------------------------------------------------------------------
def fig_div(key, height=430):
    if key not in FIGS:
        return "<div class='muted'>—</div>"
    f = FIGS[key]
    f.update_layout(height=height, autosize=True)
    return pio.to_html(f, include_plotlyjs=False, full_html=False, div_id=key,
                       config={"responsive": True, "displayModeBar": False})

def bi(key, tag="span", cls=""):
    en, fr = I18N[key]
    c = f' class="{cls}"' if cls else ""
    return (f'<{tag}{c} data-en="{en}" data-fr="{fr}">{(en, fr)[LANG0]}</{tag}>')

def table_html(df, cols, headers, fmts):
    '''cols : colonnes du DataFrame ; headers : clés I18N ; fmts : fonctions de mise en forme.'''
    th = "".join(f'<th data-en="{I18N[h][0]}" data-fr="{I18N[h][1]}">{I18N[h][LANG0]}</th>'
                 for h in headers)
    rows = []
    for _, r in df.iterrows():
        tds = "".join(f"<td>{f(r[c])}</td>" for c, f in zip(cols, fmts))
        rows.append(f"<tr>{tds}</tr>")
    return ('<div class="card flush"><table><thead><tr>' + th + '</tr></thead><tbody>'
            + "".join(rows) + '</tbody></table></div>')

def insight(k):
    en, fr = INSIGHTS[k]["en"], INSIGHTS[k]["fr"]
    return f'<p class="insight" data-en="{en.replace(chr(34), "&quot;")}" data-fr="{fr.replace(chr(34), "&quot;")}">{(en, fr)[LANG0]}</p>'

def kpi(key, value, sub_en="", sub_fr=""):
    return (f'<div class="kpi"><div class="kpi-label" data-en="{I18N[key][0]}" '
            f'data-fr="{I18N[key][1]}">{I18N[key][LANG0]}</div>'
            f'<div class="kpi-value">{value}</div>'
            f'<div class="kpi-sub" data-en="{sub_en}" data-fr="{sub_fr}">'
            f'{(sub_en, sub_fr)[LANG0]}</div></div>')

def panel(pid, hkey, body, active=False):
    return (f'<section class="panel{" active" if active else ""}" id="panel-{pid}">'
            f'<h2 data-en="{I18N[hkey][0]}" data-fr="{I18N[hkey][1]}">{I18N[hkey][LANG0]}</h2>'
            f'{body}</section>')

def chart_block(title_key, fig_key, height=430):
    '''Un graphique titré, dans une carte surélevée.'''
    return ('<div class="card">'
            f'<h3 data-en="{I18N[title_key][0]}" data-fr="{I18N[title_key][1]}">'
            f'{I18N[title_key][LANG0]}</h3>'
            + fig_div(fig_key, height) + '</div>')

# ---------------- rangée d'indicateurs clés --------------------------------------------
natA = RESULTS["A"]["national"]
kpis = "".join([
    kpi("kpi_sol",   f"{natA['sol_1']:,.0f}", f"{Y1} · nW·cm⁻²·sr⁻¹", f"{Y1} · nW·cm⁻²·sr⁻¹"),
    kpi("kpi_growth", f"{natA['sol_cagr']:+.1%}", f"per year, {Y0}–{Y1}", f"par an, {Y0}–{Y1}"),
    kpi("kpi_pop",   f"{natA['pop']:,.0f}", "WorldPop", "WorldPop"),
    kpi("kpi_dark",  f"{RESULTS['B']['national_dark']:,.0f}",
        f"{1-RESULTS['B']['national_lit_share']:.1%} of inhabited population",
        f"{1-RESULTS['B']['national_lit_share']:.1%} de la population habitante"),
    kpi("kpi_lit",   f"{RESULTS['C']['lit_area'][1]:,.0f} km²",
        f"{(RESULTS['C']['lit_area'][1]/RESULTS['C']['lit_area'][0]-1):+.1%} since {Y0}",
        f"{(RESULTS['C']['lit_area'][1]/RESULTS['C']['lit_area'][0]-1):+.1%} depuis {Y0}"),
    kpi("kpi_gini",  f"{RESULTS['D']['gini_1']:.3f}",
        f"{RESULTS['D']['gini_1']-RESULTS['D']['gini_0']:+.3f} since {Y0}",
        f"{RESULTS['D']['gini_1']-RESULTS['D']['gini_0']:+.3f} depuis {Y0}"),
])

# ---------------- tableaux -------------------------------------------------------------
tblA = table_html(RESULTS["A"]["adm1"].head(12),
                  ["name", "pop", "sol", "lit_share", "sol_cagr"],
                  ["tbl_region", "tbl_pop", "tbl_sol", "tbl_lit", "tbl_growth"],
                  [str, lambda v: fmt_n(v), lambda v: fmt_n(v), lambda v: fmt_p(v),
                   lambda v: fmt_p(v)])
tblB = table_html(RESULTS["B"]["adm1"].head(12),
                  ["name", "pop", "pop_lit_share", "pop_dark", "delta_pts"],
                  ["tbl_region", "tbl_pop", "tbl_access", "tbl_dark", "tbl_growth"],
                  [str, lambda v: fmt_n(v), lambda v: fmt_p(v), lambda v: fmt_n(v),
                   lambda v: f"{v:+.1f} pts" if np.isfinite(v) else "—"])
_e = RESULTS["E"]["table"].reindex(RESULTS["E"]["table"]["z_signal"].abs()
                                   .sort_values(ascending=False).index).head(12)
tblE = table_html(_e, ["name", "pop", "sol", "growth", "z_signal"],
                  ["tbl_region", "tbl_pop", "tbl_sol", "tbl_growth", "tbl_growth"],
                  [str, lambda v: fmt_n(v), lambda v: fmt_n(v), lambda v: fmt_p(v),
                   lambda v: f"z = {v:+.2f}" if np.isfinite(v) else "—"])

# ---------------- panneaux -------------------------------------------------------------
P = []
MAPX_HTML = '''<section class="mapx" id="mapx">
  <div class="mapx-head">
    <div>
      <div class="mapx-kicker" data-en="MAP EXPLORER" data-fr="EXPLORATEUR CARTOGRAPHIQUE">EXPLORATEUR CARTOGRAPHIQUE</div>
      <h3 class="mapx-title" id="mapx-title">—</h3>
      <p class="mapx-desc" id="mapx-desc"></p>
    </div>
    <div class="mapx-tools">
      <button class="mapx-icon" id="mapx-reset" type="button" data-en="Reset view" data-fr="Recentrer">Recentrer</button>
      <button class="mapx-icon" id="mapx-fs" type="button" data-en="Full screen" data-fr="Plein écran">Plein écran</button>
    </div>
  </div>
  <div class="mapx-bar">
    <div class="mapx-ctl">
      <label data-en="Level" data-fr="Niveau">Niveau</label>
      <div class="seg" id="mapx-level"></div>
    </div>
    <div class="mapx-ctl mapx-grow">
      <label for="mapx-metric" data-en="Indicator" data-fr="Indicateur">Indicateur</label>
      <select id="mapx-metric"></select>
    </div>
    <div class="mapx-ctl" id="mapx-yearctl">
      <label data-en="Year" data-fr="Année">Année</label>
      <div class="seg" id="mapx-year"></div>
    </div>
    <div class="mapx-ctl mapx-grow">
      <label for="mapx-search" data-en="Find a unit" data-fr="Trouver une unité">Trouver une unité</label>
      <input id="mapx-search" list="mapx-names" autocomplete="off" type="search">
      <datalist id="mapx-names"></datalist>
    </div>
  </div>
  <div class="mapx-body">
    <div class="mapx-mapwrap">
      <div id="mapx-map"></div>
      <div class="mapx-legend" id="mapx-legend"></div>
      <div class="seg seg-sm mapx-base" id="mapx-base"></div>
      <div class="mapx-hint" id="mapx-hint" data-en="Click a unit for details · scroll to zoom"
           data-fr="Cliquez sur une unité pour le détail · molette pour zoomer">Cliquez sur une unité pour le détail · molette pour zoomer</div>
    </div>
    <aside class="mapx-side">
      <div class="mapx-card" id="mapx-detail"></div>
      <div class="mapx-card">
        <div class="mapx-rankhead">
          <span id="mapx-rank-title"></span>
          <div class="seg seg-sm" id="mapx-rank-dir"></div>
        </div>
        <ol class="mapx-rank" id="mapx-rank"></ol>
      </div>
    </aside>
  </div>
  <p class="cap" id="mapx-foot"></p>
</section>
'''
P.append(panel("overview", "nav_overview",
    f'<div class="kpigrid">{kpis}</div>'
    + MAPX_HTML
    + f'<script id="mapx-data" type="application/json">{MAPDATA_JSON}</script>', active=True))
P.append(panel("a", "h_a", insight("A") + chart_block("c_share", "A_share", 480)
               + chart_block("c_elast", "A_elasticity", 420) + tblA))
P.append(panel("b", "h_b", insight("B") + chart_block("c_access", "B_access", 480) + tblB))
P.append(panel("c", "h_c", insight("C") + chart_block("c_trans", "C_transition", 300)))
P.append(panel("d", "h_d", insight("D") + chart_block("c_lorenz", "D_lorenz", 420)
               + chart_block("c_conv", "D_convergence", 420)))
P.append(panel("e", "h_e", insight("E")
               + (chart_block("c_series", "E_series", 330) if "E_series" in FIGS else "")
               + chart_block("c_growth", "E_growth", 470) + tblE))
P.append(panel("f", "h_f", insight("F") + (chart_block("c_monthly", "F_monthly", 430)
                                           if "F_monthly" in FIGS else "")))
verdict_cls = {"publishable": "ok", "experimental": "warn", "diagnostic": "bad"}.get(
    RESULTS["G"]["verdict"], "warn")
P.append(panel("g", "h_g",
    f'<div class="verdict {verdict_cls}"><div class="vlabel" data-en="{I18N["verdict"][0]}" '
    f'data-fr="{I18N["verdict"][1]}">{I18N["verdict"][LANG0]}</div>'
    f'<div class="vval">{RESULTS["G"]["verdict"].upper()}</div>'
    f'<div class="vtext" data-en="{RESULTS["G"]["verdict_en"]}" '
    f'data-fr="{RESULTS["G"]["verdict_fr"]}">{RESULTS["G"]["verdict_en"]}</div></div>'
    + chart_block("c_valid", "G_validation", 340)))

lim_html = "".join(
    f'<li data-en="{a}" data-fr="{b}">{a}</li>' for a, b in [
    (f"Thresholds: noise floor {CONFIG['NOISE_FLOOR']}, urban core {CONFIG['URBAN_CORE']} nW·cm⁻²·sr⁻¹; "
     f"every 'unlit' figure depends on the first of these.",
     f"Seuils : plancher de bruit {CONFIG['NOISE_FLOOR']}, cœur urbain {CONFIG['URBAN_CORE']} nW·cm⁻²·sr⁻¹ ; "
     f"tout chiffre « non éclairé » dépend du premier."),
    (f"The brightest 1 % of pixels hold {top1:.1%} of the national Sum of Lights.",
     f"Les 1 % de pixels les plus brillants concentrent {top1:.1%} de la somme des lumières nationale."),
    (f"Gas-flare candidates detected: {flares}. Verify against petroleum infrastructure before interpreting.",
     f"Torchères suspectées : {flares}. À vérifier sur une carte pétrolière avant interprétation."),
    ("Night-time lights do not measure electricity access, GDP or population — every such reading is an inference.",
     "Les lumières nocturnes ne mesurent ni l'accès à l'électricité, ni le PIB, ni la population — chaque lecture "
     "de ce type est une inférence."),
    ("Activity that emits no light is invisible here: subsistence agriculture, daytime informal trade, indoor activity.",
     "L'activité qui n'émet pas de lumière est invisible ici : agriculture vivrière, commerce informel diurne, "
     "activité intérieure."),
    ("No DMSP-OLS year is compared with a VIIRS year; the two are different physical quantities.",
     "Aucune année DMSP-OLS n'est comparée à une année VIIRS : ce sont deux grandeurs physiques différentes."),
])
P.append(panel("m", "nav_m",
    f'<h3 data-en="{I18N["method_h"][0]}" data-fr="{I18N["method_h"][1]}">{I18N["method_h"][LANG0]}</h3>'
    f'<p class="insight" data-en="VIIRS night-time light rasters for {Y0} and {Y1} were clipped to the national '
    f'extent, aligned to a common {CONFIG["TARGET_SCALE_M"]} m grid and intersected with {CONFIG["BOUNDARY_SOURCE"]} '
    f'administrative boundaries and a WorldPop population surface. Zonal statistics were computed with '
    f'latitude-corrected pixel areas. Radiance below {CONFIG["NOISE_FLOOR"]} nW·cm⁻²·sr⁻¹ is treated as unlit and '
    f'radiance was top-coded at the {CONFIG["TOPCODE_PCT"]}th percentile ({TOPCODE:,.1f}). Full code and the run '
    f'manifest are published with this page." '
    f'data-fr="Les rasters de lumières nocturnes VIIRS {Y0} et {Y1} ont été découpés à l\'emprise nationale, '
    f'alignés sur une grille commune de {CONFIG["TARGET_SCALE_M"]} m, puis croisés avec les limites administratives '
    f'{CONFIG["BOUNDARY_SOURCE"]} et une surface de population WorldPop. Les statistiques zonales sont calculées '
    f'avec des surfaces de pixel corrigées de la latitude. Une radiance inférieure à {CONFIG["NOISE_FLOOR"]} '
    f'nW·cm⁻²·sr⁻¹ est considérée comme non éclairée et la radiance est écrêtée au percentile '
    f'{CONFIG["TOPCODE_PCT"]} ({TOPCODE:,.1f}). Le code complet et le manifeste d\'exécution sont publiés avec '
    f'cette page.">…</p>'
    f'<h3 data-en="{I18N["limits_h"][0]}" data-fr="{I18N["limits_h"][1]}">{I18N["limits_h"][LANG0]}</h3>'
    f'<ul class="limits">{lim_html}</ul>'
    f'<h3 data-en="{I18N["sources_h"][0]}" data-fr="{I18N["sources_h"][1]}">{I18N["sources_h"][LANG0]}</h3>'
    '<ul class="limits">'
    '<li>NASA Black Marble / NOAA VIIRS DNB — US Government work, attribution requested</li>'
    '<li>geoBoundaries (gbOpen) — CC BY 4.0</li>'
    '<li>WorldPop — CC BY 4.0</li>'
    f'<li>Derived data: {CONFIG["LICENSE_DATA"]} · Code: {CONFIG["LICENSE_CODE"]}</li></ul>'
    f'<p class="cap">Run {RUN_ID} · provider {CONFIG["PROVIDER"]} · '
    f'<a href="manifest.json">manifest.json</a> · <a href="LIMITATIONS.md">LIMITATIONS.md</a> · '
    f'<a href="data/">data/</a></p>'))

NAV = [("overview", "nav_overview"), ("a", "nav_a"), ("b", "nav_b"), ("c", "nav_c"),
       ("d", "nav_d"), ("e", "nav_e"), ("f", "nav_f"), ("g", "nav_g"), ("m", "nav_m")]
nav_html = "".join(
    f'<button class="tab{" active" if i==0 else ""}" data-target="{pid}" '
    f'data-en="{I18N[k][0]}" data-fr="{I18N[k][1]}">{I18N[k][LANG0]}</button>'
    for i, (pid, k) in enumerate(NAV))
print(f"{len(P)} panneaux · {len(FIGS)} figures intégrées")

In [ ]:
# --------------------------------------------------------------------------------------
# 8.6 · La coquille HTML — style institutionnel inspiré de la BAD, entièrement responsive
# --------------------------------------------------------------------------------------
TEMPLATE = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>__TITLE__</title>
<meta name="description" content="__SUBTITLE__">
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
<style>
:root{
  --green:#00A86A; --deep:#00704A; --forest:#00553A; --gold:#F5C242; --ochre:#D49A00;
  --teal:#0E7C86; --terra:#C4621D; --brick:#B83B2E; --ink:#231F20; --slate:#5E6964;
  --mist:#F4F7F5; --mint:#E8F5EF; --sage:#D5DED9;
  --bg:#FFFFFF; --card:#FFFFFF; --text:var(--ink);
  box-sizing:border-box;
  padding-top:env(safe-area-inset-top,0px); padding-bottom:env(safe-area-inset-bottom,0px);
}
*,*::before,*::after{box-sizing:inherit}
html{scroll-padding-top:env(safe-area-inset-top,0px);-webkit-text-size-adjust:100%}
body{margin:0;background:var(--bg);color:var(--text);
  font-family:Inter,Calibri,"Segoe UI",system-ui,sans-serif;font-size:15px;line-height:1.6}
.wrap{max-width:1180px;margin:0 auto;padding:0 20px}
header.hero{background:linear-gradient(118deg,var(--forest) 0%,var(--deep) 46%,var(--green) 100%);
  color:#fff;padding:34px 0 0 0;position:relative;overflow:hidden}
header.hero::after{content:"";position:absolute;right:-90px;top:-70px;width:340px;height:340px;
  border:26px solid rgba(255,255,255,.06);border-radius:50%}
.kicker{font-size:10.5px;letter-spacing:3px;font-weight:700;color:var(--gold);text-transform:uppercase}
h1{font-size:clamp(25px,4.4vw,40px);font-weight:700;margin:12px 0 4px;line-height:1.12}
.sub{font-size:clamp(14px,2.2vw,18px);color:#E6F6EE;font-style:italic;margin:0 0 6px}
.period{font-size:13px;color:#BFE6D5;margin:0}
.goldband{height:5px;background:var(--gold);margin-top:22px}
.langbtn{position:absolute;top:22px;right:20px;display:flex;gap:0;border:1.4px solid rgba(255,255,255,.55);
  border-radius:999px;overflow:hidden;z-index:5}
.langbtn button{background:transparent;border:0;color:#fff;font:inherit;font-size:12.5px;font-weight:700;
  padding:7px 15px;cursor:pointer;letter-spacing:.6px}
.langbtn button.on{background:var(--gold);color:var(--forest)}
.demo{background:#FFF4D6;border-bottom:2px solid var(--ochre);color:#7a5800;
  font-weight:700;font-size:13px;text-align:center;padding:9px 12px}
nav.tabs{position:sticky;top:0;z-index:20;background:rgba(255,255,255,.97);
  border-bottom:1px solid var(--sage);backdrop-filter:blur(8px);
  padding-top:env(safe-area-inset-top,0px)}
nav.tabs .wrap{display:flex;gap:4px;overflow-x:auto;scrollbar-width:none}
nav.tabs .wrap::-webkit-scrollbar{display:none}
.tab{background:none;border:0;border-bottom:3px solid transparent;color:var(--slate);
  font:inherit;font-size:13.5px;font-weight:600;padding:14px 13px;white-space:nowrap;cursor:pointer}
.tab:hover{color:var(--deep)}
.tab.active{color:var(--deep);border-bottom-color:var(--green)}
main{padding:28px 0 60px}
.panel{display:none;animation:fade .28s ease}
.panel.active{display:block}
@keyframes fade{from{opacity:0;transform:translateY(5px)}to{opacity:1;transform:none}}
h2{font-size:clamp(20px,3.2vw,27px);font-weight:700;margin:4px 0 16px;color:var(--ink)}
h3{font-size:15.5px;font-weight:700;margin:30px 0 10px;color:var(--deep)}
.insight{background:var(--mint);border-left:4px solid var(--green);border-radius:0 8px 8px 0;
  padding:15px 18px;margin:0 0 20px;font-size:14.5px}
.cap{font-size:12px;font-style:italic;color:var(--slate);margin-top:6px}
.kpigrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(178px,1fr));gap:13px;margin-bottom:26px}
.kpi{position:relative;background:var(--card);border:1px solid var(--sage);border-radius:13px;
  padding:16px 16px 14px 19px;overflow:hidden;box-shadow:0 1px 2px rgba(35,31,32,.04);
  transition:transform .2s ease,box-shadow .2s ease}
.kpi::before{content:"";position:absolute;left:0;top:0;bottom:0;width:4px;background:var(--green)}
.kpi:nth-child(2)::before{background:var(--teal)} .kpi:nth-child(3)::before{background:var(--deep)}
.kpi:nth-child(4)::before{background:var(--brick)} .kpi:nth-child(5)::before{background:var(--ochre)}
.kpi:nth-child(6)::before{background:var(--terra)}
.kpi:hover{transform:translateY(-2px);box-shadow:0 10px 26px rgba(35,31,32,.10)}
.kpi-label{font-size:10.5px;letter-spacing:1.4px;text-transform:uppercase;font-weight:700;color:var(--deep)}
.kpi-value{font-size:clamp(21px,3.4vw,29px);font-weight:700;color:var(--ink);margin:6px 0 2px;line-height:1.1}
.kpi-sub{font-size:11.5px;color:var(--slate)}
table{width:100%;border-collapse:collapse;margin-top:16px;font-size:13.3px;display:block;overflow-x:auto}
thead th{background:var(--deep);color:#fff;text-align:left;padding:9px 11px;font-weight:600;
  font-size:11.5px;letter-spacing:.5px;text-transform:uppercase;white-space:nowrap}
tbody td{padding:8px 11px;border-bottom:1px solid var(--sage);white-space:nowrap}
tbody tr:nth-child(even){background:var(--mist)}
tbody tr:hover{background:var(--mint)}
.limits{margin:0;padding-left:20px}
.limits li{margin-bottom:9px;font-size:14px}
.verdict{border-radius:11px;padding:18px 20px;margin-bottom:22px;border:1.6px solid}
.verdict.ok{background:var(--mint);border-color:var(--green)}
.verdict.warn{background:#FFF7E3;border-color:var(--ochre)}
.verdict.bad{background:#FBEDEB;border-color:var(--brick)}
.vlabel{font-size:10.5px;letter-spacing:1.6px;text-transform:uppercase;font-weight:700;color:var(--slate)}
.vval{font-size:25px;font-weight:700;margin:3px 0 7px}
.verdict.ok .vval{color:var(--deep)} .verdict.warn .vval{color:var(--ochre)}
.verdict.bad .vval{color:var(--brick)}
.vtext{font-size:14px}
footer{background:var(--forest);color:#CFE9DC;padding:26px 0;font-size:12.5px}
footer a{color:var(--gold);text-decoration:none} footer a:hover{text-decoration:underline}
.fgrid{display:flex;flex-wrap:wrap;gap:26px;justify-content:space-between;align-items:flex-start}
.js-plotly-plot{width:100%!important}
/* ---- cards ------------------------------------------------------------------------ */
.card{background:var(--card);border:1px solid var(--sage);border-radius:14px;
  padding:18px 18px 10px;margin:0 0 20px;box-shadow:0 1px 2px rgba(35,31,32,.04),0 8px 24px rgba(35,31,32,.05);
  transition:box-shadow .22s ease,transform .22s ease}
.card:hover{box-shadow:0 1px 2px rgba(35,31,32,.05),0 14px 34px rgba(35,31,32,.09)}
.card h3{margin:0 0 12px;font-size:15px;letter-spacing:.1px;display:flex;align-items:center;gap:9px}
.card h3::before{content:"";width:4px;height:17px;border-radius:2px;background:var(--green);flex:none}
.card table{margin-top:0}
.card.flush{padding:0;overflow:hidden}
.card.flush table{border-radius:0}
/* ---- hero chips -------------------------------------------------------------------- */
.chips{display:flex;flex-wrap:wrap;gap:8px;margin:16px 0 2px;padding:0}
.chip{display:inline-flex;align-items:center;gap:7px;background:rgba(255,255,255,.13);
  border:1px solid rgba(255,255,255,.26);border-radius:999px;padding:5px 13px;
  font-size:11.5px;font-weight:600;letter-spacing:.4px;color:#EAF7F0;backdrop-filter:blur(3px)}
.chip .dot{width:7px;height:7px;border-radius:50%;background:var(--gold);flex:none}
.chip.v-ok .dot{background:#5CE6A8} .chip.v-warn .dot{background:var(--gold)}
.chip.v-bad .dot{background:#FF8A7A}
/* ---- print ------------------------------------------------------------------------- */
@media print{nav.tabs,.langbtn,.demo{display:none!important}
  .panel{display:block!important;page-break-after:always}
  .card{box-shadow:none;break-inside:avoid}
  header.hero{-webkit-print-color-adjust:exact;print-color-adjust:exact}}
@media (max-width:640px){.tab{padding:12px 9px;font-size:12.5px}.wrap{padding:0 14px}
  .insight{font-size:13.6px}.langbtn{top:16px;right:14px}}
/* ---- explorateur cartographique ---------------------------------------------------- */
.mapx{background:var(--card);border:1px solid var(--sage);border-radius:16px;padding:18px 18px 12px;
  margin:0 0 22px;box-shadow:0 1px 2px rgba(35,31,32,.04),0 10px 30px rgba(35,31,32,.07)}
.mapx-head{display:flex;justify-content:space-between;gap:14px;align-items:flex-start;flex-wrap:wrap}
.mapx-kicker{font-size:10px;letter-spacing:2.6px;font-weight:700;color:var(--green)}
.mapx-title{margin:4px 0 2px;font-size:clamp(17px,2.4vw,21px);color:var(--ink);display:block}
.mapx-title::before{display:none}
.mapx-desc{margin:0;font-size:13px;color:var(--slate);max-width:720px}
.mapx-tools{display:flex;gap:8px}
.mapx-icon{border:1px solid var(--sage);background:var(--card);color:var(--deep);font:inherit;font-size:12.5px;
  font-weight:600;padding:7px 12px;border-radius:9px;cursor:pointer;transition:all .15s ease}
.mapx-icon:hover{border-color:var(--green);background:var(--mint)}
.mapx-bar{display:flex;flex-wrap:wrap;gap:12px 16px;margin:14px 0 12px;padding:12px;
  background:var(--mist);border:1px solid var(--sage);border-radius:12px}
.mapx-ctl{display:flex;flex-direction:column;gap:5px;min-width:0}
.mapx-grow{flex:1 1 220px}
.mapx-ctl label{font-size:10px;letter-spacing:1.5px;text-transform:uppercase;font-weight:700;color:var(--deep)}
.mapx-ctl select,.mapx-ctl input{font:inherit;font-size:13.5px;color:var(--ink);background:var(--card);
  border:1px solid var(--sage);border-radius:9px;padding:8px 10px;min-height:38px;width:100%}
.mapx-ctl select:focus,.mapx-ctl input:focus{outline:2px solid rgba(0,168,106,.35);border-color:var(--green)}
.seg{display:inline-flex;background:var(--card);border:1px solid var(--sage);border-radius:10px;padding:3px;gap:2px}
.seg button{border:0;background:transparent;font:inherit;font-size:12.5px;font-weight:600;color:var(--slate);
  padding:7px 12px;border-radius:7px;cursor:pointer;white-space:nowrap;min-height:30px;transition:all .15s ease}
.seg button:hover{color:var(--deep)}
.seg button.on{background:var(--deep);color:#fff;box-shadow:0 1px 3px rgba(0,85,58,.35)}
.seg-sm button{font-size:11.5px;padding:5px 9px;min-height:26px}
.mapx-body{display:grid;grid-template-columns:minmax(0,1fr) 318px;gap:14px}
.mapx-mapwrap{position:relative;border-radius:13px;overflow:hidden;background:#0b0f14;
  height:clamp(380px,64vh,660px);border:1px solid #1e2a33}
#mapx-map{position:absolute;inset:0}
.mapx-legend{position:absolute;left:12px;bottom:12px;z-index:3;background:rgba(12,18,24,.82);color:#e8eef0;
  border:1px solid rgba(255,255,255,.12);border-radius:10px;padding:9px 11px 8px;min-width:210px;max-width:62%;
  backdrop-filter:blur(6px);font-size:11px;pointer-events:none}
.mapx-legend .lt{font-weight:700;font-size:11.5px;margin-bottom:6px;letter-spacing:.2px}
.mapx-legend .bar{height:10px;border-radius:5px;border:1px solid rgba(255,255,255,.18)}
.mapx-legend .ticks{display:flex;justify-content:space-between;margin-top:4px;color:#b9c7cc;font-variant-numeric:tabular-nums}
.mapx-legend .na{margin-top:5px;color:#8fa0a6;display:flex;align-items:center;gap:6px}
.mapx-legend .na i{display:inline-block;width:12px;height:9px;border-radius:2px;background:#3a444c}
.mapx-base{position:absolute;top:10px;right:10px;z-index:3;background:rgba(12,18,24,.72);border-color:rgba(255,255,255,.16)}
.mapx-base button{color:#c9d4d8}
.mapx-base button.on{background:var(--gold);color:var(--forest)}
.mapx-hint{position:absolute;top:12px;left:12px;z-index:3;font-size:11px;color:#c9d4d8;
  background:rgba(12,18,24,.6);padding:5px 9px;border-radius:7px;pointer-events:none;transition:opacity .4s}
.mapx-side{display:flex;flex-direction:column;gap:12px;min-width:0}
.mapx-card{border:1px solid var(--sage);border-radius:13px;padding:14px;background:var(--card)}
.mapx-dname{font-size:17px;font-weight:700;color:var(--ink);line-height:1.25}
.mapx-dparent{font-size:12px;color:var(--slate);margin-top:2px}
.mapx-dbig{display:flex;align-items:baseline;gap:8px;margin:12px 0 2px}
.mapx-dbig b{font-size:26px;color:var(--deep);font-variant-numeric:tabular-nums}
.mapx-dbig span{font-size:12px;color:var(--slate)}
.mapx-pos{position:relative;height:8px;border-radius:4px;margin:10px 0 4px}
.mapx-pos i{position:absolute;top:-4px;width:3px;height:16px;border-radius:2px;background:var(--ink)}
.mapx-pos em{position:absolute;top:-3px;width:2px;height:14px;background:#fff;border:1px solid var(--slate)}
.mapx-poslab{display:flex;justify-content:space-between;font-size:10.5px;color:var(--slate)}
.mapx-dl{margin:12px 0 0;padding:0;list-style:none;border-top:1px solid var(--sage)}
.mapx-dl li{display:flex;justify-content:space-between;gap:10px;padding:6px 0;border-bottom:1px solid var(--mist);
  font-size:12.5px;cursor:pointer}
.mapx-dl li:hover{color:var(--deep)}
.mapx-dl li.cur{font-weight:700;color:var(--deep)}
.mapx-dl li span:last-child{font-variant-numeric:tabular-nums;white-space:nowrap}
.mapx-rankhead{display:flex;justify-content:space-between;align-items:center;gap:8px;margin-bottom:8px;
  font-size:11px;letter-spacing:1.3px;text-transform:uppercase;font-weight:700;color:var(--deep)}
.mapx-rank{list-style:none;margin:0;padding:0;counter-reset:r}
.mapx-rank li{display:grid;grid-template-columns:22px 1fr auto;gap:8px;align-items:center;padding:6px 6px;
  border-radius:8px;cursor:pointer;font-size:12.8px}
.mapx-rank li:hover,.mapx-rank li.sel{background:var(--mint)}
.mapx-rank li b{font-size:11px;color:var(--slate);text-align:right}
.mapx-rank li .nm{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.mapx-rank li .vl{font-variant-numeric:tabular-nums;font-weight:600}
.mapx-rank li .sw{display:inline-block;width:9px;height:9px;border-radius:2px;margin-right:6px;vertical-align:0}
.mapx.fs{position:fixed;inset:0;z-index:100;margin:0;border-radius:0;overflow:auto;
  padding:calc(12px + env(safe-area-inset-top,0px)) 14px calc(12px + env(safe-area-inset-bottom,0px))}
.mapx.fs .mapx-mapwrap{height:calc(100vh - 230px);min-height:360px}
@media (max-width:900px){
  .mapx-body{grid-template-columns:1fr}
  .mapx-side{display:grid;grid-template-columns:1fr 1fr;gap:12px}
}
@media (max-width:620px){
  .mapx{padding:12px 10px 8px;border-radius:12px}
  .mapx-side{grid-template-columns:1fr}
  .mapx-mapwrap{height:auto}
  #mapx-map{position:relative;height:clamp(300px,52vh,480px)}
  .mapx-legend{position:static;min-width:0;max-width:none;border-radius:0;border:0;
    border-top:1px solid rgba(255,255,255,.12);background:#0f161c;padding:9px 12px 10px}
  .mapx-hint{display:none}
  .mapx-bar{padding:10px}
}
@media print{.mapx-bar,.mapx-tools,.mapx-base,.mapx-hint{display:none!important}}
</style>
</head>
<body>
<header class="hero">
  __LANGBTN__
  <div class="wrap">
    <div class="kicker">African Development Bank · AU STATAFRIC · STG17 · Night-Time Lights</div>
    <h1 data-en="__TITLE_EN__" data-fr="__TITLE_FR__">__TITLE_DEF__</h1>
    <p class="sub" data-en="__SUB_EN__" data-fr="__SUB_FR__">__SUB_DEF__</p>
    <p class="period" data-en="__PER_EN__" data-fr="__PER_FR__">__PER_DEF__</p>
    <div class="chips">__CHIPS__</div>
  </div>
  <div class="goldband"></div>
</header>
__DEMO__
<nav class="tabs"><div class="wrap">__NAV__</div></nav>
<main><div class="wrap">__PANELS__</div></main>
<footer><div class="wrap fgrid">
  <div><strong>__TITLE_EN__</strong><br>__AUTHOR__<br>
    <span data-en="Run __RUN__ · generated __DATE__" data-fr="Exécution __RUN__ · générée le __DATE__">Exécution __RUN__</span></div>
  <div>Sources : VIIRS/Black Marble · geoBoundaries (CC BY 4.0) · WorldPop (CC BY 4.0)<br>
    <span data-en="Derived data" data-fr="Données dérivées">Données dérivées</span> : __LICD__ ·
    Code : __LICC__</div>
  <div><a href="manifest.json">manifest.json</a> · <a href="LIMITATIONS.md">LIMITATIONS.md</a> ·
       <a href="data/">data/</a></div>
</div></footer>
<script>
const AXIS_I18N = __AXIS__;
const LANGS = __LANGS__;
let LANG = LANGS[0];
function setLang(l){
  LANG = l;
  document.documentElement.lang = l;
  var be = document.getElementById("btn-en"), bf = document.getElementById("btn-fr");
  if(be) be.classList.toggle("on", l==="en");
  if(bf) bf.classList.toggle("on", l==="fr");
  document.querySelectorAll("[data-en]").forEach(function(e){
    var v = (l==="fr" ? e.dataset.fr : e.dataset.en);
    if(v !== undefined && v !== null && v !== "") e.textContent = v;
  });
  Object.keys(AXIS_I18N).forEach(function(id){
    var el = document.getElementById(id);
    if(!el || !el.layout) return;
    var up = {};
    Object.keys(AXIS_I18N[id]).forEach(function(k){ up[k] = AXIS_I18N[id][k][l==="fr"?1:0]; });
    try{ Plotly.relayout(el, up); }catch(err){}
  });
  try{ localStorage.setItem("ntl_lang", l); }catch(e){}
  if(typeof MAPX !== "undefined" && MAPX){ try{ MAPX.onLang(); }catch(e){} }
}
document.querySelectorAll(".tab").forEach(function(b){
  b.addEventListener("click", function(){
    document.querySelectorAll(".tab").forEach(function(x){x.classList.remove("active");});
    document.querySelectorAll(".panel").forEach(function(x){x.classList.remove("active");});
    b.classList.add("active");
    var p = document.getElementById("panel-"+b.dataset.target);
    p.classList.add("active");
    p.querySelectorAll(".js-plotly-plot").forEach(function(g){
      try{ Plotly.Plots.resize(g); }catch(e){}
    });
    window.scrollTo({top:0,behavior:"smooth"});
  });
});
/* ---- explorateur cartographique ------------------------------------------------------ */
var MAPX = (function(){
  var host = document.getElementById("mapx");
  var dataEl = document.getElementById("mapx-data");
  if(!host || !dataEl){ return null; }
  var D = JSON.parse(dataEl.textContent);
  var LEVELS = Object.keys(D.levels);
  var SCALES = {
    night: [[0,"#1d2b4f"],[0.3,"#24508a"],[0.55,"#2e9aa6"],[0.78,"#9fd67a"],[0.9,"#f5c242"],[1,"#fff3c4"]],
    green: [[0,"#eaf6ef"],[0.25,"#b9e4cc"],[0.5,"#6fcaa0"],[0.75,"#1f9e6b"],[1,"#00553A"]],
    brick: [[0,"#fbf1e8"],[0.3,"#f2c7a3"],[0.6,"#e0875a"],[0.85,"#c4521d"],[1,"#8f2b22"]],
    teal:  [[0,"#e8f4f5"],[0.35,"#9fd1d5"],[0.7,"#3a9ea6"],[1,"#0b5d64"]],
    div:   [[0,"#b83b2e"],[0.25,"#e3907f"],[0.5,"#f2efe6"],[0.75,"#6fcaa0"],[1,"#00704A"]]
  };
  var BASES = [["carto-darkmatter",["Night","Nuit"]],["carto-positron",["Light","Clair"]],["plain",["Plain","Uni"]]];
  // Fond « uni » : un style MapLibre embarqué, sans aucune requête réseau. La carte reste
  // donc utilisable hors ligne (clé USB, Wi-Fi coupé) : les fonds CARTO ne sont activés
  // qu'une fois leur serveur joint.
  var PLAIN = {version: 8, sources: {}, layers: [{id: "bg", type: "background", paint: {"background-color": "#0b0f14"}}]};
  var ONLINE = null;
  var S = {lvl: LEVELS[0], m: "sol_per_capita", y: String(D.years[D.years.length-1]),
           base: "plain", want: "carto-darkmatter", sel: null, dir: "top", rev: 1, view: null, touched: false};
  function styleOf(b){ return b === "plain" ? PLAIN : b; }
  function probe(){
    var done = false;
    function fin(ok){
      if(done) return; done = true; ONLINE = ok;
      if(ok && S.base === "plain" && S.want !== "plain"){ S.base = S.want; }
      refresh();
    }
    try{
      fetch("https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json", {mode: "cors", cache: "force-cache"})
        .then(function(r){ fin(r.ok); }).catch(function(){ fin(false); });
    }catch(e){ fin(false); }
    setTimeout(function(){ fin(false); }, 4500);
  }
  var mapEl = document.getElementById("mapx-map");

  function fr(){ return (typeof LANG !== "undefined" && LANG === "fr"); }
  function t(pair){ return Array.isArray(pair) ? pair[fr() ? 1 : 0] : (fr() ? pair.fr : pair.en); }
  function L(){ return D.levels[S.lvl]; }
  function M(k){ for(var i=0;i<D.metrics.length;i++){ if(D.metrics[i].key === k) return D.metrics[i]; } return null; }
  function avail(){ return D.metrics.filter(function(m){ return L().values[m.key]; }); }
  function slot(m){ return m.kind === "change" ? "chg" : S.y; }
  function vals(k){ var m = M(k), v = L().values[k]; return v ? (v[slot(m)] || []) : []; }

  function nf(v, d){
    return v.toLocaleString(fr() ? "fr-FR" : "en-GB", {minimumFractionDigits: d, maximumFractionDigits: d});
  }
  function fmt(v, f){
    if(v === null || v === undefined || !isFinite(v)) return fr() ? "n.d." : "n/a";
    var sg = v > 0 ? "+" : "", pc = fr() ? " %" : "%";
    switch(f){
      case "int": return nf(Math.round(v), 0);
      case "dec1": return nf(v, 1);
      case "dec2": return nf(v, 2);
      case "dec3": return nf(v, Math.abs(v) < 1 ? 3 : 2);
      case "pct": return nf(v * 100, 1) + pc;
      case "pctyr": return sg + nf(v * 100, 1) + (fr() ? " %/an" : "%/yr");
      case "pts": return sg + nf(v, 1) + " pts";
      case "ratio": return nf(v, 2) + " ×";
      case "z": return sg + nf(v, 2);
    }
    return nf(v, 2);
  }
  function finite(a){ return a.filter(function(v){ return v !== null && isFinite(v); }).sort(function(a, b){ return a - b; }); }
  function q(s, p){ if(!s.length) return NaN; var i = (s.length - 1) * p, lo = Math.floor(i), hi = Math.ceil(i);
    return s[lo] + (s[hi] - s[lo]) * (i - lo); }
  function range(m, a){
    var s = finite(a), lo = q(s, 0.02), hi = q(s, 0.98);
    if(m.mid !== undefined){ var sp = Math.max(Math.abs(lo - m.mid), Math.abs(hi - m.mid)) || 1; lo = m.mid - sp; hi = m.mid + sp; }
    if(!(hi > lo)){ hi = lo + (Math.abs(lo) || 1) * 0.1; }
    return [lo, hi];
  }
  function hex(h){ return [parseInt(h.substr(1,2),16), parseInt(h.substr(3,2),16), parseInt(h.substr(5,2),16)]; }
  function colorAt(sc, x){
    x = Math.max(0, Math.min(1, x));
    for(var i = 1; i < sc.length; i++){
      if(x <= sc[i][0]){
        var a = hex(sc[i-1][1]), b = hex(sc[i][1]), f = (x - sc[i-1][0]) / ((sc[i][0] - sc[i-1][0]) || 1);
        return "rgb(" + [0,1,2].map(function(j){ return Math.round(a[j] + (b[j] - a[j]) * f); }).join(",") + ")";
      }
    }
    return sc[sc.length-1][1];
  }
  function gradient(sc){ return "linear-gradient(90deg," + sc.map(function(s){ return s[1] + " " + (s[0]*100) + "%"; }).join(",") + ")"; }

  function fit(b, extra){
    var W = mapEl.clientWidth || 800, H = mapEl.clientHeight || 520, pad = extra || 0;
    var dx = (b[2] - b[0]) * pad, dy = (b[3] - b[1]) * pad;
    b = [b[0] - dx, b[1] - dy, b[2] + dx, b[3] + dy];
    function my(lat){ var r = lat * Math.PI / 180; return Math.log(Math.tan(Math.PI / 4 + r / 2)); }
    var lonSpan = Math.max(b[2] - b[0], 1e-4);
    var yf = Math.max((my(b[3]) - my(b[1])) / (2 * Math.PI), 1e-6);
    var zx = Math.log2(W * 360 / (512 * lonSpan)), zy = Math.log2(H / (512 * yf));
    var z = Math.max(1, Math.min(Math.min(zx, zy) - 0.25, 11.5));
    var cy = (my(b[1]) + my(b[3])) / 2;
    return {center: {lon: (b[0] + b[2]) / 2, lat: (2 * Math.atan(Math.exp(cy)) - Math.PI / 2) * 180 / Math.PI}, zoom: z};
  }
  function setView(v){ S.view = v; S.rev += 1; }

  function segs(id, items, cur, onpick){
    var el = document.getElementById(id); el.innerHTML = "";
    items.forEach(function(it){
      var b = document.createElement("button"); b.type = "button";
      b.textContent = it[1]; if(String(it[0]) === String(cur)) b.className = "on";
      b.addEventListener("click", function(){ onpick(it[0]); });
      el.appendChild(b);
    });
  }
  function controls(){
    segs("mapx-level", LEVELS.map(function(l){ return [l, t(D.levels[l].label)]; }), S.lvl, function(l){
      S.lvl = l; S.sel = null; if(!L().values[S.m]) S.m = "growth" in L().values ? "growth" : "sol_per_capita";
      setView(fit(D.bounds)); refresh(); });
    segs("mapx-year", D.years.map(function(y){ return [String(y), String(y)]; }), S.y, function(y){ S.y = String(y); refresh(); });
    segs("mapx-base", BASES.map(function(b){ return [b[0], t(b[1])]; }), S.base, function(b){
      if(b !== "plain" && ONLINE === false) return;
      S.base = b; S.want = b; refresh(); });
    document.querySelectorAll("#mapx-base button").forEach(function(btn, k){
      var off = (BASES[k][0] !== "plain" && ONLINE === false);
      btn.disabled = off; btn.style.opacity = off ? 0.4 : "";
      btn.title = off ? (fr() ? "Fond de carte indisponible hors ligne" : "Basemap unavailable offline") : "";
    });
    segs("mapx-rank-dir", [["top", fr() ? "Plus élevés" : "Highest"], ["bottom", fr() ? "Plus faibles" : "Lowest"]], S.dir,
         function(d){ S.dir = d; refresh(); });
    var sel = document.getElementById("mapx-metric"); sel.innerHTML = "";
    var groups = {};
    avail().forEach(function(m){
      if(!groups[m.group]){ groups[m.group] = document.createElement("optgroup"); groups[m.group].label = t(D.groups[m.group]); sel.appendChild(groups[m.group]); }
      var o = document.createElement("option"); o.value = m.key; o.textContent = t(m); if(m.key === S.m) o.selected = true;
      groups[m.group].appendChild(o);
    });
    var dl = document.getElementById("mapx-names"); dl.innerHTML = "";
    L().names.forEach(function(n){ var o = document.createElement("option"); o.value = n; dl.appendChild(o); });
    document.getElementById("mapx-search").placeholder = fr() ? "Nom d'une unité…" : "Unit name…";
    document.getElementById("mapx-yearctl").style.display = M(S.m).kind === "change" ? "none" : "";
  }

  var BUSY = false, DIRTY = false;
  function render(){
    // Les panneaux (légende, fiche, classement) sont mis à jour immédiatement ; seul le
    // dessin de la carte, asynchrone, est sérialisé : une demande reçue pendant un dessin
    // est rejouée ensuite avec l'état le plus récent.
    try{ ui(); }catch(e){ if(window.console) console.error(e); }
    if(BUSY){ DIRTY = true; return; }
    BUSY = true; DIRTY = false;
    var p = null;
    try{ p = draw(); }catch(e){ if(window.console) console.error(e); }
    var done = function(){ BUSY = false; if(DIRTY){ DIRTY = false; BUSY = true;
      var p2 = null; try{ p2 = draw(); }catch(e){}
      if(p2 && typeof p2.then === "function"){ p2.then(done, done); } else { BUSY = false; } } };
    if(p && typeof p.then === "function"){ p.then(done, done); } else { done(); }
  }
  function draw(){
    if(!window.Plotly){ mapEl.innerHTML = "<p style='color:#ccc;padding:20px'>Plotly indisponible.</p>"; return; }
    var m = M(S.m), a = vals(S.m), lv = L(), sc = SCALES[m.scale], rg = range(m, a);
    var ranks = {}, order = lv.ids.map(function(id, i){ return i; }).filter(function(i){ return a[i] !== null && isFinite(a[i]); })
      .sort(function(i, j){ return a[j] - a[i]; });
    order.forEach(function(i, r){ ranks[i] = r + 1; });
    var ok = order.slice().sort(function(i, j){ return i - j; });
    var rankTxt = function(i){ return ranks[i] ? ((fr() ? "Rang " : "Rank ") + ranks[i] + " / " + order.length) : ""; };
    var dark = S.base !== "carto-positron";
    var base = {type: "choroplethmap", geojson: lv.geojson, featureidkey: "properties.shapeID",
      locations: lv.ids, z: lv.ids.map(function(){ return 0; }), colorscale: [[0, "#3a444c"], [1, "#3a444c"]],
      showscale: false, marker: {opacity: 0.55, line: {width: 0.4, color: dark ? "rgba(255,255,255,.25)" : "#ffffff"}},
      customdata: lv.names.map(function(n, i){ return [n, lv.parents[i]]; }),
      hovertemplate: "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>" + (fr() ? "donnée non disponible" : "no data") + "<extra></extra>"};
    var main = {type: "choroplethmap", geojson: lv.geojson, featureidkey: "properties.shapeID",
      locations: ok.map(function(i){ return lv.ids[i]; }), z: ok.map(function(i){ return a[i]; }),
      zmin: rg[0], zmax: rg[1], colorscale: sc, showscale: false,
      marker: {opacity: dark ? 0.86 : 0.9, line: {width: S.lvl === "ADM1" ? 0.9 : 0.45, color: dark ? "rgba(255,255,255,.55)" : "rgba(255,255,255,.95)"}},
      customdata: ok.map(function(i){ return [lv.names[i], lv.parents[i], fmt(a[i], m.fmt), rankTxt(i)]; }),
      hovertemplate: "<b>%{customdata[0]}</b><br><span style='color:#9fb8bf'>%{customdata[1]}</span><br>" +
        "<b style='font-size:14px'>%{customdata[2]}</b><br><i>%{customdata[3]}</i><extra></extra>"};
    var traces = [base, main];
    if(S.sel !== null){
      traces.push({type: "choroplethmap", geojson: lv.geojson, featureidkey: "properties.shapeID",
        locations: [lv.ids[S.sel]], z: [1], colorscale: [[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]], showscale: false,
        marker: {opacity: 1, line: {width: 3.2, color: "#F5C242"}}, hoverinfo: "skip"});
    }
    if(!S.view) S.view = fit(D.bounds);
    var layout = {map: {style: styleOf(S.base), center: S.view.center, zoom: S.view.zoom}, uirevision: S.rev,
      margin: {l: 0, r: 0, t: 0, b: 0}, paper_bgcolor: "rgba(0,0,0,0)", showlegend: false, dragmode: "pan",
      hoverlabel: {bgcolor: "#0f1a20", bordercolor: "#F5C242", font: {color: "#fff", size: 12, family: "Inter,Calibri,sans-serif"}}};
    var layers = [];
    if(S.lvl !== "ADM1" && D.levels.ADM1){
      layers.push({sourcetype: "geojson", source: D.levels.ADM1.geojson, type: "line",
                   color: dark ? "rgba(255,255,255,0.8)" : "#00553A", line: {width: 1.3}, opacity: 0.85});
    }
    if(D.outline){
      layers.push({sourcetype: "geojson", source: D.outline, type: "line", color: "#F5C242", line: {width: 1.8}, opacity: 0.9});
    }
    layout.map.layers = layers;
    var pr = Plotly.react(mapEl, traces, layout, {responsive: true, displayModeBar: false, scrollZoom: true});
    if(!mapEl._mapxBound){
      mapEl._mapxBound = true;
      mapEl.on("plotly_click", function(ev){
        var p = ev && ev.points && ev.points[0]; if(!p) return;
        var i = L().ids.indexOf(p.location); if(i >= 0){ select(i, false); }
      });
      mapEl.on("plotly_relayout", function(){ if(!S.touched){ S.touched = true; var h = document.getElementById("mapx-hint"); if(h) h.style.opacity = 0; } });
    }
    return pr;
  }
  function ui(){
    var m = M(S.m), a = vals(S.m), lv = L(), sc = SCALES[m.scale], rg = range(m, a);
    var ranks = {}, order = lv.ids.map(function(id, i){ return i; }).filter(function(i){ return a[i] !== null && isFinite(a[i]); })
      .sort(function(i, j){ return a[j] - a[i]; });
    order.forEach(function(i, r){ ranks[i] = r + 1; });
    legend(m, rg, sc);
    ranking(m, a, order, rg, sc);
    detail(m, a, ranks, order.length, rg, sc);
    texts(m);
  }

  function legend(m, rg, sc){
    var mid = m.mid !== undefined ? fmt(m.mid, m.fmt) : fmt((rg[0] + rg[1]) / 2, m.fmt);
    var yr = m.kind === "change" ? (D.years[0] + "–" + D.years[D.years.length-1]) : S.y;
    document.getElementById("mapx-legend").innerHTML =
      "<div class='lt'>" + t(m) + " · " + yr + "</div><div class='bar' style='background:" + gradient(sc) + "'></div>" +
      "<div class='ticks'><span>≤ " + fmt(rg[0], m.fmt) + "</span><span>" + mid + "</span><span>≥ " + fmt(rg[1], m.fmt) + "</span></div>" +
      "<div class='na'><i></i>" + (fr() ? "donnée non disponible" : "no data") + "</div>";
  }

  function ranking(m, a, order, rg, sc){
    var lv = L(), ol = document.getElementById("mapx-rank"), n = Math.min(10, order.length);
    var pick = S.dir === "top" ? order.slice(0, n) : order.slice(-n).reverse();
    document.getElementById("mapx-rank-title").textContent = (fr() ? "Classement · " : "Ranking · ") + t(lv.label).split(" · ")[0];
    ol.innerHTML = "";
    pick.forEach(function(i){
      var r = order.indexOf(i) + 1, li = document.createElement("li");
      if(S.sel === i) li.className = "sel";
      var col = colorAt(sc, (a[i] - rg[0]) / (rg[1] - rg[0]));
      li.innerHTML = "<b>" + r + "</b><span class='nm'><span class='sw' style='background:" + col + "'></span>" +
        esc(lv.names[i]) + "</span><span class='vl'>" + fmt(a[i], m.fmt) + "</span>";
      li.title = lv.names[i] + (lv.parents[i] ? " — " + lv.parents[i] : "");
      li.addEventListener("click", function(){ select(i, true); });
      ol.appendChild(li);
    });
  }

  function detail(m, a, ranks, n, rg, sc){
    var lv = L(), box = document.getElementById("mapx-detail"), nat = D.national[S.m], natv = nat ? nat[slot(m)] : null;
    var h = "";
    if(S.sel === null){
      var s = finite(a);
      h += "<div class='mapx-dname'>" + esc(D.country) + "</div><div class='mapx-dparent'>" +
           (fr() ? "Vue d'ensemble · " : "Overview · ") + t(lv.label) + "</div>";
      h += "<div class='mapx-dbig'><b>" + (natv !== null && natv !== undefined ? fmt(natv, m.fmt) : fmt(q(s, 0.5), m.fmt)) +
           "</b><span>" + (natv !== null && natv !== undefined ? (m.ext ? (fr() ? "total national" : "national total") : (fr() ? "valeur nationale" : "national value")) : (fr() ? "médiane des unités" : "median of units")) + "</span></div>";
      h += "<div class='mapx-poslab'><span>" + (fr() ? "Médiane " : "Median ") + fmt(q(s, 0.5), m.fmt) + "</span><span>" + s.length + " " + (fr() ? "unités" : "units") + "</span></div>";
      h += "<p class='mapx-desc' style='margin-top:10px'>" + (fr() ? "Cliquez sur une unité de la carte ou du classement pour afficher sa fiche." : "Click a unit on the map or in the ranking to open its profile.") + "</p>";
    } else {
      var i = S.sel, v = a[i];
      h += "<div class='mapx-dname'>" + esc(lv.names[i]) + "</div><div class='mapx-dparent'>" + esc(lv.parents[i] || "") +
           (lv.pop[i] ? " · " + fmt(lv.pop[i], "int") + (fr() ? " hab." : " people") : "") + "</div>";
      h += "<div class='mapx-dbig'><b>" + fmt(v, m.fmt) + "</b><span>" + (ranks[i] ? ((fr() ? "rang " : "rank ") + ranks[i] + " / " + n) : "") + "</span></div>";
      if(v !== null && isFinite(v)){
        var pos = Math.max(0, Math.min(1, (v - rg[0]) / (rg[1] - rg[0]))) * 100;
        h += "<div class='mapx-pos' style='background:" + gradient(sc) + "'><i style='left:calc(" + pos + "% - 1px)'></i>";
        if(!m.ext && natv !== null && natv !== undefined && isFinite(natv)){
          var np = Math.max(0, Math.min(1, (natv - rg[0]) / (rg[1] - rg[0]))) * 100;
          h += "<em style='left:calc(" + np + "% - 1px)' title='" + (fr() ? "national" : "national") + "'></em>";
        }
        h += "</div><div class='mapx-poslab'><span>" + fmt(rg[0], m.fmt) + "</span><span>" +
             (natv !== null && natv !== undefined ? ((m.ext ? (fr() ? "total national : " : "national total: ") : (fr() ? "national : " : "national: ")) + fmt(natv, m.fmt)) : "") + "</span><span>" + fmt(rg[1], m.fmt) + "</span></div>";
      }
      h += "<ul class='mapx-dl'>";
      avail().forEach(function(mm){
        var vv = (L().values[mm.key][slot(mm)] || [])[i];
        h += "<li data-k='" + mm.key + "' class='" + (mm.key === S.m ? "cur" : "") + "'><span>" + t(mm) + "</span><span>" + fmt(vv, mm.fmt) + "</span></li>";
      });
      h += "</ul><button class='mapx-icon' id='mapx-clear' type='button' style='margin-top:10px;width:100%'>" + (fr() ? "Retour à la vue d'ensemble" : "Back to overview") + "</button>";
    }
    box.innerHTML = h;
    box.querySelectorAll("li[data-k]").forEach(function(li){ li.addEventListener("click", function(){ S.m = li.getAttribute("data-k"); refresh(); }); });
    var c = document.getElementById("mapx-clear"); if(c) c.addEventListener("click", function(){ S.sel = null; setView(fit(D.bounds)); refresh(); });
  }

  function texts(m){
    document.getElementById("mapx-title").textContent = t(m) + (m.kind === "change" ? " · " + D.years[0] + "–" + D.years[D.years.length-1] : " · " + S.y);
    document.getElementById("mapx-desc").textContent = fr() ? m.dfr : m.den;
    var csv = "data/ntl_" + D.iso3 + "_" + S.lvl + "_" + (m.kind === "change" ? D.years[D.years.length-1] : S.y) + ".csv";
    document.getElementById("mapx-foot").innerHTML = esc(t(D.foot)) + " · <a href='" + csv + "'>" + (fr() ? "Télécharger les données (CSV)" : "Download the data (CSV)") + "</a>";
  }
  function esc(s){ return String(s).replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;"); }

  function select(i, fly){
    S.sel = i;
    if(fly){ setView(fit(L().bbox[i], 0.9)); }
    refresh();
  }
  function refresh(){ controls(); render(); }

  document.getElementById("mapx-metric").addEventListener("change", function(e){ S.m = e.target.value; refresh(); });
  document.getElementById("mapx-search").addEventListener("change", function(e){
    var v = (e.target.value || "").trim().toLowerCase(); if(!v) return;
    var names = L().names.map(function(n){ return n.toLowerCase(); });
    var i = names.indexOf(v); if(i < 0){ i = names.findIndex(function(n){ return n.indexOf(v) === 0; }); }
    if(i < 0){ i = names.findIndex(function(n){ return n.indexOf(v) >= 0; }); }
    if(i >= 0){ select(i, true); e.target.blur(); }
  });
  document.getElementById("mapx-reset").addEventListener("click", function(){ S.sel = null; setView(fit(D.bounds)); refresh(); });
  document.getElementById("mapx-fs").addEventListener("click", function(){
    host.classList.toggle("fs"); document.body.style.overflow = host.classList.contains("fs") ? "hidden" : "";
    setTimeout(function(){ try{ Plotly.Plots.resize(mapEl); }catch(e){} setView(fit(S.sel !== null ? L().bbox[S.sel] : D.bounds, S.sel !== null ? 0.9 : 0)); render(); }, 60);
  });
  document.addEventListener("keydown", function(e){ if(e.key === "Escape" && host.classList.contains("fs")){ document.getElementById("mapx-fs").click(); } });

  refresh();
  probe();
  return {onLang: refresh, state: S};
})();

window.addEventListener("load", function(){
  var saved = null; try{ saved = localStorage.getItem("ntl_lang"); }catch(e){}
  if(LANGS.indexOf(saved) < 0) saved = null;
  var nav = (navigator.language||"fr").slice(0,2);
  var auto = (LANGS.indexOf(nav) >= 0) ? nav : LANGS[0];
  setLang(saved || auto);
});
</script>
</body></html>'''

# ---------------- bouton de langue ------------------------------------------------------
if len(LANGS) > 1:
    _btns = "".join(
        f'<button id="btn-{l}" class="{"on" if i == 0 else ""}" '
        f"onclick=\"setLang('{l}')\">{l.upper()}</button>"
        for i, l in enumerate(LANGS))
    langbtn_html = f'<div class="langbtn">{_btns}</div>'
else:
    langbtn_html = ""

# ---------------- pastilles d'en-tête ---------------------------------------------------
_vcls = {"publishable": "v-ok", "experimental": "v-warn",
         "diagnostic": "v-bad"}.get(RESULTS["G"]["verdict"], "v-warn")
_prov = {"demo": ("Simulated data", "Données simulées"),
         "local": ("Local GeoTIFF", "GeoTIFF local"),
         "gee": ("Earth Engine VIIRS", "Earth Engine VIIRS"),
         "blackmarble": ("NASA Black Marble", "NASA Black Marble")}[CONFIG["PROVIDER"]]

def _chip(en, fr, cls=""):
    c = f" {cls}" if cls else ""
    return (f'<span class="chip{c}"><span class="dot"></span>'
            f'<span data-en="{en}" data-fr="{fr}">{(en, fr)[LANG0]}</span></span>')

_V = RESULTS["G"]["verdict"].upper()
chips_html = "".join([
    _chip(f"Verdict: {_V}", f"Verdict : {_V}", _vcls),
    _chip(f"{len(CONFIG['ADMIN_LEVELS'])} administrative levels",
          f"{len(CONFIG['ADMIN_LEVELS'])} niveaux administratifs"),
    _chip("7 analytical scenarios", "7 scénarios analytiques"),
    _chip(f"{len(FIGS)} interactive charts", f"{len(FIGS)} graphiques interactifs"),
    _chip(_prov[0], _prov[1]),
])

demo_banner = (f'<div class="demo" data-en="{I18N["demo_warn"][0]}" '
               f'data-fr="{I18N["demo_warn"][1]}">{I18N["demo_warn"][LANG0]}</div>') if IS_DEMO else ""

html = (TEMPLATE
    .replace("__TITLE_EN__", I18N["title"][0]).replace("__TITLE_FR__", I18N["title"][1])
    .replace("__SUB_EN__", I18N["subtitle"][0]).replace("__SUB_FR__", I18N["subtitle"][1])
    .replace("__PER_EN__", I18N["period"][0]).replace("__PER_FR__", I18N["period"][1])
    .replace("__TITLE_DEF__", I18N["title"][LANG0]).replace("__SUB_DEF__", I18N["subtitle"][LANG0])
    .replace("__PER_DEF__", I18N["period"][LANG0])
    .replace("__TITLE__", I18N["title"][LANG0]).replace("__SUBTITLE__", I18N["subtitle"][LANG0])
    .replace("__LANGBTN__", langbtn_html)
    .replace("__LANGS__", json.dumps(LANGS))
    .replace("__CHIPS__", chips_html)
    .replace("__DEMO__", demo_banner)
    .replace("__NAV__", nav_html)
    .replace("__PANELS__", "".join(P))
    .replace("__AXIS__", json.dumps(AXIS_I18N, ensure_ascii=False))
    .replace("__AUTHOR__", CONFIG["AUTHOR"])
    .replace("__RUN__", RUN_ID)
    .replace("__DATE__", datetime.now(timezone.utc).strftime("%Y-%m-%d"))
    .replace("__LICD__", CONFIG["LICENSE_DATA"]).replace("__LICC__", CONFIG["LICENSE_CODE"]))

SITE = OUT / "site" / "index.html"
SITE.write_text(html, encoding="utf-8")
(OUT / "site" / ".nojekyll").write_text("")
print(f"✅  {SITE}   {SITE.stat().st_size/1024:.0f} kB")
print(f"    {len(FIGS)} figures interactives · {len(P)} panneaux · langues : {', '.join(LANGS)}")
print(f"    ouvrir en local :  file://{SITE.resolve()}")

In [ ]:
# --------------------------------------------------------------------------------------
# 8.7 · Prévisualiser le tableau de bord dans le notebook
#        Fonctionne sous Colab, Kaggle et JupyterLab : la page est intégrée en data: URI,
#        donc ni serveur web local ni accès file:// ne sont nécessaires.
# --------------------------------------------------------------------------------------
import base64

try:
    from IPython.display import display, HTML as IHTML

    _b64 = base64.b64encode(SITE.read_bytes()).decode("ascii")
    display(IHTML(
        '<p style="font-family:Calibri,sans-serif;font-size:13px;color:#5E6964;margin:0 0 6px 0;">'
        'Preview · Aperçu — scroll inside the frame; the EN/FR button works here too. '
        'Faites défiler dans le cadre ; le bouton EN/FR fonctionne aussi ici.</p>'
        f'<iframe src="data:text/html;base64,{_b64}" width="100%" height="780" '
        'style="border:1px solid #D5DED9;border-radius:10px;background:#fff;"></iframe>'
    ))
except Exception as e:
    print("Aperçu indisponible dans cet environnement :", e)
    print("Ouvrez le fichier directement :", SITE.resolve())


def download_site(zip_name=None):
    '''Compresse outputs/site et le propose au téléchargement (Colab), ou affiche son chemin.'''
    import shutil
    zip_name = zip_name or f"ntl_{CONFIG['ISO3'].lower()}_dashboard"
    base = OUT / zip_name
    archive = shutil.make_archive(str(base), "zip", root_dir=str(OUT / "site"))
    size = Path(archive).stat().st_size / 1024
    print(f"✅  {archive}  ({size:.0f} kB)")
    if globals().get("ENV") == "colab":
        try:
            from google.colab import files          # noqa
            files.download(archive)
        except Exception as e:
            print("   (téléchargement navigateur indisponible :", e, ")")
    elif globals().get("ENV") == "kaggle":
        print("   Kaggle → à récupérer dans le panneau « Output », à droite.")
    return archive


print("\ndownload_site()  →  compresser le tableau de bord pour le télécharger")

---

<div style="background:linear-gradient(120deg,#00553A,#00704A 55%,#00A86A);border-radius:12px;
            padding:26px 32px;color:#fff;font-family:Calibri,sans-serif;">
  <div style="font-size:52px;font-weight:700;color:#F5C242;line-height:1;">09</div>
  <div style="font-size:11px;letter-spacing:3px;font-weight:700;color:#F5C242;margin-top:6px;">
    PUBLICATION</div>
  <div style="font-size:30px;font-weight:700;margin-top:10px;">De votre ordinateur à une adresse publique</div>
  <div style="font-size:16px;font-style:italic;color:#E6F6EE;margin-top:4px;">
    En une seule cellule</div>
</div>

### 9.1 · Obtenir un jeton

GitHub → *Settings* → *Developer settings* → *Personal access tokens*.

- **Jeton classique** (le plus simple) : cochez la portée **`repo`**. Cette seule portée couvre la création du
  dépôt, l'envoi des fichiers et l'activation de Pages.
- **Jeton à portée fine** (recommandé par les équipes de sécurité) : *All repositories* ou le dépôt cible, avec
  **Administration : Read & write**, **Contents : Read & write**, **Pages : Read & write**.

Fixez une expiration à **7 jours**. Vous créez un identifiant pendant un atelier, sur une machine qui ne vous
appartient peut-être pas.

> 🔐 **Trois règles non négociables**
> 1. Le jeton se saisit dans `getpass` : jamais dans une cellule, jamais dans `CONFIG`, jamais dans un fichier
>    `.env` susceptible d'être commité.
> 2. Si un jeton se retrouve un jour dans un commit, **révoquez-le immédiatement**. Supprimer le commit ne suffit
>    pas : le cache de GitHub et chacun des clones le conservent.
> 3. Révoquez le jeton à la fin de la semaine. Un identifiant d'atelier ne doit pas survivre à l'atelier.

### 9.2 · La publication vous est proposée, elle ne se déclenche pas toute seule

La dernière cellule de cette section vous pose la question — *« Publier ce tableau de bord sur GitHub
Pages ? [o/N] »*. Répondez `o` et elle demande alors votre jeton en saisie masquée, crée le dépôt, envoie les
fichiers et active Pages. Toute autre réponse, ou une simple touche Entrée, ne publie rien : vous pouvez
ré-exécuter la cellule autant de fois que nécessaire.

Si vous devez automatiser l'exécution du notebook (tâche planifiée, intégration continue), passez
`AUTO_PUBLISH = True` dans cette cellule : la question n'est plus posée et le jeton est lu dans la variable
d'environnement `GITHUB_TOKEN`.

**Comment fonctionne la publication** — Pas un appel HTTP par fichier. Le notebook crée un **blob** pour chaque
fichier, les assemble en un **arbre** unique, produit **un seul commit** et déplace la référence de branche.
L'historique de votre dépôt reste propre — un commit par publication, et non quarante — et l'envoi est atomique :
soit il aboutit entièrement, soit il n'aboutit pas du tout.

In [ ]:
# --------------------------------------------------------------------------------------
# 9.2 · Le mobilier du dépôt — README, LICENCE, CITATION
# --------------------------------------------------------------------------------------
PAGES_URL_PLACEHOLDER = "<published-url>"

README = f'''# {CONFIG["COUNTRY_NAME"]} — Night-Time Lights subnational indicators
# {CONFIG["COUNTRY_NAME"]} — Indicateurs infranationaux de lumières nocturnes

> Produced during the **STG17 Technical Workshop — *Emerging Issues, Emerging Practice***
> (African Development Bank · AU STATAFRIC · Day 4 — *Africa after dark*).
> Produit lors de l'**atelier technique STG17**, jour 4.

__COLAB_BADGE__

**Dashboard / Tableau de bord :** see the GitHub Pages URL of this repository.
**Verdict:** `{RESULTS["G"]["verdict"].upper()}` — see [`LIMITATIONS.md`](LIMITATIONS.md) **before citing anything**.

---

## EN — What this is

A reproducible pipeline that turns VIIRS night-time light rasters into subnational statistical indicators for
{CONFIG["COUNTRY_NAME"]} ({CONFIG["ISO3"]}), across {len(CONFIG["ADMIN_LEVELS"])} administrative levels and seven
analytical scenarios: economic activity, electrification, urbanisation, spatial inequality, change detection,
shock monitoring and validation against official statistics.

**Night-time lights are a proxy.** They measure upward radiance at ~01:30 local time and nothing else. Every
indicator here is an *inference*, and the inference is tested in the validation scenario. Read `LIMITATIONS.md`.

### Contents
| Path | What |
|---|---|
| `index.html` | Bilingual (EN/FR) interactive dashboard |
| `data/*.csv` | Indicator tables, one per administrative level and year |
| `data/*.geojson` | Simplified ADM1 boundaries used by the map |
| `manifest.json` | Full run record: parameters, thresholds, software versions, artefact diagnostics |
| `LIMITATIONS.md` | Limitations statement — part of the deliverable, not an afterthought |
| `notebook.ipynb` | The complete, documented pipeline (open it in Colab with the badge above) |
| `requirements.txt` | Pinned minimum versions, for a local run |

### Reproduce
```bash
pip install numpy pandas geopandas rasterio shapely plotly matplotlib requests
jupyter lab notebook.ipynb     # set CONFIG["ISO3"] and CONFIG["PROVIDER"], run all
```

### Key parameters of this run
| Parameter | Value |
|---|---|
| Baseline → endline | {Y0} → {Y1} |
| Provider | `{CONFIG["PROVIDER"]}`{" **— SIMULATED DATA**" if IS_DEMO else ""} |
| Noise floor | {CONFIG["NOISE_FLOOR"]} nW·cm⁻²·sr⁻¹ |
| Urban-core threshold | {CONFIG["URBAN_CORE"]} nW·cm⁻²·sr⁻¹ |
| Top-coding | p{CONFIG["TOPCODE_PCT"]} = {TOPCODE:,.1f} |
| Administrative levels | {", ".join(f"{k} ({len(v)})" for k, v in ADMIN.items())} |
| Best Spearman ρ (validation) | {RESULTS["G"]["best_rho"]:.3f} |

---

## FR — De quoi s'agit-il

Une chaîne de traitement reproductible qui transforme des rasters de lumières nocturnes VIIRS en indicateurs
statistiques infranationaux pour {CONFIG["COUNTRY_NAME"]} ({CONFIG["ISO3"]}), sur
{len(CONFIG["ADMIN_LEVELS"])} niveaux administratifs et sept scénarios d'analyse : activité économique,
électrification, urbanisation, inégalités spatiales, détection de changement, suivi des chocs et validation
face aux statistiques officielles.

**Les lumières nocturnes sont un indicateur indirect.** Elles mesurent une radiance ascendante vers 01h30 locale,
rien d'autre. Chaque indicateur est une *inférence*, testée dans le scénario de validation. Lire `LIMITATIONS.md`.

---

## Sources & licences
| Source | Licence |
|---|---|
| NOAA VIIRS DNB / NASA Black Marble | US Government work — attribution requested |
| geoBoundaries (gbOpen) | CC BY 4.0 |
| WorldPop | CC BY 4.0 |
| **Derived data in this repository** | **{CONFIG["LICENSE_DATA"]}** |
| **Code in this repository** | **{CONFIG["LICENSE_CODE"]}** |

## Citation
See `CITATION.cff`.

## Maintenance
Maintainer / responsable : **{CONFIG["AUTHOR"]}**{" · " + CONFIG["AUTHOR_EMAIL"] if CONFIG["AUTHOR_EMAIL"] else ""}
Last run / dernière exécution : `{RUN_ID}`
'''

MIT = f'''MIT License

Copyright (c) {datetime.now().year} {CONFIG["AUTHOR"]}

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
'''

CITATION = f'''cff-version: 1.2.0
title: "{CONFIG["COUNTRY_NAME"]} — Night-Time Lights subnational indicators"
message: "If you use these data or this code, please cite them as below."
type: dataset
authors:
  - name: "{CONFIG["AUTHOR"]}"
abstract: >-
  Subnational indicators derived from VIIRS night-time lights for
  {CONFIG["COUNTRY_NAME"]} ({CONFIG["ISO3"]}), {Y0}-{Y1}, produced during the
  STG17 Technical Workshop of the African Development Bank and AU STATAFRIC.
keywords:
  - night-time lights
  - VIIRS
  - official statistics
  - "{CONFIG["ISO3"]}"
  - non-traditional data
license: {CONFIG["LICENSE_DATA"]}
date-released: "{datetime.now().strftime('%Y-%m-%d')}"
'''

(OUT / "site" / "README.md").write_text(README, encoding="utf-8")
(OUT / "site" / "LICENSE").write_text(MIT, encoding="utf-8")
(OUT / "site" / "CITATION.cff").write_text(CITATION, encoding="utf-8")
(OUT / "site" / "requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
print("README.md, LICENSE, CITATION.cff et requirements.txt écrits")
print(README[:900] + "\n…")

In [ ]:
# --------------------------------------------------------------------------------------
# 9.3 · Publier sur GitHub Pages — un blob par fichier, un arbre, un seul commit
# --------------------------------------------------------------------------------------
GH = "https://api.github.com"

def _gh(method, path, token, **kw):
    r = requests.request(method, GH + path, timeout=60, headers={
        "Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28"}, **kw)
    return r

def collect_site_files(site_dir=OUT / "site", extra=None):
    '''Renvoie {chemin_dans_le_depot: octets} pour tout ce qui doit être publié.'''
    files = {}
    for p in sorted(Path(site_dir).rglob("*")):
        if p.is_file():
            files[str(p.relative_to(site_dir)).replace(os.sep, "/")] = p.read_bytes()
    for repo_path, local in (extra or {}).items():
        lp = Path(local)
        if lp.exists():
            files[repo_path] = lp.read_bytes()
    return files

def find_notebook():
    '''Retrouve ce notebook sur le disque, pour le publier à côté de ses propres résultats.'''
    roots = [Path.cwd(), Path.cwd().parent, Path("/content"), Path("/kaggle/working"),
             Path("/kaggle/input")]
    cands = []
    for r in roots:
        try:
            if r.is_dir():
                cands += sorted(r.glob("*.ipynb"))
        except Exception:
            pass
    keyed = [p for p in cands if ("ntl" in p.name.lower() or "stg17" in p.name.lower())]
    pick = keyed or [p for p in cands if not p.name.startswith(".")]
    return pick[0] if pick else None


def publish_to_github(token, repo=None, *, files=None, private=False, branch="main",
                      message=None, cfg=CONFIG):
    repo = repo or cfg["REPO_NAME"]
    files = files or collect_site_files()
    message = message or f"NTL indicators — {cfg['ISO3']} — run {RUN_ID}"

    # -- 1. identité ------------------------------------------------------------------
    u = _gh("GET", "/user", token)
    if u.status_code != 200:
        raise RuntimeError(f"Token rejected ({u.status_code}). Check scopes: repo (classic) or "
                           f"Contents+Pages+Administration read/write (fine-grained).")
    owner = u.json()["login"]

    # Le badge Colab ne peut être écrit qu'une fois le propriétaire et le dépôt connus.
    if "README.md" in files:
        _badge = ("[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]"
                  f"(https://colab.research.google.com/github/{owner}/{repo}/blob/{branch}/notebook.ipynb)")
        files["README.md"] = files["README.md"].decode("utf-8").replace(
            "__COLAB_BADGE__", _badge).encode("utf-8")
    log(f"authentifié en tant que {owner}", "ok")

    # -- 2. dépôt ---------------------------------------------------------------------
    r = _gh("GET", f"/repos/{owner}/{repo}", token)
    if r.status_code == 404:
        c = _gh("POST", "/user/repos", token, json={
            "name": repo, "description": cfg["REPO_DESCRIPTION"], "private": private,
            "auto_init": True, "has_issues": True, "has_wiki": False})
        if c.status_code not in (201,):
            raise RuntimeError(f"Could not create repository: {c.status_code} {c.text[:220]}")
        log(f"dépôt créé : {owner}/{repo}", "ok")
        time.sleep(2.5)
    elif r.status_code == 200:
        branch = r.json().get("default_branch", branch)
        log(f"dépôt existant : {owner}/{repo} (branche {branch})")
    else:
        raise RuntimeError(f"Repository check failed: {r.status_code} {r.text[:220]}")

    # -- 3. HEAD courant --------------------------------------------------------------
    base_sha = base_tree = None
    for attempt in range(6):
        ref = _gh("GET", f"/repos/{owner}/{repo}/git/ref/heads/{branch}", token)
        if ref.status_code == 200:
            base_sha = ref.json()["object"]["sha"]
            base_tree = _gh("GET", f"/repos/{owner}/{repo}/git/commits/{base_sha}",
                            token).json()["tree"]["sha"]
            break
        time.sleep(2)
    if base_sha is None:
        log(f"branche {branch} introuvable — création d'un commit initial", "warn")

    # -- 4. blobs ---------------------------------------------------------------------
    tree = []
    total = 0
    for path, blob in files.items():
        if len(blob) > 45 * 1024 * 1024:
            log(f"ignoré (>45 Mo) : {path}", "warn")
            continue
        b = _gh("POST", f"/repos/{owner}/{repo}/git/blobs", token, json={
            "content": base64.b64encode(blob).decode(), "encoding": "base64"})
        if b.status_code != 201:
            raise RuntimeError(f"Blob failed for {path}: {b.status_code} {b.text[:200]}")
        tree.append({"path": path, "mode": "100644", "type": "blob", "sha": b.json()["sha"]})
        total += len(blob)
        log(f"  blob  {path}  ({len(blob)/1024:.0f} ko)")

    # -- 5. arbre, commit, référence --------------------------------------------------
    payload = {"tree": tree}
    if base_tree:
        payload["base_tree"] = base_tree
    t = _gh("POST", f"/repos/{owner}/{repo}/git/trees", token, json=payload)
    if t.status_code != 201:
        raise RuntimeError(f"Tree failed: {t.status_code} {t.text[:220]}")
    cm = {"message": message, "tree": t.json()["sha"]}
    if base_sha:
        cm["parents"] = [base_sha]
    c = _gh("POST", f"/repos/{owner}/{repo}/git/commits", token, json=cm)
    if c.status_code != 201:
        raise RuntimeError(f"Commit failed: {c.status_code} {c.text[:220]}")
    new_sha = c.json()["sha"]
    up = _gh("PATCH", f"/repos/{owner}/{repo}/git/refs/heads/{branch}", token,
             json={"sha": new_sha, "force": True})
    if up.status_code not in (200, 201):
        up = _gh("POST", f"/repos/{owner}/{repo}/git/refs", token,
                 json={"ref": f"refs/heads/{branch}", "sha": new_sha})
    log(f"commit {new_sha[:8]} poussé — {len(tree)} fichiers, {total/1024:.0f} ko", "ok")

    # -- 6. GitHub Pages --------------------------------------------------------------
    pg = _gh("POST", f"/repos/{owner}/{repo}/pages", token,
             json={"source": {"branch": branch, "path": "/"}})
    if pg.status_code in (201, 204):
        log("GitHub Pages activé", "ok")
    elif pg.status_code in (409,):
        _gh("PUT", f"/repos/{owner}/{repo}/pages", token,
            json={"source": {"branch": branch, "path": "/"}})
        log("GitHub Pages déjà activé — source rafraîchie", "ok")
    else:
        log(f"l'API Pages a renvoyé {pg.status_code} : {pg.text[:160]}", "warn")
        log("À activer manuellement : Settings → Pages → Deploy from a branch → "
            f"{branch} / (root)", "warn")

    url = f"https://{owner}.github.io/{repo}/"
    return {"owner": owner, "repo": repo, "branch": branch, "commit": new_sha,
            "repo_url": f"https://github.com/{owner}/{repo}", "pages_url": url,
            "files": len(tree), "bytes": total}

print("publish_to_github() est prête — exécutez la cellule suivante pour publier.")

In [ ]:
# ======================================================================================
# 🚀  PUBLICATION — cette cellule vous pose la question et s'occupe du reste.
# ======================================================================================
# Le jeton est saisi interactivement, jamais stocké, et effacé de la mémoire aussitôt.
#
# Pour une exécution automatisée (aucune saisie possible), mettez AUTO_PUBLISH à True :
# la question n'est plus posée et le jeton est lu dans la variable d'environnement
# GITHUB_TOKEN.
# ======================================================================================
AUTO_PUBLISH = False
# Un verdict « diagnostic » interdit la publication publique. Ne passez ceci à True que
# pour un dépôt privé ou une démonstration, jamais pour diffuser des chiffres.
PUBLIER_MALGRE_DIAGNOSTIC = False

_files = collect_site_files()
_nb = find_notebook()

print("═" * 78)
print(f"  Tableau de bord prêt : {len(_files)} fichiers"
      f"  ·  notebook détecté : {_nb.name if _nb else 'aucun'}")
print(f"  Dépôt proposé : {CONFIG['REPO_NAME']}"
      f"   ·   verdict de diffusion : {RESULTS['G']['verdict'].upper()}")
if IS_DEMO:
    print("  ⚠️   ATTENTION : ces données sont SIMULÉES. Ne les publiez pas comme")
    print("       statistiques. Repassez CONFIG['PROVIDER'] sur 'gee' pour des")
    print("       données réelles avant toute publication publique.")
print("═" * 78)

_verdict = RESULTS["G"]["verdict"]
_blocked = _verdict == "diagnostic" and not PUBLIER_MALGRE_DIAGNOSTIC

if _blocked:
    token, PUBLISH = "", False
    print("\n  ⛔  PUBLICATION BLOQUÉE — le verdict de diffusion est « diagnostic ».")
    print("      L'indicateur n'est pas assez corrélé à une statistique indépendante pour être")
    print("      diffusé publiquement. Motif : " + " ; ".join(RESULTS["G"]["reasons"]) + ".")
    print("      Pour une démonstration ou un dépôt privé : PUBLIER_MALGRE_DIAGNOSTIC = True.")
elif AUTO_PUBLISH:
    token, PUBLISH = os.environ.get("GITHUB_TOKEN", ""), True
else:
    PUBLISH = ask_yes_no("\n  Publier ce tableau de bord sur GitHub Pages ? [o/N] ")
    if PUBLISH and IS_DEMO:
        PUBLISH = ask_yes_no("  ⚠️  Ce sont des DONNÉES SIMULÉES. Publier quand même, à titre de "
                             "démonstration uniquement ? [o/N] ")
    token = ""
    if PUBLISH:
        print("\n  Jeton d'accès personnel GitHub — portée « repo » (voir §9.1).")
        print("  La saisie est masquée : rien ne s'affiche pendant que vous tapez.")
        token = ask("  Jeton : ", secret=True)

if PUBLISH and not token:
    print("\n  ⚠️  Aucun jeton fourni — rien n'a été envoyé.")
    print("     Ré-exécutez cette cellule pour réessayer.")
elif PUBLISH:
    try:
        _extra = {"notebook.ipynb": str(_nb)} if _nb is not None else {}
        info = publish_to_github(token, files=collect_site_files(extra=_extra),
                                 private=(_verdict == "diagnostic"))
        print("\n" + "═" * 78)
        print("  ✅  PUBLIÉ")
        print(f"  Dépôt           : {info['repo_url']}")
        print(f"  Tableau de bord : {info['pages_url']}")
        print(f"  {info['files']} fichiers · {info['bytes']/1024:.0f} ko · commit {info['commit'][:8]}")
        print("═" * 78)
        print("  La première publication prend 1 à 3 minutes avant d'être visible.")
        print("  Ensuite : Settings → Pages pour confirmer.")
        print("  Révoquez votre jeton à la fin de la semaine.")
    finally:
        del token          # retiré immédiatement de l'espace de noms du noyau
else:
    if not _blocked:
        print("\n  ⏸️   Rien n'a été publié.")
        print("     Ré-exécutez cette cellule et répondez « o » quand vous serez prêt.")

---

## 10 · Votre diapositive du vendredi

Le jour 5 demande à chaque équipe de présenter le travail de la semaine dans un créneau court. La cellule
ci-dessous produit une fiche pays unique au format 16:9 — chiffres clés, carte, verdict — dimensionnée pour être
déposée telle quelle dans une présentation. Elle reprend la palette institutionnelle inspirée de la BAD, de sorte
qu'elle se place à côté du matériel de l'atelier sans détonner.

In [ ]:
# --------------------------------------------------------------------------------------
# 10.1 · Fiche pays d'une page (16:9) pour la présentation du jour 5
# --------------------------------------------------------------------------------------
fig = plt.figure(figsize=(13.33, 7.5), facecolor="white")
fig.patch.set_facecolor("white")

# hero band
ax_h = fig.add_axes([0, .80, 1, .20]); ax_h.axis("off")
ax_h.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax_h.transAxes, color=PAL["forest"], zorder=0))
grad = np.linspace(0, 1, 512).reshape(1, -1)
ax_h.imshow(grad, extent=[0, 1, 0, 1], aspect="auto", zorder=1,
            cmap=LinearSegmentedColormap.from_list("hb", [PAL["forest"], PAL["deep"], PAL["green"]]))
ax_h.text(.035, .70, "AFRICAN DEVELOPMENT BANK · AU STATAFRIC · STG17", color=PAL["gold"],
          fontsize=9, fontweight="bold", zorder=3, transform=ax_h.transAxes)
ax_h.text(.035, .36, f"{CONFIG['COUNTRY_NAME']} — Lumières nocturnes {Y0}–{Y1}", color="white",
          fontsize=25, fontweight="bold", zorder=3, transform=ax_h.transAxes)
ax_h.text(.035, .13, "Indicateurs infranationaux · atelier STG17", color="#E6F6EE",
          fontsize=12.5, style="italic", zorder=3, transform=ax_h.transAxes)
ax_h.add_patch(plt.Rectangle((0, 0), 1, .035, transform=ax_h.transAxes, color=PAL["gold"], zorder=4))

# rangée d'indicateurs clés
kpi_vals = [
    (f"{natA['sol_cagr']:+.1%}", "Croissance de la SoL", "par an", PAL["green"]),
    (f"{RESULTS['B']['national_dark']/1e6:.2f}M", "Personnes en cellules",
     "habitées non éclairées", PAL["brick"]),
    (f"{RESULTS['C']['lit_area'][1]:,.0f}", "km² de surface éclairée", f"en {Y1}", PAL["teal"]),
    (f"{RESULTS['D']['gini_1']:.3f}", "Gini de la lumière", "par habitant", PAL["ochre"]),
]
for i, (v, en, fr, c) in enumerate(kpi_vals):
    ax = fig.add_axes([.035 + i * .235, .615, .215, .15]); ax.axis("off")
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, facecolor=PAL["mist"],
                               edgecolor=PAL["sage"], lw=.8))
    ax.text(.06, .62, v, fontsize=25, fontweight="bold", color=c, transform=ax.transAxes)
    ax.text(.06, .36, en, fontsize=9.5, color=PAL["ink"], transform=ax.transAxes)
    ax.text(.06, .15, fr, fontsize=8.5, color=PAL["slate"], style="italic", transform=ax.transAxes)

# map
ax_m = fig.add_axes([.035, .06, .44, .52])
mm = ADMIN["ADM1"].merge(Z[("ADM1", Y1)][["shapeID", "sol_per_capita"]], on="shapeID", how="left")
mm.plot(column="sol_per_capita", ax=ax_m, cmap=LinearSegmentedColormap.from_list("r", RAMP),
        edgecolor="white", linewidth=.6, legend=True,
        legend_kwds={"label": "SoL par habitant", "shrink": .62})
ax_m.axis("off")
ax_m.set_title(f"Lumière par habitant, {Y1}", loc="left",
               fontsize=12, color=PAL["ink"])

# text block
ax_t = fig.add_axes([.50, .06, .465, .52]); ax_t.axis("off")
vcol = {"publishable": PAL["green"], "experimental": PAL["ochre"],
        "diagnostic": PAL["brick"]}.get(RESULTS["G"]["verdict"], PAL["ochre"])
ax_t.add_patch(plt.Rectangle((0, .74), 1, .26, transform=ax_t.transAxes,
                             facecolor=PAL["mintmist"], edgecolor=vcol, lw=1.6))
ax_t.text(.03, .92, "VERDICT DE DIFFUSION", fontsize=8.5,
          fontweight="bold", color=PAL["slate"], transform=ax_t.transAxes)
ax_t.text(.03, .80, RESULTS["G"]["verdict"].upper(), fontsize=19, fontweight="bold",
          color=vcol, transform=ax_t.transAxes)
body = textwrap.fill(INSIGHTS["A"]["fr"], 74) + "\n\n" + textwrap.fill(INSIGHTS["B"]["fr"], 74)
ax_t.text(.02, .68, body, fontsize=9.3, color=PAL["ink"], va="top", transform=ax_t.transAxes)
lim = (f"Limites : plancher de bruit {CONFIG['NOISE_FLOOR']} nW·cm⁻²·sr⁻¹ · le 1 % de pixels les "
       f"plus vifs concentre {top1:.0%} de la SoL nationale · {flares} torchère(s) suspectée(s) · "
       f"les NTL sont un indicateur indirect, ni une mesure de l'accès à l'électricité, ni du PIB.")
ax_t.text(.02, .18, textwrap.fill(lim, 80), fontsize=8.2, color=PAL["slate"], style="italic",
          va="top", transform=ax_t.transAxes)
ax_t.text(.02, .02, f"Sources: VIIRS/Black Marble · geoBoundaries CC BY 4.0 · WorldPop CC BY 4.0"
          f"   |   run {RUN_ID}", fontsize=7.4, color=PAL["slate"], transform=ax_t.transAxes)

if IS_DEMO:
    fig.text(.5, .40, "DONNÉES SIMULÉES", fontsize=62, color=PAL["brick"], alpha=.13,
             ha="center", va="center", rotation=22, fontweight="bold")

card = OUT / "figures" / f"country_card_{CONFIG['ISO3']}.png"
plt.savefig(card, dpi=170, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅  {card}  — à déposer telle quelle dans votre présentation de vendredi")

## 11 · Exercices

Classés par difficulté croissante. Les exercices 1 à 4 sont attendus pendant le laboratoire ; les exercices 5 à 7
sont ceux qui transforment ce travail en produit national.

| # | Exercice | Compétence |
|---|---|---|
| **1** | Relancer le §5.2 avec `NOISE_FLOOR` = 0,15 / 0,25 / 0,50 et tabuler la part éclairée nationale. Publier l'intervalle obtenu. | Analyse de sensibilité |
| **2** | Identifier lesquels des six artefacts sont présents dans **votre** pays, avec la sortie de cellule qui le prouve pour chacun. | Qualité des données |
| **3** | Remplacer `geoBoundaries` par votre fichier ADM1/ADM2 **officiel** et relancer. Comparer les deux jeux de chiffres zonaux et expliquer les écarts importants. | Provenance des limites |
| **4** | Ajouter une troisième année et vérifier si le classement de croissance ADM2 reste stable sur les trois comparaisons deux à deux. Un classement qui s'inverse n'est pas un classement. | Robustesse |
| **5** | Apporter une **statistique officielle indépendante** au niveau ADM1 (taux d'électrification, PIB régional, nombre d'entreprises) dans `OFFICIAL_STATS_CSV` et rejouer le scénario G. Le verdict change-t-il ? | Validation |
| **6** | Joindre l'indicateur de connectivité **Ookla** du jour 3 à cette table et tester si lumière et connectivité racontent la même histoire spatiale. Les désaccords sont la partie intéressante. | Croisement de sources |
| **7** | Rédiger la note de diffusion en deux paragraphes que votre institut joindrait à ce produit : ce qu'il dit, ce qu'il ne dit pas, qui contacter. | Diffusion |

---

## 12 · Éthique, licences et gouvernance des données

Le paquet de travail 4.3 du plan d'action STG17 demande un guide de référence sur la gouvernance des données.
Cette liste de contrôle en est la version opérationnelle pour un produit NTL. Cochez chaque ligne avant de
publier.

- [ ] **Compatibilité des licences.** Chaque licence d'entrée autorise la redistribution du produit dérivé, et la
      licence dérivée est indiquée sur la page. *(NTL : œuvre du gouvernement américain · geoBoundaries et
      WorldPop : CC BY 4.0 — toutes publiables. À comparer aux données Ookla du jour 3 : CC BY-NC-SA 4.0, donc
      non commerciale et partage à l'identique.)*
- [ ] **Aucun risque de ré-identification.** L'agrégation se fait au niveau administratif, jamais au bâtiment. Un
      pixel de 500 m sur une zone peu peuplée peut, combiné à d'autres données, désigner une seule concession :
      ne publiez pas d'extraits au pixel sur des zones habitées.
- [ ] **Ne pas nuire.** Une carte publique des « cellules habitées non éclairées » est une carte des
      établissements vulnérables. En contexte de conflit ou de déplacement, demandez-vous si l'agrégation ADM2
      est assez grossière, ou si cette couche doit rester interne.
- [ ] **Attribution lisible par machine.** `CITATION.cff` et `manifest.json` sont présents dans le dépôt.
- [ ] **La déclaration de limites est publiée avec le produit**, et non fournie sur demande.
- [ ] **Le caractère indirect de l'indicateur est mentionné** dans le titre ou le sous-titre de chaque graphique
      qui quitte le bureau.
- [ ] **Un responsable nommé et une date de revue.** Un tableau de bord public non entretenu devient, avec le
      temps, de la désinformation.
- [ ] **Hygiène des jetons.** Aucun identifiant dans un commit ; jetons d'atelier révoqués en fin de semaine.

## 13 · Sources et lectures

**Produits de données**

| Produit | Fournisseur | Accès |
|---|---|---|
| VNP46A2 / A3 / A4 (Black Marble, journalier/mensuel/annuel) | NASA LAADS DAAC | `ladsweb.modaps.eosdis.nasa.gov` — compte Earthdata |
| Composites mensuels et annuels VIIRS DNB (V2.x) | NOAA / Earth Observation Group, Colorado School of Mines | `eogdata.mines.edu/nighttime_light/` |
| VIIRS DNB dans Earth Engine | Google | `NOAA/VIIRS/DNB/ANNUAL_V21` (2012–2021), `NOAA/VIIRS/DNB/ANNUAL_V22` (2022 →), `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG` |
| Limites administratives | geoBoundaries (gbOpen) | `geoboundaries.org` — CC BY 4.0 |
| Population maillée | WorldPop | `worldpop.org` — CC BY 4.0 |

**Références méthodologiques**

- Elvidge, C. D., Baugh, K., Zhizhin, M., Hsu, F.-C. et Ghosh, T. — *VIIRS night-time lights*. La description de
  référence de la famille de produits VNL et de ses corrections.
- Henderson, J. V., Storeygard, A. et Weil, D. N. (2012) — *Measuring Economic Growth from Outer Space*,
  American Economic Review. L'article fondateur de l'usage des lumières comme indicateur indirect du PIB là où la
  comptabilité nationale est fragile.
- Chen, X. et Nordhaus, W. D. (2011) — *Using luminosity data as a proxy for economic statistics*, PNAS. Établit
  quand les lumières apportent de l'information et quand elles n'en apportent pas.
- Gibson, J., Olivia, S. et Boe-Gibson, G. — travaux critiques sur l'erreur de mesure, le halo lumineux et les
  limites des lumières nocturnes en faible densité. **À lire avant de publier un indicateur rural.**
- Román, M. O. et al. (2018) — *NASA's Black Marble night-time lights product suite*, Remote Sensing of
  Environment. Les corrections BRDF et atmosphériques qui rendent les séries temporelles comparables.
- Comité d'experts des Nations unies sur les mégadonnées et la science des données pour la statistique officielle
  — travaux des équipes de projet sur l'imagerie satellitaire et cadre de qualité des sources non probabilistes.

**Contexte de l'atelier** — Plan d'action STG17 2025-2030 (SHaSA II, domaine prioritaire 18 — questions
émergentes), activités 2.1.1, 3.1.1, 4.2.1 et 4.3. Ce notebook est un instrument opérationnel du paquet de
travail 4, activité 4.2.

---

<div style="background:linear-gradient(120deg,#00553A,#00704A 55%,#00A86A);border-radius:12px;
            padding:30px 34px;color:#fff;font-family:Calibri,sans-serif;">
  <div style="font-size:11px;letter-spacing:3px;font-weight:700;color:#F5C242;">
    FIN DU NOTEBOOK</div>
  <div style="font-size:27px;font-weight:700;margin-top:10px;">
    Vous avez maintenant une adresse publique. Le plus dur commence.</div>
  <div style="font-size:15px;color:#E6F6EE;margin-top:10px;max-width:900px;line-height:1.6">
    Un tableau de bord n'est pas une statistique. Ce qui en fait un produit officiel, c'est la phrase que vous
    acceptez de signer : <i>« cet indicateur mesure X, il ne mesure pas Y, voici les preuves, voici qui
    l'entretient. »</i> Le code ci-dessus s'est écrit en deux minutes ; cette phrase-là représente une semaine de
    travail, et c'est la seule partie que votre institut est seul à pouvoir produire.
  </div>
  <div style="height:5px;background:#F5C242;border-radius:3px;margin-top:24px;width:210px;"></div>
</div>